In [1]:
# ============================================================
# ADD MULTIPLE HTS-SPECIFIC NOISES TO SYNTHETIC BENGALURU HTS
# ============================================================
# This script adds 8 realistic Household Travel Survey noises:
# 1. Income rounding and reporting error
# 2. Income non-response / missing income
# 3. Trip underreporting
# 4. Trip-purpose misclassification
# 5. Mode misreporting
# 6. Departure-time recall error
# 7. Trip-distance and duration estimation error
# 8. Occupation misclassification
#
# Input : HTS_Bengaluru_Synthetic_Responses_1200.xlsx
# Output: output/bengaluru_hts_noisy_low.csv
#         output/bengaluru_hts_noisy_medium.csv
#         output/bengaluru_hts_noisy_high.csv
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------
# [ERROR] INPUT_FILE = r"C:\Users\Admin\Downloads\HTS\HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
# [FIX] Windows path does not exist on this Mac; the workbook is in the same folder as this notebook
INPUT_FILE = "HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
OUTPUT_FOLDER = "output"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# 2. Load dataset (Use read_excel for .xlsx files)
# ------------------------------------------------------------
# [BUG] df = pd.read_excel(INPUT_FILE)
# [FIX] With no sheet name, read_excel loads the README sheet (25 rows of text), so every noise step
#       below found none of its columns. Build the person-trip table from the survey sheets instead,
#       with labels from the workbook's own Codebook sheet.
sheets = pd.read_excel(
    INPUT_FILE, sheet_name=["A_Households", "B_Person_Roster", "E_Trips_Full", "Codebook"]
)
codebook = sheets["Codebook"]
def codebook_labels(variable):
    """Code -> label dictionary for one variable of the workbook's Codebook sheet."""
    rows = codebook[codebook["Variable"] == variable]
    return dict(zip(rows["Code"].astype(int), rows["Meaning"]))


BAND_TO_INR = {
    "Below ₹15,000": 12000,
    "₹15,000 to ₹30,000": 22500,
    "₹30,000 to ₹60,000": 45000,
    "₹60,000 to ₹1,00,000": 80000,
    "Above ₹1,00,000": 150000,
    # "Refused / don't know" is not listed, so it stays missing (NaN)
}
persons = sheets["B_Person_Roster"]
households = sheets["A_Households"]
df = sheets["E_Trips_Full"].copy()
df["mode"] = df["E8_Primary_mode"].map(codebook_labels("Mode (C4,C5,D3,E8,E9)"))
df["trip_purpose"] = df["E4_Purpose"].map(codebook_labels("Purpose (E4)"))
df["trip_distance_km"] = df["E7_Distance_km"]
df["trip_duration_min"] = df["E6_Travel_time_min"]
df["departure_time_hhmm"] = df["E5_Departure"].astype(str)
df["travel_cost_inr"] = df["E10_Cost_INR"]
persons = persons.assign(occupation=persons["B4_Activity"].map(codebook_labels("B4 Activity")))
households = households.assign(
    household_monthly_income_inr=households["A4_Expenditure_band"].map(BAND_TO_INR)
)
df = df.merge(persons[["HH_ID", "Person_ID", "occupation"]], on=["HH_ID", "Person_ID"], how="left")
df = df.merge(households[["HH_ID", "household_monthly_income_inr"]], on="HH_ID", how="left")

print("Original dataset shape:", df.shape)
print("Original columns:", df.columns.tolist())


# ------------------------------------------------------------
# 3. Helper function: convert HH:MM to minutes
# ------------------------------------------------------------
def hhmm_to_minutes(time_value):
    try:
        hours, minutes = str(time_value).split(":")
        return int(hours) * 60 + int(minutes)
    except Exception:
        return np.nan


# ------------------------------------------------------------
# 4. Helper function: convert minutes to HH:MM
# ------------------------------------------------------------
def minutes_to_hhmm(minutes):
    minutes = int(minutes) % (24 * 60)
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"


# ------------------------------------------------------------
# 5. Main noise-adding function
# ------------------------------------------------------------
def add_hts_noise(df, noise_level=0.05, random_state=42):
    """noise_level:

    0.03 = low noise
    0.05 = medium noise
    0.10 = high noise
    """
    rng = np.random.default_rng(random_state)
    noisy = df.copy()
    n = len(noisy)

    # [BUG] # ========================================================
    # [BUG] # NOISE 1: INCOME ROUNDING AND REPORTING ERROR
    # [BUG] # ========================================================
    # [BUG] if "household_monthly_income_inr" in noisy.columns:
    # [BUG]     income_noise = rng.normal(
    # [BUG]         loc=1.0, scale=noise_level * 1.5, size=n
    # [BUG]     )
    # [BUG]     noisy["household_monthly_income_inr"] = (
    # [BUG]         noisy["household_monthly_income_inr"] * income_noise
    # [BUG]     )
    # [BUG]     # Round income to nearest ₹500
    # [BUG]     noisy["household_monthly_income_inr"] = (
    # [BUG]         noisy["household_monthly_income_inr"] / 500
    # [BUG]     ).round() * 500
    # [BUG]     noisy["household_monthly_income_inr"] = noisy[
    # [BUG]         "household_monthly_income_inr"
    # [BUG]     ].clip(lower=5000)
    # [BUG]
    # [BUG] # ========================================================
    # [BUG] # NOISE 2: INCOME NON-RESPONSE / MISSING INCOME
    # [BUG] # ========================================================
    # [BUG] if "household_monthly_income_inr" in noisy.columns:
    # [BUG]     missing_income_fraction = noise_level * 0.6
    # [BUG]     missing_income_idx = noisy.sample(
    # [BUG]         frac=missing_income_fraction, random_state=random_state
    # [BUG]     ).index
    # [BUG]     noisy.loc[missing_income_idx, "household_monthly_income_inr"] = np.nan
    # [FIX] Removed: this noise changes a LABEL the models are scored against. Labels must stay
    #       clean (noise belongs on the travel inputs only); noising them per trip also gave one
    #       household several incomes and one person several occupations.

    # ========================================================
    # NOISE 3: TRIP UNDERREPORTING
    # ========================================================
    # [FIX] This step removed no trips before: purposes were numeric codes ('4', '7'), so the
    #       words below never matched. Purposes are now Codebook names ('Shopping', 'Social').
    if "trip_purpose" in noisy.columns and "trip_distance_km" in noisy.columns:
        short_nonwork_trips = noisy[
            (noisy["trip_distance_km"] < 3)
            & (
                noisy["trip_purpose"]
                .astype(str)
                .str.lower()
                .isin(["shopping", "other", "social", "recreation", "leisure"])
            )
        ].index

        remove_fraction = noise_level * 1.5
        remove_count = int(len(short_nonwork_trips) * remove_fraction)

        if remove_count > 0 and len(short_nonwork_trips) > 0:
            remove_idx = rng.choice(
                short_nonwork_trips, size=remove_count, replace=False
            )
            noisy = noisy.drop(index=remove_idx).reset_index(drop=True)
            n = len(noisy)

    # ========================================================
    # NOISE 4: TRIP-PURPOSE MISCLASSIFICATION
    # ========================================================
    if "trip_purpose" in noisy.columns:
        purposes = noisy["trip_purpose"].dropna().unique()
        purpose_noise_fraction = noise_level * 0.7
        purpose_noise_idx = noisy.sample(
            frac=purpose_noise_fraction, random_state=random_state + 1
        ).index
        noisy.loc[purpose_noise_idx, "trip_purpose"] = rng.choice(
            purposes, size=len(purpose_noise_idx)
        )

    # [BUG] # ========================================================
    # [BUG] # NOISE 5: MODE MISREPORTING
    # [BUG] # ========================================================
    # [BUG] if "mode" in noisy.columns:
    # [BUG]     modes = noisy["mode"].dropna().unique()
    # [BUG]     mode_noise_fraction = noise_level * 0.6
    # [BUG]     mode_noise_idx = noisy.sample(
    # [BUG]         frac=mode_noise_fraction, random_state=random_state + 2
    # [BUG]     ).index
    # [BUG]     noisy.loc[mode_noise_idx, "mode"] = rng.choice(
    # [BUG]         modes, size=len(mode_noise_idx)
    # [BUG]     )
    # [FIX] Removed: 'mode' is only used to build the primary-mode LABEL (no model uses it as an
    #       input), so this was label noise. It also swapped modes completely at random
    #       (e.g. walk -> intercity train).

    # ========================================================
    # NOISE 6: DEPARTURE-TIME RECALL ERROR
    # ========================================================
    if "departure_time_hhmm" in noisy.columns:
        departure_minutes = noisy["departure_time_hhmm"].apply(hhmm_to_minutes)
        time_noise = rng.normal(
            loc=0, scale=noise_level * 150, size=n
        )
        departure_minutes = departure_minutes + time_noise
        departure_minutes = departure_minutes.clip(0, 1439)
        noisy["departure_time_hhmm"] = departure_minutes.apply(minutes_to_hhmm)

    # ========================================================
    # NOISE 7: TRIP-DISTANCE AND DURATION ERROR
    # ========================================================
    if "trip_distance_km" in noisy.columns:
        distance_noise = rng.normal(
            loc=1.0, scale=noise_level * 1.5, size=n
        )
        noisy["trip_distance_km"] = (
            noisy["trip_distance_km"] * distance_noise
        ).clip(lower=0.1)

    if "trip_duration_min" in noisy.columns:
        duration_noise = rng.normal(
            loc=1.0, scale=noise_level * 2.0, size=n
        )
        noisy["trip_duration_min"] = (
            noisy["trip_duration_min"] * duration_noise
        ).clip(lower=1)

    if "travel_cost_inr" in noisy.columns:
        cost_noise = rng.normal(
            loc=1.0, scale=noise_level * 1.2, size=n
        )
        noisy["travel_cost_inr"] = (
            noisy["travel_cost_inr"] * cost_noise
        ).clip(lower=0)

    # [BUG] # ========================================================
    # [BUG] # NOISE 8: OCCUPATION MISCLASSIFICATION
    # [BUG] # ========================================================
    # [BUG] if "occupation" in noisy.columns:
    # [BUG]     occupations = noisy["occupation"].dropna().unique()
    # [BUG]     occupation_noise_fraction = noise_level * 0.5
    # [BUG]     occupation_noise_idx = noisy.sample(
    # [BUG]         frac=occupation_noise_fraction, random_state=random_state + 3
    # [BUG]     ).index
    # [BUG]     noisy.loc[occupation_noise_idx, "occupation"] = rng.choice(
    # [BUG]         occupations, size=len(occupation_noise_idx)
    # [BUG]     )
    # [FIX] Removed: this noise changes a LABEL the models are scored against. Labels must stay
    #       clean (noise belongs on the travel inputs only); noising them per trip also gave one
    #       household several incomes and one person several occupations.

    return noisy


# ------------------------------------------------------------
# 6. Create multiple noise scenarios
# ------------------------------------------------------------
noise_scenarios = {
    "low": 0.03,
    "medium": 0.05,
    "high": 0.10
}

summary = []

for scenario_name, noise_level in noise_scenarios.items():
    print("\n" + "=" * 60)
    print(f"Creating {scenario_name} noise dataset")
    print(f"Noise level: {noise_level}")
    print("=" * 60)

    noisy_df = add_hts_noise(
        df=df,
        noise_level=noise_level,
        random_state=42
    )

    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"bengaluru_hts_noisy_{scenario_name}.csv"
    )

    noisy_df.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")
    print(f"Noisy dataset shape: {noisy_df.shape}")

    summary.append({
        "scenario": scenario_name,
        "noise_level": noise_level,
        "original_rows": len(df),
        "noisy_rows": len(noisy_df),
        "output_file": output_file
    })

# ------------------------------------------------------------
# 7. Save noise-summary report
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary)
summary_file = os.path.join(OUTPUT_FOLDER, "hts_noise_summary.csv")
summary_df.to_csv(summary_file, index=False)

print("\nNoise-generation completed.")
print(summary_df)


Original dataset shape: (8687, 28)
Original columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr']

Creating low noise dataset
Noise level: 0.03
Saved: output/bengaluru_hts_noisy_low.csv
Noisy dataset shape: (8662, 28)

Creating medium noise dataset
Noise level: 0.05
Saved: output/bengaluru_hts_noisy_medium.csv
Noisy dataset shape: (8645, 28)

Creating high noise dataset
Noise level: 0.1
Saved: output/bengaluru_hts_noisy_high.csv
Noisy dataset shape: (8603, 28)

Noise-generation completed.
  scenario  noise_level  original_rows  noisy_ro

In [2]:
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS & DIRECTORIES
# ------------------------------------------------------------
# [ERROR] INPUT_FILE = (
# [ERROR]     r"C:\Users\Admin\Downloads\HTS\HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
# [ERROR] )
# [FIX] Windows path does not exist on this Mac; the workbook is in the same folder as this notebook
INPUT_FILE = "HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
OUTPUT_FOLDER = "output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD & MERGE REAL SURVEY SHEETS (NOT README!)
# ------------------------------------------------------------
print("Loading survey data sheets from Excel...")
df_hh = pd.read_excel(INPUT_FILE, sheet_name="A_Households")
df_person = pd.read_excel(INPUT_FILE, sheet_name="B_Person_Roster")
df_trips = pd.read_excel(INPUT_FILE, sheet_name="E_Trips_Full")

# [BUG] # Convert expenditure bands to monthly INR
# [BUG] expenditure_map = {
# [BUG]     1: 15000,
# [BUG]     2: 25000,
# [BUG]     3: 45000,
# [BUG]     4: 75000,
# [BUG]     5: 125000,
# [BUG]     6: 200000,
# [BUG]     "< 20,000": 15000,
# [BUG]     "20,001 - 35,000": 25000,
# [BUG]     "35,001 - 60,000": 45000,
# [BUG]     "60,001 - 1,00,000": 75000,
# [BUG]     "1,00,001 - 1,50,000": 125000,
# [BUG]     "> 1,50,000": 200000,
# [BUG] }
# [BUG] if "A4_Expenditure_band" in df_hh.columns:
# [BUG]     df_hh["household_monthly_income_inr"] = (
# [BUG]         df_hh["A4_Expenditure_band"].map(expenditure_map).fillna(45000)
# [BUG]     )
# [FIX] These band names ("< 20,000" ...) do not exist in the workbook (it uses "Below ₹15,000" ...),
#       so 0 of 1,200 households matched and every household became ₹45,000. "Refused" was also
#       filled in as ₹45,000. Use the workbook's band names; leave "Refused" missing.
BAND_TO_INR = {
    "Below ₹15,000": 12000,
    "₹15,000 to ₹30,000": 22500,
    "₹30,000 to ₹60,000": 45000,
    "₹60,000 to ₹1,00,000": 80000,
    "Above ₹1,00,000": 150000,
    # "Refused / don't know" is not listed, so it stays missing (NaN)
}
df_hh["household_monthly_income_inr"] = df_hh["A4_Expenditure_band"].map(BAND_TO_INR)

# Rename coded columns to standard names expected by your noise script
df_trips = df_trips.rename(
    columns={
        "E8_Primary_mode": "mode",
        "E4_Purpose": "trip_purpose",
        "E7_Distance_km": "trip_distance_km",
        "E6_Travel_time_min": "trip_duration_min",
        "E5_Departure": "departure_time_hhmm",
        "E10_Cost_INR": "travel_cost_inr",
    }
)
df_person = df_person.rename(columns={"B4_Activity": "occupation"})

# [FIX] Mode, purpose and activity were left as numeric codes. Label them with the workbook's
#       Codebook sheet (as the 3rd cell does), so the noise steps and later cells can read them.
codebook = pd.read_excel(INPUT_FILE, sheet_name="Codebook")
def codebook_labels(variable):
    """Code -> label dictionary for one variable of the workbook's Codebook sheet."""
    rows = codebook[codebook["Variable"] == variable]
    return dict(zip(rows["Code"].astype(int), rows["Meaning"]))


df_trips["mode"] = df_trips["mode"].map(codebook_labels("Mode (C4,C5,D3,E8,E9)"))
df_trips["trip_purpose"] = df_trips["trip_purpose"].map(codebook_labels("Purpose (E4)"))
df_person["occupation"] = df_person["occupation"].map(codebook_labels("B4 Activity"))

# Merge into unified Person-Trip dataset
original = df_trips.merge(
    df_person[["HH_ID", "Person_ID", "occupation"]],
    on=["HH_ID", "Person_ID"],
    how="left",
)
original = original.merge(
    df_hh[["HH_ID", "household_monthly_income_inr"]], on="HH_ID", how="left"
)

print(f"✅ Successfully loaded real dataset! Shape: {original.shape}")


# ------------------------------------------------------------
# 3. NOISE FUNCTION
# ------------------------------------------------------------
def hhmm_to_minutes(time_value):
    try:
        hours, minutes = str(time_value).split(":")
        return int(hours) * 60 + int(minutes)
    except Exception:
        return np.nan


def minutes_to_hhmm(minutes):
    minutes = int(minutes) % (24 * 60)
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"


def add_hts_noise(df, noise_level=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    noisy = df.copy()
    n = len(noisy)

    # [BUG] # 1. Income rounding & reporting error
    # [BUG] if "household_monthly_income_inr" in noisy.columns:
    # [BUG]     income_noise = rng.normal(loc=1.0, scale=noise_level * 1.5, size=n)
    # [BUG]     noisy["household_monthly_income_inr"] = (
    # [BUG]         noisy["household_monthly_income_inr"] * income_noise
    # [BUG]     )
    # [BUG]     noisy["household_monthly_income_inr"] = (
    # [BUG]         noisy["household_monthly_income_inr"] / 500
    # [BUG]     ).round() * 500
    # [BUG]     noisy["household_monthly_income_inr"] = noisy[
    # [BUG]         "household_monthly_income_inr"
    # [BUG]     ].clip(lower=5000)
    # [BUG]
    # [BUG]     # 2. Income non-response
    # [BUG]     missing_idx = noisy.sample(
    # [BUG]         frac=noise_level * 0.6, random_state=random_state
    # [BUG]     ).index
    # [BUG]     noisy.loc[missing_idx, "household_monthly_income_inr"] = np.nan
    # [FIX] Removed: this noise changes a LABEL the models are scored against. Labels must stay
    #       clean (noise belongs on the travel inputs only); noising them per trip also gave one
    #       household several incomes and one person several occupations.

    # 3. Trip underreporting (short non-work trips dropped)
    # [FIX] This step removed no trips before: purposes were numeric codes ('4', '7'), so the
    #       words below never matched. Purposes are now Codebook names ('Shopping', 'Social').
    if "trip_purpose" in noisy.columns and "trip_distance_km" in noisy.columns:
        short_nonwork_trips = noisy[
            (noisy["trip_distance_km"] < 3)
            & (
                noisy["trip_purpose"]
                .astype(str)
                .str.lower()
                .isin(["shopping", "other", "social", "recreation", "leisure"])
            )
        ].index
        remove_count = int(len(short_nonwork_trips) * (noise_level * 1.5))
        if remove_count > 0 and len(short_nonwork_trips) > 0:
            remove_idx = rng.choice(
                short_nonwork_trips, size=remove_count, replace=False
            )
            noisy = noisy.drop(index=remove_idx).reset_index(drop=True)
            n = len(noisy)

    # [BUG] # 4. Mode misreporting
    # [BUG] if "mode" in noisy.columns:
    # [BUG]     modes = noisy["mode"].dropna().unique()
    # [BUG]     mode_idx = noisy.sample(
    # [BUG]         frac=noise_level * 0.6, random_state=random_state + 2
    # [BUG]     ).index
    # [BUG]     noisy.loc[mode_idx, "mode"] = rng.choice(modes, size=len(mode_idx))
    # [FIX] Removed: 'mode' is only used to build the primary-mode LABEL (no model uses it as an
    #       input), so this was label noise. It also swapped modes completely at random
    #       (e.g. walk -> intercity train).

    # 5. Departure-time recall error
    if "departure_time_hhmm" in noisy.columns:
        departure_mins = noisy["departure_time_hhmm"].apply(hhmm_to_minutes)
        time_noise = rng.normal(loc=0, scale=noise_level * 150, size=n)
        departure_mins = (departure_mins + time_noise).clip(0, 1439)
        noisy["departure_time_hhmm"] = departure_mins.apply(minutes_to_hhmm)

    # 6. Trip distance & duration error
    if "trip_distance_km" in noisy.columns:
        noisy["trip_distance_km"] = (
            noisy["trip_distance_km"]
            * rng.normal(loc=1.0, scale=noise_level * 1.5, size=n)
        ).clip(lower=0.1)

    if "trip_duration_min" in noisy.columns:
        noisy["trip_duration_min"] = (
            noisy["trip_duration_min"]
            * rng.normal(loc=1.0, scale=noise_level * 2.0, size=n)
        ).clip(lower=1)

    # [BUG] # 7. Occupation misclassification
    # [BUG] if "occupation" in noisy.columns:
    # [BUG]     occs = noisy["occupation"].dropna().unique()
    # [BUG]     occ_idx = noisy.sample(
    # [BUG]         frac=noise_level * 0.5, random_state=random_state + 3
    # [BUG]     ).index
    # [BUG]     noisy.loc[occ_idx, "occupation"] = rng.choice(occs, size=len(occ_idx))
    # [FIX] Removed: this noise changes a LABEL the models are scored against. Labels must stay
    #       clean (noise belongs on the travel inputs only); noising them per trip also gave one
    #       household several incomes and one person several occupations.

    return noisy


# ------------------------------------------------------------
# 4. GENERATE NOISY DATASETS
# ------------------------------------------------------------
for s_name, n_level in {"low": 0.03, "medium": 0.05, "high": 0.10}.items():
    noisy_df = add_hts_noise(original, noise_level=n_level, random_state=42)
    noisy_df.to_csv(
        os.path.join(OUTPUT_FOLDER, f"bengaluru_hts_noisy_{s_name}.csv"),
        index=False,
    )

print("✅ Noisy datasets successfully written to output/ directory!\n")

# ------------------------------------------------------------
# 5. COMPARISON REPORT
# ------------------------------------------------------------
noisy = pd.read_csv("output/bengaluru_hts_noisy_medium.csv")

print("=" * 60)
print(f"Original shape: {original.shape} | Noisy shape: {noisy.shape}")
print("=" * 60)

print("\n--- Original Income vs Noisy Income ---")
print("Original:")
print(original["household_monthly_income_inr"].describe())
print("\nNoisy (with non-response & reporting noise):")
print(noisy["household_monthly_income_inr"].describe())

print("\n--- Original Mode Distribution (%) ---")
print(original["mode"].value_counts(normalize=True).round(4) * 100)
print("\n--- Noisy Mode Distribution (%) ---")
print(noisy["mode"].value_counts(normalize=True).round(4) * 100)

print("\n--- Original Occupation Distribution (%) ---")
print(original["occupation"].value_counts(normalize=True).round(4) * 100)
print("\n--- Noisy Occupation Distribution (%) ---")
print(noisy["occupation"].value_counts(normalize=True).round(4) * 100)


Loading survey data sheets from Excel...
✅ Successfully loaded real dataset! Shape: (8687, 22)
✅ Noisy datasets successfully written to output/ directory!

Original shape: (8687, 22) | Noisy shape: (8645, 22)

--- Original Income vs Noisy Income ---
Original:
count      8335.000000
mean      50277.204559
std       38708.561638
min       12000.000000
25%       22500.000000
50%       45000.000000
75%       80000.000000
max      150000.000000
Name: household_monthly_income_inr, dtype: float64

Noisy (with non-response & reporting noise):
count      8294.000000
mean      50343.320473
std       38743.921070
min       12000.000000
25%       22500.000000
50%       45000.000000
75%       80000.000000
max      150000.000000
Name: household_monthly_income_inr, dtype: float64

--- Original Mode Distribution (%) ---
mode
Walk                             23.94
City bus                         23.25
Personal / shared two-wheeler    18.50
Auto-rickshaw                    10.30
Personal / shared car  

In [3]:
import os
import pandas as pd

# [ERROR] INPUT_FILE = (
# [ERROR]     r"C:\Users\Admin\Downloads\HTS\HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
# [ERROR] )
# [FIX] Windows path does not exist on this Mac; the workbook is in the same folder as this notebook
INPUT_FILE = "HTS_Bengaluru_Synthetic_Responses_1200.xlsx"
os.makedirs("output", exist_ok=True)

# ------------------------------------------------------------
# 1. CREATE & SAVE 'bengaluru_synthetic_person_trip_survey.csv'
# ------------------------------------------------------------
print("Saving original clean survey to CSV...")
df_hh = pd.read_excel(INPUT_FILE, sheet_name="A_Households")
df_person = pd.read_excel(INPUT_FILE, sheet_name="B_Person_Roster")
df_trips = pd.read_excel(INPUT_FILE, sheet_name="E_Trips_Full")

# Map income
exp_map = {
    "Below ₹15,000": 12000,
    "₹15,000 to ₹30,000": 22500,
    "₹30,000 to ₹60,000": 45000,
    "₹60,000 to ₹1,00,000": 80000,
    "Above ₹1,00,000": 150000,
    1: 12000,
    2: 22500,
    3: 45000,
    4: 80000,
    5: 150000,
}
# [BUG] df_hh["household_monthly_income_inr"] = (
# [BUG]     df_hh["A4_Expenditure_band"].map(exp_map).fillna(45000)
# [BUG] )
# [FIX] "Refused / don't know" was filled in as ₹45,000, i.e. counted as a known lower-middle income.
#       Leave it missing.
df_hh["household_monthly_income_inr"] = df_hh["A4_Expenditure_band"].map(exp_map)

# Map mode & occupation
# [BUG] mode_map = {
# [BUG]     1: "Walk",
# [BUG]     2: "Bicycle",
# [BUG]     3: "Two-Wheeler",
# [BUG]     4: "Two-Wheeler (Pillion)",
# [BUG]     5: "Car",
# [BUG]     6: "Car (Passenger)",
# [BUG]     7: "Auto Rickshaw",
# [BUG]     8: "App Cab",
# [BUG]     9: "BMTC Bus",
# [BUG]     10: "BMTC AC Bus",
# [BUG]     11: "Namma Metro",
# [BUG]     12: "Rail",
# [BUG] }
# [BUG] df_trips["mode"] = df_trips["E8_Primary_mode"].map(mode_map).fillna("Other")
# [FIX] This map does not follow the workbook's Codebook: from code 3 on every mode is shifted
#       (10 Auto-rickshaw -> "BMTC AC Bus", 11 City bus -> "Namma Metro", 12 School/company bus -> "Rail")
#       and codes 13-20 fall through to "Other" (all 400 Metro trips). Use the Codebook sheet.
codebook = pd.read_excel(INPUT_FILE, sheet_name="Codebook")
def codebook_labels(variable):
    """Code -> label dictionary for one variable of the workbook's Codebook sheet."""
    rows = codebook[codebook["Variable"] == variable]
    return dict(zip(rows["Code"].astype(int), rows["Meaning"]))


df_trips["mode"] = df_trips["E8_Primary_mode"].map(codebook_labels("Mode (C4,C5,D3,E8,E9)"))

# [BUG] activity_map = {
# [BUG]     1: "Employed Full-time",
# [BUG]     2: "Employed Part-time",
# [BUG]     3: "Student",
# [BUG]     4: "Homemaker",
# [BUG]     5: "Retired",
# [BUG]     6: "Unemployed",
# [BUG] }
# [BUG] df_person["occupation"] = (
# [BUG]     df_person["B4_Activity"].map(activity_map).fillna("Other")
# [BUG] )
# [FIX] Shifted by one against the Codebook: 2 Student -> "Employed Part-time", 3 Homemaker -> "Student",
#       4 Retired -> "Homemaker", 5 Unemployed -> "Retired", 6 Other -> "Unemployed".
df_person["occupation"] = df_person["B4_Activity"].map(codebook_labels("B4 Activity"))

# [BUG] df_trips["trip_purpose"] = df_trips["E4_Purpose"].astype(str)
# [FIX] Purpose was stored as code text ("4", "7"). The trip-underreporting noise looks for names such as
#       "shopping", and the later purpose features search for "work" / "school" / "shop", so nothing ever
#       matched. Use the Codebook names.
df_trips["trip_purpose"] = df_trips["E4_Purpose"].map(codebook_labels("Purpose (E4)"))
df_trips["trip_distance_km"] = pd.to_numeric(
    df_trips["E7_Distance_km"], errors="coerce"
)
df_trips["trip_duration_min"] = pd.to_numeric(
    df_trips["E6_Travel_time_min"], errors="coerce"
)
df_trips["departure_time_hhmm"] = df_trips["E5_Departure"].astype(str)
df_trips["travel_cost_inr"] = pd.to_numeric(
    df_trips["E10_Cost_INR"], errors="coerce"
)

# Merge and write to disk
clean_original = df_trips.merge(
    df_person[["HH_ID", "Person_ID", "occupation"]],
    on=["HH_ID", "Person_ID"],
    how="left",
)
clean_original = clean_original.merge(
    df_hh[["HH_ID", "household_monthly_income_inr"]], on="HH_ID", how="left"
)

clean_original.to_csv(
    "output/bengaluru_synthetic_person_trip_survey.csv", index=False
)
print("✅ File successfully saved to: output/bengaluru_synthetic_person_trip_survey.csv")

# ------------------------------------------------------------
# 2. RUN YOUR EXACT COMPARISON SCRIPT
# ------------------------------------------------------------
original = pd.read_csv("output/bengaluru_synthetic_person_trip_survey.csv")
noisy = pd.read_csv("output/bengaluru_hts_noisy_medium.csv")

print("\n" + "=" * 55)
print("Original shape:", original.shape)
print("Noisy shape:", noisy.shape)
print("=" * 55)

print("\n--- Original Income ---")
print(original["household_monthly_income_inr"].describe())

print("\n--- Noisy Income ---")
print(noisy["household_monthly_income_inr"].describe())

print("\n--- Original Mode Distribution (%) ---")
print(original["mode"].value_counts(normalize=True).round(4) * 100)

print("\n--- Noisy Mode Distribution (%) ---")
print(noisy["mode"].value_counts(normalize=True).round(4) * 100)

print("\n--- Original Occupation Distribution (%) ---")
print(original["occupation"].value_counts(normalize=True).round(4) * 100)

print("\n--- Noisy Occupation Distribution (%) ---")
print(noisy["occupation"].value_counts(normalize=True).round(4) * 100)


Saving original clean survey to CSV...
✅ File successfully saved to: output/bengaluru_synthetic_person_trip_survey.csv

Original shape: (8687, 28)
Noisy shape: (8645, 22)

--- Original Income ---
count      8335.000000
mean      50277.204559
std       38708.561638
min       12000.000000
25%       22500.000000
50%       45000.000000
75%       80000.000000
max      150000.000000
Name: household_monthly_income_inr, dtype: float64

--- Noisy Income ---
count      8294.000000
mean      50343.320473
std       38743.921070
min       12000.000000
25%       22500.000000
50%       45000.000000
75%       80000.000000
max      150000.000000
Name: household_monthly_income_inr, dtype: float64

--- Original Mode Distribution (%) ---
mode
Walk                             23.94
City bus                         23.25
Personal / shared two-wheeler    18.50
Auto-rickshaw                    10.30
Personal / shared car             6.22
School / company bus              6.17
Metro                            

In [4]:
# Re-apply noise directly to the text-labeled clean dataset
# [BUG] noisy = add_hts_noise(original, noise_level=0.05, random_state=42)
# [BUG] noisy.to_csv("output/bengaluru_hts_noisy_medium.csv", index=False)
# [FIX] Only the medium file was rebuilt from the labelled data; low and high stayed as written by the
#       2nd cell. Rebuild all three from the same labelled table.
for level_name, level in {"low": 0.03, "medium": 0.05, "high": 0.10}.items():
    add_hts_noise(original, noise_level=level, random_state=42).to_csv(
        f"output/bengaluru_hts_noisy_{level_name}.csv", index=False
    )
noisy = pd.read_csv("output/bengaluru_hts_noisy_medium.csv")

print("\n--- Clean Mode vs Noisy Mode (With Text Labels) ---")
comp = pd.DataFrame(
    {
        "Original (%)": (original["mode"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str)
        + "%",
        "Noisy (%)": (noisy["mode"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str)
        + "%",
    }
)
print(comp)



--- Clean Mode vs Noisy Mode (With Text Labels) ---
                              Original (%) Noisy (%)
mode                                                
Walk                                23.94%    23.77%
City bus                            23.25%    23.26%
Personal / shared two-wheeler        18.5%    18.58%
Auto-rickshaw                        10.3%    10.29%
Personal / shared car                6.22%     6.25%
School / company bus                 6.17%      6.2%
Metro                                 4.6%     4.63%
Ride-hail car                        1.97%     1.97%
Ride-hail two-wheeler                 1.9%     1.91%
Taxi                                 1.53%     1.54%
Shared auto                          0.93%     0.93%
Bicycle                              0.66%     0.66%
Suburban rail                        0.02%     0.02%


In [5]:
# ============================================================
# MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC AND MODE INFERENCE
# ============================================================
# Input : Synthetic Bengaluru person-trip HTS dataset
# Output: Predict occupation, income group, vehicle ownership,
#         and travel mode from person-level mobility graphs
#
# Split:
#   Training   = 70%
#   Validation = 10%
#   Testing    = 20%
#
# Models:
#   1. Logistic Regression
#   2. Random Forest
#   3. Extra Trees
#   4. XGBoost
#   5. LightGBM
#   6. CatBoost
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. File path
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/mobility_graph_ml_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
# [FIX] KeyError 'person_id' / 'home_zone': the noisy file has HH_ID, Person_ID and E2_Origin_PIN,
#       not the person_id / home_zone columns this draft expects.
#       Person_ID restarts at 1 in every household, so the person key needs HH_ID too.
df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
# [FIX] home zone = origin PIN of each person's first trip (every diary day starts at home)
df["home_zone"] = df["person_id"].map(
    df.loc[df["Trip_no_full"] == 1].set_index("person_id")["E2_Origin_PIN"]
)
# [FIX] destination zone = destination PIN of each trip
df["destination_zone"] = df["E3_Dest_PIN"]

print("Loaded dataset:", df.shape)
print("Columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. Convert departure time to minutes
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan


df["departure_time_min"] = df["departure_time_hhmm"].apply(
    hhmm_to_minutes
)


# ------------------------------------------------------------
# 3. Build person-level mobility-graph features
# ------------------------------------------------------------
def build_mobility_graph_features(person_trips):
    """
    Creates mobility-graph features for one person.

    Mobility graph:
    - Nodes = visited zones
    - Edges = movements between zones
    - Edge weight = number of trips between zones
    """

    features = {}

    person_trips = person_trips.sort_values(
        by=["departure_time_min"]
    )

    # --------------------------------------------------------
    # Basic graph size
    # --------------------------------------------------------
    unique_zones = pd.concat([
        person_trips["home_zone"],
        person_trips["destination_zone"]
        if "destination_zone" in person_trips.columns
        else pd.Series(dtype=object)
    ]).dropna().unique()

    features["graph_num_nodes"] = len(unique_zones)
    features["graph_num_trips"] = len(person_trips)

    # --------------------------------------------------------
    # Edge construction: home_zone -> destination_zone
    # --------------------------------------------------------
    if "destination_zone" in person_trips.columns:

        edge_counts = person_trips.groupby(
            ["home_zone", "destination_zone"]
        ).size()

        features["graph_num_edges"] = len(edge_counts)
        features["graph_total_edge_weight"] = edge_counts.sum()
        features["graph_max_edge_weight"] = edge_counts.max()
        features["graph_mean_edge_weight"] = edge_counts.mean()

        # Graph density
        possible_edges = len(unique_zones) * (len(unique_zones) - 1)
        features["graph_density"] = (
            len(edge_counts) / possible_edges
            if possible_edges > 0 else 0
        )

    else:
        features["graph_num_edges"] = 0
        features["graph_total_edge_weight"] = 0
        features["graph_max_edge_weight"] = 0
        features["graph_mean_edge_weight"] = 0
        features["graph_density"] = 0

    # --------------------------------------------------------
    # Home-zone dominance
    # --------------------------------------------------------
    home_trip_count = len(person_trips)

    features["home_zone_dominance"] = 1.0
    features["num_unique_destination_zones"] = (
        person_trips["destination_zone"].nunique()
        if "destination_zone" in person_trips.columns
        else 0
    )

    # --------------------------------------------------------
    # Activity entropy
    # --------------------------------------------------------
    if "destination_zone" in person_trips.columns:

        zone_counts = person_trips["destination_zone"].value_counts(
            normalize=True
        )

        entropy = -np.sum(
            zone_counts * np.log(zone_counts + 1e-9)
        )

        features["destination_entropy"] = entropy
    else:
        features["destination_entropy"] = 0

    # --------------------------------------------------------
    # Trip-distance features
    # --------------------------------------------------------
    features["mean_trip_distance"] = person_trips[
        "trip_distance_km"
    ].mean()

    features["max_trip_distance"] = person_trips[
        "trip_distance_km"
    ].max()

    features["std_trip_distance"] = person_trips[
        "trip_distance_km"
    ].std()

    # --------------------------------------------------------
    # Trip-duration features
    # --------------------------------------------------------
    features["mean_trip_duration"] = person_trips[
        "trip_duration_min"
    ].mean()

    features["max_trip_duration"] = person_trips[
        "trip_duration_min"
    ].max()

    # --------------------------------------------------------
    # Travel-cost features
    # --------------------------------------------------------
    features["mean_travel_cost"] = person_trips[
        "travel_cost_inr"
    ].mean()

    # --------------------------------------------------------
    # Temporal mobility features
    # --------------------------------------------------------
    features["mean_departure_time"] = person_trips[
        "departure_time_min"
    ].mean()

    features["std_departure_time"] = person_trips[
        "departure_time_min"
    ].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360)
        & (person_trips["departure_time_min"] < 720)
    )

    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720)
        & (person_trips["departure_time_min"] < 1020)
    )

    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1200)
    )

    # --------------------------------------------------------
    # Mode-use features
    # --------------------------------------------------------
    mode_counts = person_trips["mode"].value_counts(
        normalize=True
    )

    for mode_name, share in mode_counts.items():
        safe_name = str(mode_name).replace(" ", "_").replace("-", "_")
        features[f"mode_share_{safe_name}"] = share

    # --------------------------------------------------------
    # Trip-purpose features
    # --------------------------------------------------------
    purpose_counts = person_trips["trip_purpose"].value_counts(
        normalize=True
    )

    for purpose_name, share in purpose_counts.items():
        safe_name = str(purpose_name).replace(" ", "_").replace("-", "_")
        features[f"purpose_share_{safe_name}"] = share

    # --------------------------------------------------------
    # Labels: use first available value for person-level targets
    # --------------------------------------------------------
    label_columns = [
        "occupation",
        "household_monthly_income_inr",
        "owns_car",
        "owns_two_wheeler",
        "mode"
    ]

    for label in label_columns:
        if label in person_trips.columns:
            features[f"label_{label}"] = person_trips[label].iloc[0]

    return features


# ------------------------------------------------------------
# 4. Apply graph-feature generation to every person
# ------------------------------------------------------------
person_features = []

for person_id, person_trips in df.groupby("person_id"):

    graph_features = build_mobility_graph_features(person_trips)
    graph_features["person_id"] = person_id

    person_features.append(graph_features)

person_df = pd.DataFrame(person_features)

print("\nPerson-level mobility-graph dataset:", person_df.shape)


# ------------------------------------------------------------
# 5. Create income-class label
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 25000:
        return "low"
    elif income < 50000:
        return "lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"


person_df["income_class"] = person_df[
    "label_household_monthly_income_inr"
].apply(income_to_class)


# ------------------------------------------------------------
# 6. Define prediction tasks
# ------------------------------------------------------------
TARGETS = {
    "occupation": "label_occupation",
    "income_class": "income_class",
    # [ERROR]     "car_ownership": "label_owns_car",
    # [ERROR]     "two_wheeler_ownership": "label_owns_two_wheeler",
    # [FIX] KeyError 'label_owns_car': the noisy file has no vehicle-ownership columns, so these
    #       labels are never created. Targets removed (the 7th cell onward skips them the same way).
    "primary_mode": "label_mode"
}

# Remove label columns from feature matrix
label_columns = [
    col for col in person_df.columns
    if col.startswith("label_")
] + ["income_class"]

feature_columns = [
    col for col in person_df.columns
    if col not in label_columns + ["person_id"]
]

# Fill missing values in numeric features
person_df[feature_columns] = person_df[feature_columns].fillna(
    person_df[feature_columns].median(numeric_only=True)
)

# Fill missing categorical values
for col in feature_columns:
    if person_df[col].dtype == "object":
        person_df[col] = person_df[col].fillna("unknown")


# ------------------------------------------------------------
# 7. Train-validation-test split: 70 / 10 / 20
# ------------------------------------------------------------
# [BUG] def split_data(X, y):
# [BUG]
# [BUG]     # First split: 70% train, 30% temporary
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X,
# [BUG]         y,
# [BUG]         test_size=0.30,
# [BUG]         stratify=y,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     # Second split: 10% validation, 20% test
# [BUG]     # 10 / 30 = 0.3333
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp,
# [BUG]         y_temp,
# [BUG]         test_size=0.6667,
# [BUG]         stratify=y_temp,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_data(X, y, groups):
    return household_split(X, y, groups)


# ------------------------------------------------------------
# 8. Define six ML models
# ------------------------------------------------------------
def get_models():

    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier

        models["xgboost"] = XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        )

    except ImportError:
        print("XGBoost not installed.")

    try:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    except ImportError:
        print("LightGBM not installed.")

    try:
        from catboost import CatBoostClassifier

        models["catboost"] = CatBoostClassifier(
            iterations=400,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=0,
            auto_class_weights="Balanced"
        )

    except ImportError:
        print("CatBoost not installed.")

    return models


# ------------------------------------------------------------
# 9. Train and evaluate models for each target
# ------------------------------------------------------------
final_results = []
best_models = {}

for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    data = person_df.dropna(
        subset=[target_column]
    ).reset_index(drop=True)

    X = data[feature_columns]
    y = data[target_column]

    # Encode categorical target labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        X,
        y_encoded,
        groups=household_of(data)  # [FIX] split by household
    )

    # Scale numeric features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()

    validation_results = []

    for model_name, model in models.items():

        print(f"\nTraining: {model_name}")

        try:
            model.fit(X_train_scaled, y_train)

            val_pred = model.predict(X_val_scaled)

            val_macro_f1 = f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )

            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })

            print(
                f"Validation macro F1: {val_macro_f1:.4f}"
            )

        except Exception as error:
            print(f"Failed to train {model_name}: {error}")

    # Select best model using validation macro F1
    validation_df = pd.DataFrame(validation_results)
    best_model_name = validation_df.sort_values(
        by="validation_macro_f1",
        ascending=False
    ).iloc[0]["model"]

    best_model = models[best_model_name]

    print(
        f"\nBest model for {task_name}: {best_model_name}"
    )

    # Final evaluation on test set
    test_pred = best_model.predict(X_test_scaled)

    test_accuracy = accuracy_score(y_test, test_pred)
    test_macro_f1 = f1_score(
        y_test,
        test_pred,
        average="macro",
        zero_division=0
    )
    test_balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_pred
    )

    print(f"Test accuracy: {test_accuracy:.4f}")
    print(f"Test macro F1: {test_macro_f1:.4f}")
    print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            test_pred,
            # [FIX] household split: a rare class can be missing from the test set
            labels=np.arange(len(label_encoder.classes_)),
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        test_pred,
        labels=range(len(label_encoder.classes_))
    )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(
        f"Test Confusion Matrix: {task_name}\nBest model: {best_model_name}"
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    cm_file = os.path.join(
        OUTPUT_FOLDER,
        f"confusion_matrix_{task_name}.png"
    )

    plt.savefig(cm_file, dpi=300)
    plt.close()

    final_results.append({
        "task": task_name,
        "best_model": best_model_name,
        "validation_macro_f1": validation_df[
            validation_df["model"] == best_model_name
        ]["validation_macro_f1"].iloc[0],
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_balanced_accuracy": test_balanced_accuracy
    })

    best_models[task_name] = {
        "model": best_model,
        "scaler": scaler,
        "label_encoder": label_encoder,
        "features": feature_columns
    }

# ------------------------------------------------------------
# 10. Save final results
# ------------------------------------------------------------
results_df = pd.DataFrame(final_results)

results_file = os.path.join(
    OUTPUT_FOLDER,
    "mobility_graph_model_results.csv"
)

results_df.to_csv(results_file, index=False)

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(results_df)

print(f"\nResults saved to: {results_file}")

Loaded dataset: (8645, 31)
Columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr', 'person_id', 'home_zone', 'destination_zone']

Person-level mobility-graph dataset: (3383, 50)

TASK: occupation
CatBoost not installed.

Training: logistic_regression
Validation macro F1: 0.4333

Training: random_forest
Validation macro F1: 0.4107

Training: extra_trees
Validation macro F1: 0.4120

Training: xgboost
Validation macro F1: 0.4163

Training: lightgbm
Validation macro F1: 0.4074

Best model for occupation: logistic_regression
Test accuracy: 0.5

In [6]:
# ============================================================
# FINAL CORRECTED MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC
# AND SOCIOECONOMIC INFERENCE MODEL
# ============================================================
# Input : Synthetic Bengaluru person-trip HTS dataset
# Output: Predict occupation, income class, vehicle ownership,
#         and person-level primary mode from mobility graphs
#
# Correct logic:
# - Mobility graph features = input (X)
# - Sociodemographic / socioeconomic / primary mode = labels (y)
# - Person-level split: 70% train, 10% validation, 20% test
# - No mode leakage
# - Validation used for model selection
# - Test set used only once
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/final_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
# [FIX] KeyError 'person_id' / 'home_zone': the noisy file has HH_ID, Person_ID and E2_Origin_PIN,
#       not the person_id / home_zone columns this draft expects.
#       Person_ID restarts at 1 in every household, so the person key needs HH_ID too.
df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
# [FIX] home zone = origin PIN of each person's first trip (every diary day starts at home)
df["home_zone"] = df["person_id"].map(
    df.loc[df["Trip_no_full"] == 1].set_index("person_id")["E2_Origin_PIN"]
)

print("Loaded dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. STANDARDIZE DESTINATION COLUMN NAME
# ------------------------------------------------------------
# [ERROR] possible_destination_columns = [
# [ERROR]     "destination_zone",
# [ERROR]     "destination",
# [ERROR]     "dest_zone",
# [ERROR]     "trip_destination"
# [ERROR] ]
# [FIX] ValueError 'No destination-zone column found': the file names it E3_Dest_PIN
possible_destination_columns = [
    "destination_zone",
    "E3_Dest_PIN",
    "destination",
    "dest_zone",
    "trip_destination"
]

destination_column = None

for col in possible_destination_columns:
    if col in df.columns:
        destination_column = col
        break

if destination_column is None:
    raise ValueError(
        "No destination-zone column found. "
        "Expected one of: destination_zone, destination, "
        "dest_zone, trip_destination."
    )

df = df.rename(columns={destination_column: "destination_zone"})

print("\nDestination column standardized to: destination_zone")


# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan


df["departure_time_min"] = df["departure_time_hhmm"].apply(
    hhmm_to_minutes
)


# ------------------------------------------------------------
# 4. BUILD PERSON-LEVEL MOBILITY GRAPH FEATURES
# ------------------------------------------------------------
def build_mobility_graph_features(person_trips):
    """
    Builds one mobility graph per person.

    Graph structure:
    - Nodes = visited zones
    - Edges = movements between zones
    - Edge weight = number of trips between zones
    """

    person_trips = person_trips.sort_values(
        by=["departure_time_min"]
    )

    features = {}

    # --------------------------------------------------------
    # Graph nodes
    # --------------------------------------------------------
    visited_zones = pd.concat([
        person_trips["home_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = len(person_trips)

    # --------------------------------------------------------
    # Graph edges: home_zone -> destination_zone
    # --------------------------------------------------------
    edge_counts = person_trips.groupby(
        ["home_zone", "destination_zone"]
    ).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max()
    features["graph_mean_edge_weight"] = edge_counts.mean()

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)

    features["graph_density"] = (
        len(edge_counts) / possible_edges
        if possible_edges > 0
        else 0
    )

    # --------------------------------------------------------
    # Destination diversity and entropy
    # --------------------------------------------------------
    destination_counts = person_trips[
        "destination_zone"
    ].value_counts(normalize=True)

    features["num_unique_destinations"] = len(destination_counts)

    destination_entropy = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )

    features["destination_entropy"] = destination_entropy

    # --------------------------------------------------------
    # Distance features
    # --------------------------------------------------------
    features["mean_trip_distance"] = person_trips[
        "trip_distance_km"
    ].mean()

    features["max_trip_distance"] = person_trips[
        "trip_distance_km"
    ].max()

    features["std_trip_distance"] = person_trips[
        "trip_distance_km"
    ].std()

    # --------------------------------------------------------
    # Duration features
    # --------------------------------------------------------
    features["mean_trip_duration"] = person_trips[
        "trip_duration_min"
    ].mean()

    features["max_trip_duration"] = person_trips[
        "trip_duration_min"
    ].max()

    # --------------------------------------------------------
    # Cost features
    # --------------------------------------------------------
    features["mean_travel_cost"] = person_trips[
        "travel_cost_inr"
    ].mean()

    # --------------------------------------------------------
    # Temporal mobility features
    # --------------------------------------------------------
    features["mean_departure_time"] = person_trips[
        "departure_time_min"
    ].mean()

    features["std_departure_time"] = person_trips[
        "departure_time_min"
    ].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360)
        & (person_trips["departure_time_min"] < 720)
    )

    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720)
        & (person_trips["departure_time_min"] < 1020)
    )

    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1200)
    )

    # --------------------------------------------------------
    # Trip-purpose features
    # --------------------------------------------------------
    purpose_counts = person_trips["trip_purpose"].value_counts(
        normalize=True
    )

    for purpose, share in purpose_counts.items():
        safe_name = str(purpose).replace(" ", "_").replace("-", "_")
        features[f"purpose_share_{safe_name}"] = share

    # --------------------------------------------------------
    # Person-level labels
    # --------------------------------------------------------
    features["label_occupation"] = person_trips["occupation"].iloc[0]

    features["label_income"] = person_trips[
        "household_monthly_income_inr"
    ].iloc[0]

# [ERROR]     features["label_owns_car"] = person_trips["owns_car"].iloc[0]
# [ERROR]
# [ERROR]     features["label_owns_two_wheeler"] = person_trips[
# [ERROR]         "owns_two_wheeler"
# [ERROR]     ].iloc[0]
    # [FIX] KeyError 'owns_car': the noisy file has no vehicle-ownership columns.
    #       Only add these labels when present (as the 8th cell does).
    if "owns_car" in person_trips.columns:
        features["label_owns_car"] = person_trips["owns_car"].iloc[0]
    if "owns_two_wheeler" in person_trips.columns:
        features["label_owns_two_wheeler"] = person_trips["owns_two_wheeler"].iloc[0]

    # --------------------------------------------------------
    # Person-level primary mode
    # --------------------------------------------------------
    # Primary mode = most frequently used mode by the person.
    # Mode is NOT used as an input feature.

    primary_mode = person_trips["mode"].value_counts().idxmax()
    features["label_primary_mode"] = primary_mode

    return features


# ------------------------------------------------------------
# 5. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):

    graph_features = build_mobility_graph_features(person_trips)
    graph_features["person_id"] = person_id

    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)

print("\nPerson-level mobility-graph dataset:", person_df.shape)


# ------------------------------------------------------------
# 6. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 25000:
        return "low"
    elif income < 50000:
        return "lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"


person_df["label_income_class"] = person_df[
    "label_income"
].apply(income_to_class)


# ------------------------------------------------------------
# 7. DEFINE TRANSFERABLE GRAPH-FEATURE SET
# ------------------------------------------------------------
# These features do not use mode, so the trained model can later
# be applied to GPS, telecom, or other crowdsourced mobility graphs.

base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "mean_trip_distance",
    "max_trip_distance",
    "std_trip_distance",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share"
]

purpose_features = [
    col for col in person_df.columns
    if col.startswith("purpose_share_")
]

FEATURE_COLUMNS = base_graph_features + purpose_features

# Keep only available columns
FEATURE_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col in person_df.columns
]

# Fill missing numeric values
numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(
    include=[np.number]
).columns

person_df[numeric_features] = person_df[numeric_features].fillna(
    person_df[numeric_features].median()
)

# Fill missing categorical values, if any
for col in FEATURE_COLUMNS:
    if person_df[col].dtype == "object":
        person_df[col] = person_df[col].fillna("unknown")


# ------------------------------------------------------------
# 8. DEFINE PREDICTION TASKS
# ------------------------------------------------------------
TARGETS = {
    "occupation": "label_occupation",
    "income_class": "label_income_class",
    # [ERROR]     "car_ownership": "label_owns_car",
    # [ERROR]     "two_wheeler_ownership": "label_owns_two_wheeler",
    # [FIX] KeyError 'label_owns_car': the noisy file has no vehicle-ownership columns, so these
    #       labels are never created. Targets removed (the 7th cell onward skips them the same way).
    "primary_mode": "label_primary_mode"
}


# ------------------------------------------------------------
# 9. PERSON-LEVEL 70 / 10 / 20 SPLIT
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]
# [BUG]     # Step 1: 70% train, 30% temporary
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X,
# [BUG]         y,
# [BUG]         test_size=0.30,
# [BUG]         stratify=y,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     # Step 2: split temporary 30% into:
# [BUG]     # 10% validation and 20% test
# [BUG]     # Validation share within temporary = 10 / 30 = 1/3
# [BUG]     # Test share within temporary = 20 / 30 = 2/3
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp,
# [BUG]         y_temp,
# [BUG]         test_size=0.6667,
# [BUG]         stratify=y_temp,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)


# ------------------------------------------------------------
# 10. DEFINE SIX ML MODELS
# ------------------------------------------------------------
def get_models():

    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier

        models["xgboost"] = XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        )

    except ImportError:
        print("XGBoost is not installed.")

    try:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    except ImportError:
        print("LightGBM is not installed.")

    try:
        from catboost import CatBoostClassifier

        models["catboost"] = CatBoostClassifier(
            iterations=400,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=0,
            auto_class_weights="Balanced"
        )

    except ImportError:
        print("CatBoost is not installed.")

    return models


# ------------------------------------------------------------
# 11. TRAIN, VALIDATE, SELECT BEST MODEL, AND TEST
# ------------------------------------------------------------
final_results = []

for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(
        subset=[target_column]
    ).reset_index(drop=True)

    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    # Encode categorical target labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # Person-level 70 / 10 / 20 split
    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(
        X,
        y_encoded,
        groups=household_of(task_data)  # [FIX] split by household
    )

    # Scale features using training data only
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()

    validation_results = []

    for model_name, model in models.items():

        print(f"\nTraining model: {model_name}")

        try:
            model.fit(X_train_scaled, y_train)

            val_pred = model.predict(X_val_scaled)

            val_macro_f1 = f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )

            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })

            print(
                f"Validation macro F1: {val_macro_f1:.4f}"
            )

        except Exception as error:
            print(f"Failed to train {model_name}: {error}")

    if len(validation_results) == 0:
        print(f"No models trained for {task_name}. Skipping.")
        continue

    validation_df = pd.DataFrame(validation_results)

    best_model_name = validation_df.sort_values(
        by="validation_macro_f1",
        ascending=False
    ).iloc[0]["model"]

    best_model = models[best_model_name]

    print(
        f"\nBest model for {task_name}: {best_model_name}"
    )

    # Final evaluation on untouched test set
    test_pred = best_model.predict(X_test_scaled)

    test_accuracy = accuracy_score(y_test, test_pred)

    test_macro_f1 = f1_score(
        y_test,
        test_pred,
        average="macro",
        zero_division=0
    )

    test_balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_pred
    )

    print(f"Test accuracy: {test_accuracy:.4f}")
    print(f"Test macro F1: {test_macro_f1:.4f}")
    print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            test_pred,
            # [FIX] household split: a rare class can be missing from the test set
            labels=np.arange(len(label_encoder.classes_)),
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        test_pred,
        labels=range(len(label_encoder.classes_))
    )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )

    plt.title(
        f"Test Confusion Matrix: {task_name}\nBest model: {best_model_name}"
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    confusion_file = os.path.join(
        OUTPUT_FOLDER,
        f"confusion_matrix_{task_name}.png"
    )

    plt.savefig(confusion_file, dpi=300)
    plt.close()

    final_results.append({
        "task": task_name,
        "best_model": best_model_name,
        "validation_macro_f1": validation_df[
            validation_df["model"] == best_model_name
        ]["validation_macro_f1"].iloc[0],
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_balanced_accuracy": test_balanced_accuracy
    })

# ------------------------------------------------------------
# 12. SAVE FINAL RESULTS
# ------------------------------------------------------------
results_df = pd.DataFrame(final_results)

results_file = os.path.join(
    OUTPUT_FOLDER,
    "mobility_graph_model_results.csv"
)

results_df.to_csv(results_file, index=False)

# Save person-level graph-feature dataset
person_feature_file = os.path.join(
    OUTPUT_FOLDER,
    "person_level_mobility_graph_features.csv"
)

person_df.to_csv(person_feature_file, index=False)

print("\n" + "=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)
print(results_df)

print(f"\nPerson-level graph features saved to: {person_feature_file}")
print(f"Model results saved to: {results_file}")

Loaded dataset shape: (8645, 30)
Available columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr', 'person_id', 'home_zone']

Destination column standardized to: destination_zone

Person-level mobility-graph dataset: (3383, 36)

TASK: occupation
CatBoost is not installed.

Training model: logistic_regression
Validation macro F1: 0.4346

Training model: random_forest
Validation macro F1: 0.4087

Training model: extra_trees
Validation macro F1: 0.4245

Training model: xgboost
Validation macro F1: 0.4136

Training model: lightgbm
Validation 

In [7]:
# ============================================================
# FINAL CORRECTED MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC
# AND SOCIOECONOMIC INFERENCE MODEL
# ============================================================
# Input : Synthetic Bengaluru person-trip HTS dataset
# Output: Predict occupation, income class, vehicle ownership,
#         and person-level primary mode from mobility graphs
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS & LOADING
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/final_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Loaded dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. STANDARDIZE COLUMN NAMES (ORIGIN, DESTINATION, PERSON_ID)
# ------------------------------------------------------------
# Standardize Person ID
# [ERROR] if "person_id" not in df.columns:
# [ERROR]     for col in ["Person_ID", "person_id", "HH_Person_ID"]:
# [ERROR]         if col in df.columns:
# [ERROR]             df["person_id"] = df[col]
# [ERROR]             break
# [FIX] ValueError 'n_samples=3': Person_ID alone runs 1-8 inside every household, so all
#       3,383 travellers were merged into 8 'people'. The person key needs HH_ID too.
df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)

# Standardize Origin Zone
origin_col_candidates = ["origin_zone", "home_zone", "E2_Origin_PIN", "E2_Origin_type", "origin"]
origin_col = next((c for c in origin_col_candidates if c in df.columns), None)
if origin_col:
    df["origin_zone"] = df[origin_col]
else:
    raise ValueError(f"No origin-zone column found. Expected one of: {origin_col_candidates}")

# Standardize Destination Zone
dest_col_candidates = ["destination_zone", "E3_Dest_PIN", "E3_Destination_name", "destination", "dest_zone"]
dest_col = next((c for c in dest_col_candidates if c in df.columns), None)
if dest_col:
    df["destination_zone"] = df[dest_col]
else:
    raise ValueError(f"No destination-zone column found. Expected one of: {dest_col_candidates}")

print(f"\nStandardized:\n  Person ID    -> 'person_id'\n  Origin zone  -> 'origin_zone' (from '{origin_col}')\n  Dest zone    -> 'destination_zone' (from '{dest_col}')")

# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan

df["departure_time_min"] = df["departure_time_hhmm"].apply(hhmm_to_minutes)

# ------------------------------------------------------------
# 4. BUILD PERSON-LEVEL MOBILITY GRAPH FEATURES
# ------------------------------------------------------------
def build_mobility_graph_features(person_trips):
    """
    Builds one mobility graph per person:
    - Nodes: Visited zones (origins and destinations)
    - Edges: Directed movements between zones (origin -> destination)
    - Weights: Number of trips between zone pairs
    """
    person_trips = person_trips.sort_values(by=["departure_time_min"])
    features = {}

    # Graph nodes
    visited_zones = pd.concat([
        person_trips["origin_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = len(person_trips)

    # Graph edges: origin_zone -> destination_zone
    edge_counts = person_trips.groupby(["origin_zone", "destination_zone"]).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max() if len(edge_counts) > 0 else 0
    features["graph_mean_edge_weight"] = edge_counts.mean() if len(edge_counts) > 0 else 0

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)
    features["graph_density"] = (
        len(edge_counts) / possible_edges if possible_edges > 0 else 0
    )

    # Destination diversity and entropy
    destination_counts = person_trips["destination_zone"].value_counts(normalize=True)
    features["num_unique_destinations"] = len(destination_counts)
    features["destination_entropy"] = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )

    # Distance features
    features["mean_trip_distance"] = person_trips["trip_distance_km"].mean()
    features["max_trip_distance"] = person_trips["trip_distance_km"].max()
    features["std_trip_distance"] = person_trips["trip_distance_km"].std()

    # Duration features
    features["mean_trip_duration"] = person_trips["trip_duration_min"].mean()
    features["max_trip_duration"] = person_trips["trip_duration_min"].max()

    # Cost features
    features["mean_travel_cost"] = person_trips["travel_cost_inr"].mean()

    # Temporal mobility features
    features["mean_departure_time"] = person_trips["departure_time_min"].mean()
    features["std_departure_time"] = person_trips["departure_time_min"].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360) & (person_trips["departure_time_min"] < 720)
    )
    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720) & (person_trips["departure_time_min"] < 1020)
    )
    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020) & (person_trips["departure_time_min"] < 1200)
    )

    # Trip-purpose features
    if "trip_purpose" in person_trips.columns:
        purpose_counts = person_trips["trip_purpose"].value_counts(normalize=True)
        for purpose, share in purpose_counts.items():
            safe_name = str(purpose).replace(" ", "_").replace("-", "_")
            features[f"purpose_share_{safe_name}"] = share

    # Person-level sociodemographic labels (only if present in dataset)
    if "occupation" in person_trips.columns:
        features["label_occupation"] = person_trips["occupation"].iloc[0]

    if "household_monthly_income_inr" in person_trips.columns:
        features["label_income"] = person_trips["household_monthly_income_inr"].iloc[0]

    if "owns_car" in person_trips.columns:
        features["label_owns_car"] = person_trips["owns_car"].iloc[0]

    if "owns_two_wheeler" in person_trips.columns:
        features["label_owns_two_wheeler"] = person_trips["owns_two_wheeler"].iloc[0]

    # Person-level primary mode (target only, NOT used as feature)
    if "mode" in person_trips.columns:
        features["label_primary_mode"] = person_trips["mode"].value_counts().idxmax()

    return features

# ------------------------------------------------------------
# 5. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):
    graph_features = build_mobility_graph_features(person_trips)
    graph_features["person_id"] = person_id
    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)
print("\nPerson-level mobility-graph dataset shape:", person_df.shape)

# ------------------------------------------------------------
# 6. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 25000:
        return "low"
    elif income < 50000:
        return "lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"

if "label_income" in person_df.columns:
    person_df["label_income_class"] = person_df["label_income"].apply(income_to_class)

# ------------------------------------------------------------
# 7. DEFINE GRAPH-FEATURE SET
# ------------------------------------------------------------
base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "mean_trip_distance",
    "max_trip_distance",
    "std_trip_distance",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share"
]

purpose_features = [col for col in person_df.columns if col.startswith("purpose_share_")]
FEATURE_COLUMNS = [col for col in (base_graph_features + purpose_features) if col in person_df.columns]

# Impute missing values
numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(include=[np.number]).columns
person_df[numeric_features] = person_df[numeric_features].fillna(0.0)

for col in FEATURE_COLUMNS:
    if person_df[col].dtype == "object":
        person_df[col] = person_df[col].fillna("unknown")

# ------------------------------------------------------------
# 8. DEFINE PREDICTION TASKS (DYNAMIC)
# ------------------------------------------------------------
possible_targets = {
    "occupation": "label_occupation",
    "income_class": "label_income_class",
    "car_ownership": "label_owns_car",
    "two_wheeler_ownership": "label_owns_two_wheeler",
    "primary_mode": "label_primary_mode"
}

TARGETS = {
    task: col for task, col in possible_targets.items()
    if col in person_df.columns and person_df[col].dropna().nunique() > 1
}

print("\nActive Prediction Tasks:", list(TARGETS.keys()))

# ------------------------------------------------------------
# 9. SAFE TRAIN / VALIDATION / TEST SPLIT (70 / 10 / 20)
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]     min_count = pd.Series(y).value_counts().min()
# [BUG]     stratify_1 = y if min_count >= 2 else None
# [BUG]
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X, y, test_size=0.30, stratify=stratify_1, random_state=42
# [BUG]     )
# [BUG]
# [BUG]     min_count_temp = pd.Series(y_temp).value_counts().min()
# [BUG]     stratify_2 = y_temp if min_count_temp >= 2 else None
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp, y_temp, test_size=0.6667, stratify=stratify_2, random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)

# ------------------------------------------------------------
# 10. DEFINE ML MODELS
# ------------------------------------------------------------
def get_models():
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier
        models["xgboost"] = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, random_state=42, eval_metric="mlogloss", n_jobs=-1
        )
    except ImportError:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight="balanced",
            random_state=42, n_jobs=-1, verbose=-1
        )
    except ImportError:
        pass

    try:
        from catboost import CatBoostClassifier
        models["catboost"] = CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0
        )
    except ImportError:
        pass

    return models

# ------------------------------------------------------------
# 11. TRAIN, VALIDATE, SELECT BEST MODEL, AND TEST
# ------------------------------------------------------------
final_results = []

for task_name, target_column in TARGETS.items():
    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(subset=[target_column]).reset_index(drop=True)
    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(X, y_encoded, groups=household_of(task_data))  # [FIX] split by household

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()
    validation_results = []

    for model_name, model in models.items():
        print(f"Training: {model_name}...")
        try:
            model.fit(X_train_scaled, y_train)
            val_pred = np.asarray(model.predict(X_val_scaled)).ravel()
            val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })
            print(f"  Validation macro F1: {val_macro_f1:.4f}")
        except Exception as error:
            print(f"  Failed to train {model_name}: {error}")

    if not validation_results:
        print(f"No models succeeded for {task_name}.")
        continue

    validation_df = pd.DataFrame(validation_results)
    best_model_name = validation_df.sort_values(
        by="validation_macro_f1", ascending=False
    ).iloc[0]["model"]
    best_model = models[best_model_name]

    print(f"\n>> Best model for {task_name}: {best_model_name}")

    test_pred = np.asarray(best_model.predict(X_test_scaled)).ravel()
    test_accuracy = accuracy_score(y_test, test_pred)
    test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
    test_balanced_accuracy = balanced_accuracy_score(y_test, test_pred)

    print(f"Test Accuracy:          {test_accuracy:.4f}")
    print(f"Test Macro F1:          {test_macro_f1:.4f}")
    print(f"Test Balanced Accuracy: {test_balanced_accuracy:.4f}")

    print("\nClassification report:")
    print(classification_report(y_test, test_pred, 
        # [FIX] household split: a rare class can be missing from the test set
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_test, test_pred, labels=range(len(label_encoder.classes_)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(f"Test Confusion Matrix: {task_name}\nBest model: {best_model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    confusion_file = os.path.join(OUTPUT_FOLDER, f"confusion_matrix_{task_name}.png")
    plt.savefig(confusion_file, dpi=300)
    plt.close()

    final_results.append({
        "task": task_name,
        "best_model": best_model_name,
        "validation_macro_f1": validation_df[validation_df["model"] == best_model_name]["validation_macro_f1"].iloc[0],
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_balanced_accuracy": test_balanced_accuracy
    })

# ------------------------------------------------------------
# 12. SAVE FINAL RESULTS
# ------------------------------------------------------------
results_df = pd.DataFrame(final_results)
results_file = os.path.join(OUTPUT_FOLDER, "mobility_graph_model_results.csv")
results_df.to_csv(results_file, index=False)

person_feature_file = os.path.join(OUTPUT_FOLDER, "person_level_mobility_graph_features.csv")
person_df.to_csv(person_feature_file, index=False)

print("\n" + "=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)
print(results_df)
print(f"\nSaved feature table: {person_feature_file}")
print(f"Saved results summary: {results_file}")


Loaded dataset shape: (8645, 28)
Available columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr']

Standardized:
  Person ID    -> 'person_id'
  Origin zone  -> 'origin_zone' (from 'E2_Origin_PIN')
  Dest zone    -> 'destination_zone' (from 'E3_Dest_PIN')

Person-level mobility-graph dataset shape: (3383, 36)

Active Prediction Tasks: ['occupation', 'income_class', 'primary_mode']

TASK: occupation
Training: logistic_regression...
  Validation macro F1: 0.4820
Training: random_forest...
  Validation macro F1: 0.4566
Training: extra_trees

In [8]:
# ============================================================
# FINAL CORRECTED MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC
# AND SOCIOECONOMIC INFERENCE MODEL
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS & LOADING
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/final_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Loaded dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. STANDARDIZE COLUMN NAMES (UNIQUE PERSON, ORIGIN, DEST)
# ------------------------------------------------------------
# In HTS, Person_ID is member 1, 2, 3 within household.
# A unique person requires HH_ID + Person_ID.
if "HH_ID" in df.columns and "Person_ID" in df.columns:
    df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
elif "Person_ID" in df.columns:
    df["person_id"] = df["Person_ID"].astype(str)
else:
    df["person_id"] = df.index.astype(str)

# Standardize Origin Zone
origin_col_candidates = ["origin_zone", "home_zone", "E2_Origin_PIN", "E2_Origin_type", "origin"]
origin_col = next((c for c in origin_col_candidates if c in df.columns), None)
if origin_col:
    df["origin_zone"] = df[origin_col]
else:
    raise ValueError("No origin-zone column found.")

# Standardize Destination Zone
dest_col_candidates = ["destination_zone", "E3_Dest_PIN", "E3_Destination_name", "destination", "dest_zone"]
dest_col = next((c for c in dest_col_candidates if c in df.columns), None)
if dest_col:
    df["destination_zone"] = df[dest_col]
else:
    raise ValueError("No destination-zone column found.")

print(f"Total trips: {len(df)}")
print(f"Total unique individuals identified: {df['person_id'].nunique()}")

# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan

df["departure_time_min"] = df["departure_time_hhmm"].apply(hhmm_to_minutes)

# ------------------------------------------------------------
# 4. BUILD PERSON-LEVEL MOBILITY GRAPH FEATURES
# ------------------------------------------------------------
def build_mobility_graph_features(person_trips):
    person_trips = person_trips.sort_values(by=["departure_time_min"])
    features = {}

    # Graph nodes
    visited_zones = pd.concat([
        person_trips["origin_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = len(person_trips)

    # Graph edges: origin_zone -> destination_zone
    edge_counts = person_trips.groupby(["origin_zone", "destination_zone"]).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max() if len(edge_counts) > 0 else 0
    features["graph_mean_edge_weight"] = edge_counts.mean() if len(edge_counts) > 0 else 0

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)
    features["graph_density"] = (
        len(edge_counts) / possible_edges if possible_edges > 0 else 0
    )

    # Destination diversity and entropy
    destination_counts = person_trips["destination_zone"].value_counts(normalize=True)
    features["num_unique_destinations"] = len(destination_counts)
    features["destination_entropy"] = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )

    # Distance features
    features["mean_trip_distance"] = person_trips["trip_distance_km"].mean()
    features["max_trip_distance"] = person_trips["trip_distance_km"].max()
    features["std_trip_distance"] = person_trips["trip_distance_km"].std()

    # Duration features
    features["mean_trip_duration"] = person_trips["trip_duration_min"].mean()
    features["max_trip_duration"] = person_trips["trip_duration_min"].max()

    # Cost features
    features["mean_travel_cost"] = person_trips["travel_cost_inr"].mean()

    # Temporal mobility features
    features["mean_departure_time"] = person_trips["departure_time_min"].mean()
    features["std_departure_time"] = person_trips["departure_time_min"].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360) & (person_trips["departure_time_min"] < 720)
    )
    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720) & (person_trips["departure_time_min"] < 1020)
    )
    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020) & (person_trips["departure_time_min"] < 1200)
    )

    # Trip-purpose features
    if "trip_purpose" in person_trips.columns:
        purpose_counts = person_trips["trip_purpose"].value_counts(normalize=True)
        for purpose, share in purpose_counts.items():
            safe_name = str(purpose).replace(" ", "_").replace("-", "_")
            features[f"purpose_share_{safe_name}"] = share

    # Person-level labels
    if "occupation" in person_trips.columns:
        features["label_occupation"] = person_trips["occupation"].iloc[0]

    if "household_monthly_income_inr" in person_trips.columns:
        features["label_income"] = person_trips["household_monthly_income_inr"].iloc[0]

    if "owns_car" in person_trips.columns:
        features["label_owns_car"] = person_trips["owns_car"].iloc[0]

    if "owns_two_wheeler" in person_trips.columns:
        features["label_owns_two_wheeler"] = person_trips["owns_two_wheeler"].iloc[0]

    if "mode" in person_trips.columns:
        features["label_primary_mode"] = person_trips["mode"].value_counts().idxmax()

    return features

# ------------------------------------------------------------
# 5. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):
    graph_features = build_mobility_graph_features(person_trips)
    graph_features["person_id"] = person_id
    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)
print("\nPerson-level mobility-graph dataset shape:", person_df.shape)

# ------------------------------------------------------------
# 6. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 25000:
        return "low"
    elif income < 50000:
        return "lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"

if "label_income" in person_df.columns:
    person_df["label_income_class"] = person_df["label_income"].apply(income_to_class)

# ------------------------------------------------------------
# 7. DEFINE GRAPH-FEATURE SET
# ------------------------------------------------------------
base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "mean_trip_distance",
    "max_trip_distance",
    "std_trip_distance",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share"
]

purpose_features = [col for col in person_df.columns if col.startswith("purpose_share_")]
FEATURE_COLUMNS = [col for col in (base_graph_features + purpose_features) if col in person_df.columns]

# Impute missing values
numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(include=[np.number]).columns
person_df[numeric_features] = person_df[numeric_features].fillna(0.0)

for col in FEATURE_COLUMNS:
    if person_df[col].dtype == "object":
        person_df[col] = person_df[col].fillna("unknown")

# ------------------------------------------------------------
# 8. DEFINE PREDICTION TASKS
# ------------------------------------------------------------
possible_targets = {
    "occupation": "label_occupation",
    "income_class": "label_income_class",
    "car_ownership": "label_owns_car",
    "two_wheeler_ownership": "label_owns_two_wheeler",
    "primary_mode": "label_primary_mode"
}

TARGETS = {
    task: col for task, col in possible_targets.items()
    if col in person_df.columns and person_df[col].dropna().nunique() > 1
}

print("\nActive Prediction Tasks:", list(TARGETS.keys()))

# ------------------------------------------------------------
# 9. INTEGER-SAFE TRAIN / VALIDATION / TEST SPLIT (70 / 10 / 20)
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]     n_samples = len(X)
# [BUG]
# [BUG]     # Exact sample counts for 70% train, 10% val, 20% test
# [BUG]     n_val = max(1, int(round(0.10 * n_samples)))
# [BUG]     n_test = max(1, int(round(0.20 * n_samples)))
# [BUG]     n_temp = n_val + n_test
# [BUG]
# [BUG]     # Stratify check for split 1
# [BUG]     class_counts = pd.Series(y).value_counts()
# [BUG]     use_stratify_1 = y if class_counts.min() >= 2 else None
# [BUG]
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X, y,
# [BUG]         test_size=n_temp,
# [BUG]         stratify=use_stratify_1,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     # Stratify check for split 2
# [BUG]     temp_counts = pd.Series(y_temp).value_counts()
# [BUG]     use_stratify_2 = y_temp if temp_counts.min() >= 2 else None
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp, y_temp,
# [BUG]         test_size=n_test,
# [BUG]         stratify=use_stratify_2,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)

# ------------------------------------------------------------
# 10. DEFINE ML MODELS
# ------------------------------------------------------------
def get_models():
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier
        models["xgboost"] = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, random_state=42, eval_metric="mlogloss", n_jobs=-1
        )
    except ImportError:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight="balanced",
            random_state=42, n_jobs=-1, verbose=-1
        )
    except ImportError:
        pass

    try:
        from catboost import CatBoostClassifier
        models["catboost"] = CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0
        )
    except ImportError:
        pass

    return models

# ------------------------------------------------------------
# 11. TRAIN, VALIDATE, SELECT BEST MODEL, AND TEST
# ------------------------------------------------------------
final_results = []

for task_name, target_column in TARGETS.items():
    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(subset=[target_column]).reset_index(drop=True)
    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(X, y_encoded, groups=household_of(task_data))  # [FIX] split by household
    print(f"Data split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()
    validation_results = []

    for model_name, model in models.items():
        print(f"Training: {model_name}...")
        try:
            model.fit(X_train_scaled, y_train)
            val_pred = np.asarray(model.predict(X_val_scaled)).ravel()
            val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })
            print(f"  Validation macro F1: {val_macro_f1:.4f}")
        except Exception as error:
            print(f"  Failed to train {model_name}: {error}")

    if not validation_results:
        print(f"No models succeeded for {task_name}.")
        continue

    validation_df = pd.DataFrame(validation_results)
    best_model_name = validation_df.sort_values(
        by="validation_macro_f1", ascending=False
    ).iloc[0]["model"]
    best_model = models[best_model_name]

    print(f"\n>> Best model for {task_name}: {best_model_name}")

    test_pred = np.asarray(best_model.predict(X_test_scaled)).ravel()
    test_accuracy = accuracy_score(y_test, test_pred)
    test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
    test_balanced_accuracy = balanced_accuracy_score(y_test, test_pred)

    print(f"Test Accuracy:          {test_accuracy:.4f}")
    print(f"Test Macro F1:          {test_macro_f1:.4f}")
    print(f"Test Balanced Accuracy: {test_balanced_accuracy:.4f}")

    print("\nClassification report:")
    print(classification_report(y_test, test_pred, 
        # [FIX] household split: a rare class can be missing from the test set
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_test, test_pred, labels=range(len(label_encoder.classes_)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(f"Test Confusion Matrix: {task_name}\nBest model: {best_model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    confusion_file = os.path.join(OUTPUT_FOLDER, f"confusion_matrix_{task_name}.png")
    plt.savefig(confusion_file, dpi=300)
    plt.close()

    final_results.append({
        "task": task_name,
        "best_model": best_model_name,
        "validation_macro_f1": validation_df[validation_df["model"] == best_model_name]["validation_macro_f1"].iloc[0],
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_balanced_accuracy": test_balanced_accuracy
    })

# ------------------------------------------------------------
# 12. SAVE FINAL RESULTS
# ------------------------------------------------------------
results_df = pd.DataFrame(final_results)
results_file = os.path.join(OUTPUT_FOLDER, "mobility_graph_model_results.csv")
results_df.to_csv(results_file, index=False)

person_feature_file = os.path.join(OUTPUT_FOLDER, "person_level_mobility_graph_features.csv")
person_df.to_csv(person_feature_file, index=False)

print("\n" + "=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)
print(results_df)
print(f"\nSaved feature table: {person_feature_file}")
print(f"Saved results summary: {results_file}")


Loaded dataset shape: (8645, 28)
Total trips: 8645
Total unique individuals identified: 3383

Person-level mobility-graph dataset shape: (3383, 36)

Active Prediction Tasks: ['occupation', 'income_class', 'primary_mode']

TASK: occupation
Data split: Train=2375, Val=334, Test=674
Training: logistic_regression...
  Validation macro F1: 0.4820
Training: random_forest...
  Validation macro F1: 0.4566
Training: extra_trees...
  Validation macro F1: 0.4676
Training: xgboost...
  Validation macro F1: 0.4466
Training: lightgbm...
  Validation macro F1: 0.4254

>> Best model for occupation: logistic_regression
Test Accuracy:          0.6528
Test Macro F1:          0.4072
Test Balanced Accuracy: 0.4243

Classification report:
              precision    recall  f1-score   support

    Employed       1.00      0.78      0.88       339
   Homemaker       0.65      0.28      0.39       131
       Other       0.03      0.21      0.06        14
     Retired       0.11      0.36      0.17        28
  

In [9]:
 # ============================================================
# FULL ML MODEL COMPARISON FOR ALL TASKS
# ============================================================
# This code evaluates all trained models on the test set
# and creates a complete comparison table and charts.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Output folder
# ------------------------------------------------------------
COMPARISON_FOLDER = "output/final_mobility_graph_results/model_comparison"
os.makedirs(COMPARISON_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# 2. Store all model comparison results
# ------------------------------------------------------------
all_model_results = []

# ------------------------------------------------------------
# 3. Evaluate all models for each task
# ------------------------------------------------------------
for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"FULL MODEL COMPARISON: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(
        subset=[target_column]
    ).reset_index(drop=True)

    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(
        X,
        y_encoded,
        groups=household_of(task_data)  # [FIX] split by household (function from the 8th cell)
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()

    for model_name, model in models.items():

        print(f"\nEvaluating: {model_name}")

        try:
            # Train on training data
            model.fit(X_train_scaled, y_train)

            # Validation performance
            val_pred = model.predict(X_val_scaled)

            val_macro_f1 = f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )

            # Test performance
            test_pred = model.predict(X_test_scaled)

            test_accuracy = accuracy_score(y_test, test_pred)
            test_macro_f1 = f1_score(
                y_test,
                test_pred,
                average="macro",
                zero_division=0
            )
            test_balanced_accuracy = balanced_accuracy_score(
                y_test,
                test_pred
            )

            print(f"Validation macro F1: {val_macro_f1:.4f}")
            print(f"Test accuracy: {test_accuracy:.4f}")
            print(f"Test macro F1: {test_macro_f1:.4f}")
            print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")

            # Save result
            all_model_results.append({
                "task": task_name,
                "model": model_name,
                "validation_macro_f1": val_macro_f1,
                "test_accuracy": test_accuracy,
                "test_macro_f1": test_macro_f1,
                "test_balanced_accuracy": test_balanced_accuracy
            })

            # Confusion matrix
            cm = confusion_matrix(
                y_test,
                test_pred,
                labels=range(len(label_encoder.classes_))
            )

            plt.figure(figsize=(9, 7))
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_
            )

            plt.title(
                f"{task_name}\nModel: {model_name}"
            )
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()

            confusion_file = os.path.join(
                COMPARISON_FOLDER,
                f"confusion_matrix_{task_name}_{model_name}.png"
            )

            plt.savefig(confusion_file, dpi=300)
            plt.close()

        except Exception as error:
            print(f"Failed to evaluate {model_name}: {error}")

# ------------------------------------------------------------
# 4. Create comparison DataFrame
# ------------------------------------------------------------
comparison_df = pd.DataFrame(all_model_results)

comparison_file = os.path.join(
    COMPARISON_FOLDER,
    "all_ml_model_comparison_results.csv"
)

comparison_df.to_csv(comparison_file, index=False)

print("\n" + "=" * 70)
print("FULL MODEL COMPARISON RESULTS")
print("=" * 70)
print(comparison_df)

print(f"\nComparison results saved to: {comparison_file}")

# ------------------------------------------------------------
# 5. Create comparison charts
# ------------------------------------------------------------
for task_name in comparison_df["task"].unique():

    task_results = comparison_df[
        comparison_df["task"] == task_name
    ].sort_values(
        by="validation_macro_f1",
        ascending=False
    )

    # Validation macro F1 chart
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=task_results,
        x="model",
        y="validation_macro_f1",
        palette="viridis"
    )
    plt.title(
        f"Validation Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Validation Macro F1")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    validation_chart = os.path.join(
        COMPARISON_FOLDER,
        f"validation_macro_f1_{task_name}.png"
    )

    plt.savefig(validation_chart, dpi=300)
    plt.close()

    # Test macro F1 chart
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=task_results,
        x="model",
        y="test_macro_f1",
        palette="mako"
    )
    plt.title(
        f"Test Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Test Macro F1")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    test_chart = os.path.join(
        COMPARISON_FOLDER,
        f"test_macro_f1_{task_name}.png"
    )

    plt.savefig(test_chart, dpi=300)
    plt.close()

print("\nAll comparison charts saved successfully.")


FULL MODEL COMPARISON: occupation

Evaluating: logistic_regression
Validation macro F1: 0.4820
Test accuracy: 0.6528
Test macro F1: 0.4072
Test balanced accuracy: 0.4243

Evaluating: random_forest
Validation macro F1: 0.4566
Test accuracy: 0.7671
Test macro F1: 0.4430
Test balanced accuracy: 0.4545

Evaluating: extra_trees
Validation macro F1: 0.4676
Test accuracy: 0.7196
Test macro F1: 0.4316
Test balanced accuracy: 0.4333

Evaluating: xgboost
Validation macro F1: 0.4466
Test accuracy: 0.7596
Test macro F1: 0.4359
Test balanced accuracy: 0.4315

Evaluating: lightgbm
Validation macro F1: 0.4254
Test accuracy: 0.7552
Test macro F1: 0.4468
Test balanced accuracy: 0.4408

FULL MODEL COMPARISON: income_class

Evaluating: logistic_regression
Validation macro F1: 0.2180
Test accuracy: 0.2903
Test macro F1: 0.2706
Test balanced accuracy: 0.2978

Evaluating: random_forest
Validation macro F1: 0.3018
Test accuracy: 0.3652
Test macro F1: 0.2675
Test balanced accuracy: 0.2799

Evaluating: extra_

In [10]:
# ============================================================
# FULL ML MODEL COMPARISON FOR ALL TASKS (CLEAN - NO WARNINGS)
# ============================================================

import os
import sys
import warnings

# 1. Completely silence all warnings (including child worker threads)
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Output folder
# ------------------------------------------------------------
COMPARISON_FOLDER = "output/final_mobility_graph_results/model_comparison"
os.makedirs(COMPARISON_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# 2. Define models with n_jobs=1 (Eliminates parallel warnings)
# ------------------------------------------------------------
def get_clean_models():
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=1  # Eliminates joblib parallel warning completely
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=1  # Eliminates joblib parallel warning completely
        )
    }

    try:
        from xgboost import XGBClassifier
        models["xgboost"] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=1
        )
    except ImportError:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
            verbose=-1
        )
    except ImportError:
        pass

    try:
        from catboost import CatBoostClassifier
        models["catboost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=0
        )
    except ImportError:
        pass

    return models

# ------------------------------------------------------------
# 3. Evaluate all models for each task
# ------------------------------------------------------------
all_model_results = []

for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"FULL MODEL COMPARISON: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(subset=[target_column]).reset_index(drop=True)

    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(X, y_encoded, groups=household_of(task_data))  # [FIX] split by household (function from the 8th cell)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_clean_models()

    for model_name, model in models.items():

        print(f"\nEvaluating: {model_name}...")

        try:
            # Train
            model.fit(X_train_scaled, y_train)

            # Validation
            val_pred = np.asarray(model.predict(X_val_scaled)).ravel()
            val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)

            # Test
            test_pred = np.asarray(model.predict(X_test_scaled)).ravel()
            test_accuracy = accuracy_score(y_test, test_pred)
            test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
            test_balanced_accuracy = balanced_accuracy_score(y_test, test_pred)

            print(f"  -> Validation Macro F1: {val_macro_f1:.4f}")
            print(f"  -> Test Accuracy:       {test_accuracy:.4f}")
            print(f"  -> Test Macro F1:       {test_macro_f1:.4f}")
            print(f"  -> Test Balanced Acc:   {test_balanced_accuracy:.4f}")

            all_model_results.append({
                "task": task_name,
                "model": model_name,
                "validation_macro_f1": val_macro_f1,
                "test_accuracy": test_accuracy,
                "test_macro_f1": test_macro_f1,
                "test_balanced_accuracy": test_balanced_accuracy
            })

            # Confusion matrix
            cm = confusion_matrix(y_test, test_pred, labels=range(len(label_encoder.classes_)))

            plt.figure(figsize=(8, 6))
            sns.heatmap(
                cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_
            )
            plt.title(f"{task_name} | {model_name}")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()

            confusion_file = os.path.join(
                COMPARISON_FOLDER,
                f"confusion_matrix_{task_name}_{model_name}.png"
            )
            plt.savefig(confusion_file, dpi=300)
            plt.close()

        except Exception as error:
            print(f"  Failed to evaluate {model_name}: {error}")

# ------------------------------------------------------------
# 4. Save comparison CSV
# ------------------------------------------------------------
comparison_df = pd.DataFrame(all_model_results)
comparison_file = os.path.join(COMPARISON_FOLDER, "all_ml_model_comparison_results.csv")
comparison_df.to_csv(comparison_file, index=False)

print("\n" + "=" * 70)
print("FULL MODEL COMPARISON SUMMARY")
print("=" * 70)
print(comparison_df.to_string(index=False))
print(f"\nSaved CSV to: {comparison_file}")

# ------------------------------------------------------------
# 5. Generate and save comparison bar charts
# ------------------------------------------------------------
for task_name in comparison_df["task"].unique():
    task_results = comparison_df[comparison_df["task"] == task_name].sort_values(
        by="validation_macro_f1", ascending=False
    )

    # Validation F1 Chart
    plt.figure(figsize=(10, 5))
    sns.barplot(data=task_results, x="model", y="validation_macro_f1", palette="viridis")
    plt.title(f"Validation Macro F1 Comparison | Task: {task_name}")
    plt.xlabel("ML Model")
    plt.ylabel("Validation Macro F1")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_FOLDER, f"validation_macro_f1_{task_name}.png"), dpi=300)
    plt.close()

    # Test F1 Chart
    plt.figure(figsize=(10, 5))
    sns.barplot(data=task_results, x="model", y="test_macro_f1", palette="mako")
    plt.title(f"Test Macro F1 Comparison | Task: {task_name}")
    plt.xlabel("ML Model")
    plt.ylabel("Test Macro F1")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_FOLDER, f"test_macro_f1_{task_name}.png"), dpi=300)
    plt.close()

print("\nAll comparison charts and confusion matrices successfully saved!")



FULL MODEL COMPARISON: occupation

Evaluating: logistic_regression...
  -> Validation Macro F1: 0.4820
  -> Test Accuracy:       0.6528
  -> Test Macro F1:       0.4072
  -> Test Balanced Acc:   0.4243

Evaluating: random_forest...
  -> Validation Macro F1: 0.4566
  -> Test Accuracy:       0.7671
  -> Test Macro F1:       0.4430
  -> Test Balanced Acc:   0.4545

Evaluating: extra_trees...
  -> Validation Macro F1: 0.4676
  -> Test Accuracy:       0.7196
  -> Test Macro F1:       0.4316
  -> Test Balanced Acc:   0.4333

Evaluating: xgboost...
  -> Validation Macro F1: 0.4466
  -> Test Accuracy:       0.7596
  -> Test Macro F1:       0.4359
  -> Test Balanced Acc:   0.4315

Evaluating: lightgbm...
  -> Validation Macro F1: 0.4254
  -> Test Accuracy:       0.7552
  -> Test Macro F1:       0.4468
  -> Test Balanced Acc:   0.4408

FULL MODEL COMPARISON: income_class

Evaluating: logistic_regression...
  -> Validation Macro F1: 0.2180
  -> Test Accuracy:       0.2903
  -> Test Macro F1:    

In [11]:
# ============================================================
# COMPLETE IMPROVED MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC
# AND SOCIOECONOMIC INFERENCE PIPELINE
# ============================================================
# Input : Noisy synthetic Bengaluru person-trip HTS dataset
# Output: Predict simplified occupation, income class,
#         and simplified primary mode from mobility graphs
#
# Correct logic:
# - Mobility graph features = input (X)
# - Sociodemographic / socioeconomic / primary mode = labels (y)
# - Person-level split: 70% train, 10% validation, 20% test
# - No mode leakage
# - Validation used for model selection
# - Test set used only once
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/improved_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
# [FIX] KeyError 'person_id' / 'home_zone': the noisy file has HH_ID, Person_ID and E2_Origin_PIN,
#       not the person_id / home_zone columns this draft expects.
#       Person_ID restarts at 1 in every household, so the person key needs HH_ID too.
df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
# [FIX] home zone = origin PIN of each person's first trip (every diary day starts at home)
df["home_zone"] = df["person_id"].map(
    df.loc[df["Trip_no_full"] == 1].set_index("person_id")["E2_Origin_PIN"]
)

print("Loaded dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. STANDARDIZE DESTINATION COLUMN NAME
# ------------------------------------------------------------
# [ERROR] possible_destination_columns = [
# [ERROR]     "destination_zone",
# [ERROR]     "destination",
# [ERROR]     "dest_zone",
# [ERROR]     "trip_destination"
# [ERROR] ]
# [FIX] ValueError 'No destination-zone column found': the file names it E3_Dest_PIN
possible_destination_columns = [
    "destination_zone",
    "E3_Dest_PIN",
    "destination",
    "dest_zone",
    "trip_destination"
]

destination_column = None

for col in possible_destination_columns:
    if col in df.columns:
        destination_column = col
        break

if destination_column is None:
    raise ValueError(
        "No destination-zone column found. "
        "Expected one of: destination_zone, destination, "
        "dest_zone, trip_destination."
    )

df = df.rename(columns={destination_column: "destination_zone"})

print("\nDestination column standardized to: destination_zone")


# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan


df["departure_time_min"] = df["departure_time_hhmm"].apply(
    hhmm_to_minutes
)


# ------------------------------------------------------------
# 4. SIMPLIFY RARE OCCUPATION AND MODE CLASSES
# ------------------------------------------------------------
def simplify_occupation(occupation):
    occupation = str(occupation).strip().lower()

    # [BUG] if "employ" in occupation:
    # [BUG]     return "employed"
    # [BUG] elif "student" in occupation or "child" in occupation:
    # [BUG]     return "student_child"
    # [BUG] elif "home" in occupation:
    # [BUG]     return "homemaker"
    # [BUG] elif "retire" in occupation or "unemploy" in occupation:
    # [BUG]     return "non_employed"
    # [BUG] else:
    # [BUG]     return "other"
    # [FIX] "unemployed" contains "employ", so every Unemployed person was counted as employed.
    #       Check unemployed / retired first.
    if "unemploy" in occupation or "retire" in occupation:
        return "non_employed"
    elif "employ" in occupation:
        return "employed"
    elif "student" in occupation or "child" in occupation:
        return "student_child"
    elif "home" in occupation:
        return "homemaker"
    else:
        return "other"


def simplify_mode(mode):
    mode = str(mode).strip().lower()
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in mode for word in ['ride-hail', 'taxi', 'rickshaw']):
        return "paid_private_vehicle"

    if "walk" in mode or "bicycle" in mode:
        return "active_transport"
    elif "metro" in mode or "rail" in mode:
        return "rail_transit"
    elif "bmtc" in mode or "bus" in mode:
        return "bus"
    elif "two" in mode:
        return "two_wheeler"
    elif "car" in mode:
        return "private_car"
    elif "auto" in mode or "app cab" in mode:
        return "paid_private_vehicle"
    else:
        return "other"


df["occupation_simplified"] = df["occupation"].apply(
    simplify_occupation
)

df["mode_simplified"] = df["mode"].apply(
    simplify_mode
)

print("\nSimplified occupation distribution:")
print(df["occupation_simplified"].value_counts(normalize=True) * 100)

print("\nSimplified mode distribution:")
print(df["mode_simplified"].value_counts(normalize=True) * 100)


# ------------------------------------------------------------
# 5. BUILD IMPROVED PERSON-LEVEL MOBILITY-GRAPH FEATURES
# ------------------------------------------------------------
def build_improved_mobility_graph_features(person_trips):

    person_trips = person_trips.sort_values(
        by=["departure_time_min"]
    )

    features = {}
    total_trips = len(person_trips)

    # --------------------------------------------------------
    # Graph nodes
    # --------------------------------------------------------
    visited_zones = pd.concat([
        person_trips["home_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = total_trips

    # --------------------------------------------------------
    # Graph edges: home_zone -> destination_zone
    # --------------------------------------------------------
    edge_counts = person_trips.groupby(
        ["home_zone", "destination_zone"]
    ).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max()
    features["graph_mean_edge_weight"] = edge_counts.mean()

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)

    features["graph_density"] = (
        len(edge_counts) / possible_edges
        if possible_edges > 0
        else 0
    )

    # --------------------------------------------------------
    # Destination diversity and entropy
    # --------------------------------------------------------
    destination_counts = person_trips[
        "destination_zone"
    ].value_counts(normalize=True)

    features["num_unique_destinations"] = len(destination_counts)

    features["destination_entropy"] = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )

    features["max_destination_share"] = destination_counts.max()

    # --------------------------------------------------------
    # Distance features
    # --------------------------------------------------------
    features["mean_trip_distance"] = person_trips[
        "trip_distance_km"
    ].mean()

    features["max_trip_distance"] = person_trips[
        "trip_distance_km"
    ].max()

    features["std_trip_distance"] = person_trips[
        "trip_distance_km"
    ].std()

    features["short_trip_share"] = np.mean(
        person_trips["trip_distance_km"] < 3
    )

    features["medium_trip_share"] = np.mean(
        (person_trips["trip_distance_km"] >= 3)
        & (person_trips["trip_distance_km"] < 10)
    )

    features["long_trip_share"] = np.mean(
        person_trips["trip_distance_km"] >= 10
    )

    # --------------------------------------------------------
    # Duration and cost features
    # --------------------------------------------------------
    features["mean_trip_duration"] = person_trips[
        "trip_duration_min"
    ].mean()

    features["max_trip_duration"] = person_trips[
        "trip_duration_min"
    ].max()

    features["mean_travel_cost"] = person_trips[
        "travel_cost_inr"
    ].mean()

    # --------------------------------------------------------
    # Temporal mobility features
    # --------------------------------------------------------
    features["mean_departure_time"] = person_trips[
        "departure_time_min"
    ].mean()

    features["std_departure_time"] = person_trips[
        "departure_time_min"
    ].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360)
        & (person_trips["departure_time_min"] < 720)
    )

    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720)
        & (person_trips["departure_time_min"] < 1020)
    )

    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1200)
    )

    features["peak_morning_share"] = np.mean(
        (person_trips["departure_time_min"] >= 480)
        & (person_trips["departure_time_min"] < 600)
    )

    features["peak_evening_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1140)
    )

    # --------------------------------------------------------
    # Trip-purpose features
    # --------------------------------------------------------
    purpose_counts = person_trips["trip_purpose"].value_counts(
        normalize=True
    )

    features["purpose_entropy"] = -np.sum(
        purpose_counts * np.log(purpose_counts + 1e-9)
    )

    work_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("work", na=False)
    ]

    features["work_trip_share"] = len(work_trips) / total_trips

    features["mean_work_distance"] = (
        work_trips["trip_distance_km"].mean()
        if len(work_trips) > 0
        else 0
    )

    education_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("education|school|college", na=False)
    ]

    features["education_trip_share"] = (
        len(education_trips) / total_trips
    )

    features["mean_education_distance"] = (
        education_trips["trip_distance_km"].mean()
        if len(education_trips) > 0
        else 0
    )

    shopping_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("shop", na=False)
    ]

    features["shopping_trip_share"] = (
        len(shopping_trips) / total_trips
    )

    # --------------------------------------------------------
    # Return-home feature
    # --------------------------------------------------------
    home_return_trips = person_trips[
        person_trips["destination_zone"] == person_trips["home_zone"]
    ]

    features["return_home_share"] = (
        len(home_return_trips) / total_trips
    )

    # --------------------------------------------------------
    # Person-level labels
    # --------------------------------------------------------
    features["label_occupation_simplified"] = person_trips[
        "occupation_simplified"
    ].iloc[0]

    features["label_income"] = person_trips[
        "household_monthly_income_inr"
    ].iloc[0]

    features["label_primary_mode_simplified"] = person_trips[
        "mode_simplified"
    ].value_counts().idxmax()

    return features


# ------------------------------------------------------------
# 6. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):

    graph_features = build_improved_mobility_graph_features(
        person_trips
    )

    graph_features["person_id"] = person_id

    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)

print("\nImproved person-level mobility-graph dataset:", person_df.shape)


# ------------------------------------------------------------
# 7. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 50000:
        return "low_or_lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"


person_df["label_income_class"] = person_df[
    "label_income"
].apply(income_to_class)


# ------------------------------------------------------------
# 8. DEFINE TRANSFERABLE GRAPH-FEATURE SET
# ------------------------------------------------------------
base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "max_destination_share",
    "mean_trip_distance",
    "max_trip_distance",
    "std_trip_distance",
    "short_trip_share",
    "medium_trip_share",
    "long_trip_share",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share",
    "peak_morning_share",
    "peak_evening_share",
    "purpose_entropy",
    "work_trip_share",
    "mean_work_distance",
    "education_trip_share",
    "mean_education_distance",
    "shopping_trip_share",
    "return_home_share"
]

purpose_features = [
    col for col in person_df.columns
    if col.startswith("purpose_share_")
]

FEATURE_COLUMNS = base_graph_features + purpose_features

FEATURE_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col in person_df.columns
]

numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(
    include=[np.number]
).columns

person_df[numeric_features] = person_df[numeric_features].fillna(
    person_df[numeric_features].median()
)


# ------------------------------------------------------------
# 9. DEFINE PREDICTION TASKS
# ------------------------------------------------------------
TARGETS = {
    "occupation": "label_occupation_simplified",
    "income_class": "label_income_class",
    "primary_mode": "label_primary_mode_simplified"
}


# ------------------------------------------------------------
# 10. PERSON-LEVEL 70 / 10 / 20 SPLIT
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]
# [BUG]     # Step 1: 70% train, 30% temporary
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X,
# [BUG]         y,
# [BUG]         test_size=0.30,
# [BUG]         stratify=y,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     # Step 2: split 30% temporary into:
# [BUG]     # 10% validation and 20% test
# [BUG]     # Validation share within temporary = 10 / 30 = 1/3
# [BUG]     # Test share within temporary = 20 / 30 = 2/3
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp,
# [BUG]         y_temp,
# [BUG]         test_size=0.6667,
# [BUG]         stratify=y_temp,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)


# ------------------------------------------------------------
# 11. DEFINE BASE ML MODELS
# ------------------------------------------------------------
def get_models():

    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier

        models["xgboost"] = XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        )

    except ImportError:
        print("XGBoost is not installed.")

    try:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    except ImportError:
        print("LightGBM is not installed.")

    return models


# ------------------------------------------------------------
# 12. HYPERPARAMETER TUNING FUNCTIONS
# ------------------------------------------------------------
def tune_random_forest(X_train, y_train):

    param_grid = {
        "n_estimators": [300, 500, 800],
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        param_distributions=param_grid,
        n_iter=25,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best Random Forest parameters:")
    print(search.best_params_)

    return search.best_estimator_


def tune_xgboost(X_train, y_train):

    from xgboost import XGBClassifier

    param_grid = {
        "n_estimators": [300, 500, 800],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "min_child_weight": [1, 3, 5]
    }

    search = RandomizedSearchCV(
        estimator=XGBClassifier(
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        ),
        param_distributions=param_grid,
        n_iter=25,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best XGBoost parameters:")
    print(search.best_params_)

    return search.best_estimator_


# ------------------------------------------------------------
# 13. TRAIN, TUNE, COMPARE, AND EVALUATE ALL MODELS
# ------------------------------------------------------------
all_results = []
best_model_results = []

for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(
        subset=[target_column]
    ).reset_index(drop=True)

    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(
        X,
        y_encoded,
        groups=household_of(task_data)  # [FIX] split by household
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()

    # --------------------------------------------------------
    # Tune selected models
    # --------------------------------------------------------
    if task_name in ["occupation", "income_class"]:

        print("\nTuning Random Forest...")
        models["random_forest_tuned"] = tune_random_forest(
            X_train_scaled,
            y_train
        )

    if task_name == "primary_mode":

        print("\nTuning XGBoost...")
        try:
            models["xgboost_tuned"] = tune_xgboost(
                X_train_scaled,
                y_train
            )
        except ImportError:
            print("XGBoost tuning skipped.")

    # --------------------------------------------------------
    # Train and evaluate all models
    # --------------------------------------------------------
    validation_results = []

    for model_name, model in models.items():

        print(f"\nEvaluating model: {model_name}")

        try:
            model.fit(X_train_scaled, y_train)

            val_pred = model.predict(X_val_scaled)

            val_macro_f1 = f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )

            test_pred = model.predict(X_test_scaled)

            test_accuracy = accuracy_score(y_test, test_pred)

            test_macro_f1 = f1_score(
                y_test,
                test_pred,
                average="macro",
                zero_division=0
            )

            test_balanced_accuracy = balanced_accuracy_score(
                y_test,
                test_pred
            )

            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })

            all_results.append({
                "task": task_name,
                "model": model_name,
                "validation_macro_f1": val_macro_f1,
                "test_accuracy": test_accuracy,
                "test_macro_f1": test_macro_f1,
                "test_balanced_accuracy": test_balanced_accuracy
            })

            print(f"Validation macro F1: {val_macro_f1:.4f}")
            print(f"Test accuracy: {test_accuracy:.4f}")
            print(f"Test macro F1: {test_macro_f1:.4f}")
            print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")

            print("\nClassification report:")
            print(
                classification_report(
                    y_test,
                    test_pred,
                    # [FIX] household split: a rare class can be missing from the test set
                    labels=np.arange(len(label_encoder.classes_)),
                    target_names=label_encoder.classes_,
                    zero_division=0
                )
            )

            # Confusion matrix
            cm = confusion_matrix(
                y_test,
                test_pred,
                labels=range(len(label_encoder.classes_))
            )

            plt.figure(figsize=(8, 6))
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_
            )

            plt.title(
                f"{task_name}\nModel: {model_name}"
            )
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()

            confusion_file = os.path.join(
                OUTPUT_FOLDER,
                f"confusion_matrix_{task_name}_{model_name}.png"
            )

            plt.savefig(confusion_file, dpi=300)
            plt.close()

        except Exception as error:
            print(f"Failed to train {model_name}: {error}")

    # --------------------------------------------------------
    # Select best model using validation macro F1
    # --------------------------------------------------------
    validation_df = pd.DataFrame(validation_results)

    best_model_name = validation_df.sort_values(
        by="validation_macro_f1",
        ascending=False
    ).iloc[0]["model"]

    best_result = next(
        result for result in all_results
        if result["task"] == task_name
        and result["model"] == best_model_name
    )

    best_model_results.append(best_result)

    print(
        f"\nBest model for {task_name}: {best_model_name}"
    )

# ------------------------------------------------------------
# 14. SAVE ALL RESULTS
# ------------------------------------------------------------
all_results_df = pd.DataFrame(all_results)

all_results_file = os.path.join(
    OUTPUT_FOLDER,
    "all_model_comparison_results.csv"
)

all_results_df.to_csv(all_results_file, index=False)

best_results_df = pd.DataFrame(best_model_results)

best_results_file = os.path.join(
    OUTPUT_FOLDER,
    "best_model_results.csv"
)

best_results_df.to_csv(best_results_file, index=False)

person_feature_file = os.path.join(
    OUTPUT_FOLDER,
    "person_level_improved_mobility_graph_features.csv"
)

person_df.to_csv(person_feature_file, index=False)

# ------------------------------------------------------------
# 15. CREATE COMPARISON CHARTS
# ------------------------------------------------------------
for task_name in all_results_df["task"].unique():

    task_results = all_results_df[
        all_results_df["task"] == task_name
    ].sort_values(
        by="validation_macro_f1",
        ascending=False
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=task_results,
        x="model",
        y="validation_macro_f1",
        palette="viridis"
    )

    plt.title(
        f"Validation Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Validation Macro F1")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

    chart_file = os.path.join(
        OUTPUT_FOLDER,
        f"validation_macro_f1_{task_name}.png"
    )

    plt.savefig(chart_file, dpi=300)
    plt.close()

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=task_results,
        x="model",
        y="test_macro_f1",
        palette="mako"
    )

    plt.title(
        f"Test Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Test Macro F1")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

    test_chart_file = os.path.join(
        OUTPUT_FOLDER,
        f"test_macro_f1_{task_name}.png"
    )

    plt.savefig(test_chart_file, dpi=300)
    plt.close()

# ------------------------------------------------------------
# 16. PRINT FINAL RESULTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL BEST-MODEL RESULTS")
print("=" * 70)
print(best_results_df)

print("\n" + "=" * 70)
print("FULL MODEL-COMPARISON RESULTS")
print("=" * 70)
print(all_results_df)

print(f"\nAll results saved in: {OUTPUT_FOLDER}")

Loaded dataset shape: (8645, 30)
Available columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr', 'person_id', 'home_zone']

Destination column standardized to: destination_zone

Simplified occupation distribution:
occupation_simplified
employed         52.562175
homemaker        20.647773
student_child    18.403702
non_employed      6.119144
other             2.267206
Name: proportion, dtype: float64

Simplified mode distribution:
mode_simplified
bus                     29.462117
active_transport        24.430307
two_wheeler            

In [12]:
# ============================================================
# COMPLETE IMPROVED MOBILITY-GRAPH-BASED SOCIODEMOGRAPHIC
# AND SOCIOECONOMIC INFERENCE PIPELINE (CORRECTED)
# ============================================================
# Input : Noisy synthetic Bengaluru person-trip HTS dataset
# Output: Predict simplified occupation, income class,
#         and simplified primary mode from mobility graphs
# ============================================================

import os
import sys
import warnings

# Completely suppress parallel & scikit-learn warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. FILE PATHS & LOADING
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/improved_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Loaded dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. STANDARDIZE COLUMN NAMES (UNIQUE PERSON, ORIGIN, DEST)
# ------------------------------------------------------------
# Construct unique person ID across households
if "HH_ID" in df.columns and "Person_ID" in df.columns:
    df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
elif "Person_ID" in df.columns:
    df["person_id"] = df["Person_ID"].astype(str)
else:
    df["person_id"] = df.index.astype(str)

# Standardize Origin Zone
origin_col_candidates = ["origin_zone", "home_zone", "E2_Origin_PIN", "E2_Origin_type", "origin"]
origin_col = next((c for c in origin_col_candidates if c in df.columns), None)
if origin_col:
    df["origin_zone"] = df[origin_col]
else:
    raise ValueError("No origin-zone column found.")

# Standardize Destination Zone
dest_col_candidates = ["destination_zone", "E3_Dest_PIN", "E3_Destination_name", "destination", "dest_zone"]
dest_col = next((c for c in dest_col_candidates if c in df.columns), None)
if dest_col:
    df["destination_zone"] = df[dest_col]
else:
    raise ValueError("No destination-zone column found.")

print(f"Total trips: {len(df)}")
print(f"Total unique travelers identified: {df['person_id'].nunique()}")

# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan

df["departure_time_min"] = df["departure_time_hhmm"].apply(hhmm_to_minutes)

# ------------------------------------------------------------
# 4. SIMPLIFY RARE OCCUPATION AND MODE CLASSES
# ------------------------------------------------------------
def simplify_occupation(occupation):
    occupation = str(occupation).strip().lower()
    # [BUG] if "employ" in occupation:
    # [BUG]     return "employed"
    # [BUG] elif "student" in occupation or "child" in occupation:
    # [BUG]     return "student_child"
    # [BUG] elif "home" in occupation:
    # [BUG]     return "homemaker"
    # [BUG] elif "retire" in occupation or "unemploy" in occupation:
    # [BUG]     return "non_employed"
    # [BUG] else:
    # [BUG]     return "other"
    # [FIX] "unemployed" contains "employ", so every Unemployed person was counted as employed.
    #       Check unemployed / retired first.
    if "unemploy" in occupation or "retire" in occupation:
        return "non_employed"
    elif "employ" in occupation:
        return "employed"
    elif "student" in occupation or "child" in occupation:
        return "student_child"
    elif "home" in occupation:
        return "homemaker"
    else:
        return "other"

def simplify_mode(mode):
    mode = str(mode).strip().lower()
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in mode for word in ['ride-hail', 'taxi', 'rickshaw']):
        return "paid_private_vehicle"
    if "walk" in mode or "bicycle" in mode:
        return "active_transport"
    elif "metro" in mode or "rail" in mode:
        return "rail_transit"
    elif "bmtc" in mode or "bus" in mode:
        return "bus"
    elif "two" in mode:
        return "two_wheeler"
    elif "car" in mode:
        return "private_car"
    elif "auto" in mode or "app cab" in mode:
        return "paid_private_vehicle"
    else:
        return "other"

df["occupation_simplified"] = df["occupation"].apply(simplify_occupation)
df["mode_simplified"] = df["mode"].apply(simplify_mode)

print("\nSimplified occupation distribution (%):")
print(df["occupation_simplified"].value_counts(normalize=True) * 100)

print("\nSimplified mode distribution (%):")
print(df["mode_simplified"].value_counts(normalize=True) * 100)

# ------------------------------------------------------------
# 5. BUILD IMPROVED PERSON-LEVEL MOBILITY-GRAPH FEATURES
# ------------------------------------------------------------
def build_improved_mobility_graph_features(person_trips):
    person_trips = person_trips.sort_values(by=["departure_time_min"])
    features = {}
    total_trips = len(person_trips)

    # Establish person's home anchor zone (first origin of the day)
    home_zone = person_trips["origin_zone"].iloc[0]

    # Graph nodes
    visited_zones = pd.concat([
        person_trips["origin_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = total_trips

    # Graph edges: origin_zone -> destination_zone
    edge_counts = person_trips.groupby(["origin_zone", "destination_zone"]).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max() if len(edge_counts) > 0 else 0
    features["graph_mean_edge_weight"] = edge_counts.mean() if len(edge_counts) > 0 else 0

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)
    features["graph_density"] = (
        len(edge_counts) / possible_edges if possible_edges > 0 else 0
    )

    # Destination diversity and entropy
    destination_counts = person_trips["destination_zone"].value_counts(normalize=True)
    features["num_unique_destinations"] = len(destination_counts)
    features["destination_entropy"] = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )
    features["max_destination_share"] = destination_counts.max() if len(destination_counts) > 0 else 0

    # Distance features
    features["mean_trip_distance"] = person_trips["trip_distance_km"].mean()
    features["max_trip_distance"] = person_trips["trip_distance_km"].max()
    features["std_trip_distance"] = person_trips["trip_distance_km"].std()
    features["short_trip_share"] = np.mean(person_trips["trip_distance_km"] < 3)
    features["medium_trip_share"] = np.mean(
        (person_trips["trip_distance_km"] >= 3) & (person_trips["trip_distance_km"] < 10)
    )
    features["long_trip_share"] = np.mean(person_trips["trip_distance_km"] >= 10)

    # Duration and cost features
    features["mean_trip_duration"] = person_trips["trip_duration_min"].mean()
    features["max_trip_duration"] = person_trips["trip_duration_min"].max()
    features["mean_travel_cost"] = person_trips["travel_cost_inr"].mean()

    # Temporal mobility features
    features["mean_departure_time"] = person_trips["departure_time_min"].mean()
    features["std_departure_time"] = person_trips["departure_time_min"].std()
    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360) & (person_trips["departure_time_min"] < 720)
    )
    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720) & (person_trips["departure_time_min"] < 1020)
    )
    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020) & (person_trips["departure_time_min"] < 1200)
    )
    features["peak_morning_share"] = np.mean(
        (person_trips["departure_time_min"] >= 480) & (person_trips["departure_time_min"] < 600)
    )
    features["peak_evening_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020) & (person_trips["departure_time_min"] < 1140)
    )

    # Trip-purpose features
    purpose_counts = person_trips["trip_purpose"].value_counts(normalize=True)
    features["purpose_entropy"] = -np.sum(purpose_counts * np.log(purpose_counts + 1e-9))

    work_trips = person_trips[
        person_trips["trip_purpose"].astype(str).str.lower().str.contains("work", na=False)
    ]
    features["work_trip_share"] = len(work_trips) / total_trips
    features["mean_work_distance"] = work_trips["trip_distance_km"].mean() if len(work_trips) > 0 else 0

    education_trips = person_trips[
        person_trips["trip_purpose"].astype(str).str.lower().str.contains("education|school|college", na=False)
    ]
    features["education_trip_share"] = len(education_trips) / total_trips
    features["mean_education_distance"] = education_trips["trip_distance_km"].mean() if len(education_trips) > 0 else 0

    shopping_trips = person_trips[
        person_trips["trip_purpose"].astype(str).str.lower().str.contains("shop", na=False)
    ]
    features["shopping_trip_share"] = len(shopping_trips) / total_trips

    # Return-home feature
    home_return_trips = person_trips[person_trips["destination_zone"] == home_zone]
    features["return_home_share"] = len(home_return_trips) / total_trips

    # Person-level labels
    features["label_occupation_simplified"] = person_trips["occupation_simplified"].iloc[0]
    features["label_income"] = person_trips["household_monthly_income_inr"].iloc[0]
    features["label_primary_mode_simplified"] = person_trips["mode_simplified"].value_counts().idxmax()

    return features

# ------------------------------------------------------------
# 6. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):
    graph_features = build_improved_mobility_graph_features(person_trips)
    graph_features["person_id"] = person_id
    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)
print("\nImproved person-level mobility-graph dataset shape:", person_df.shape)

# ------------------------------------------------------------
# 7. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 50000:
        return "low_or_lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"

person_df["label_income_class"] = person_df["label_income"].apply(income_to_class)

# ------------------------------------------------------------
# 8. DEFINE TRANSFERABLE GRAPH-FEATURE SET
# ------------------------------------------------------------
base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "max_destination_share",
    "mean_trip_distance",
    "max_trip_distance",
    "std_trip_distance",
    "short_trip_share",
    "medium_trip_share",
    "long_trip_share",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share",
    "peak_morning_share",
    "peak_evening_share",
    "purpose_entropy",
    "work_trip_share",
    "mean_work_distance",
    "education_trip_share",
    "mean_education_distance",
    "shopping_trip_share",
    "return_home_share"
]

FEATURE_COLUMNS = [col for col in base_graph_features if col in person_df.columns]

# Fill missing values
numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(include=[np.number]).columns
person_df[numeric_features] = person_df[numeric_features].fillna(0.0)

# ------------------------------------------------------------
# 9. DEFINE PREDICTION TASKS
# ------------------------------------------------------------
TARGETS = {
    "occupation": "label_occupation_simplified",
    "income_class": "label_income_class",
    "primary_mode": "label_primary_mode_simplified"
}

# ------------------------------------------------------------
# 10. INTEGER-SAFE 70 / 10 / 20 SPLIT
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]     n_samples = len(X)
# [BUG]     n_val = max(1, int(round(0.10 * n_samples)))
# [BUG]     n_test = max(1, int(round(0.20 * n_samples)))
# [BUG]     n_temp = n_val + n_test
# [BUG]
# [BUG]     class_counts = pd.Series(y).value_counts()
# [BUG]     use_stratify_1 = y if class_counts.min() >= 2 else None
# [BUG]
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X, y,
# [BUG]         test_size=n_temp,
# [BUG]         stratify=use_stratify_1,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     temp_counts = pd.Series(y_temp).value_counts()
# [BUG]     use_stratify_2 = y_temp if temp_counts.min() >= 2 else None
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp, y_temp,
# [BUG]         test_size=n_test,
# [BUG]         stratify=use_stratify_2,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)

# ------------------------------------------------------------
# 11. DEFINE BASE ML MODELS (n_jobs=1 to suppress warnings)
# ------------------------------------------------------------
def get_models():
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        )
    }

    try:
        from xgboost import XGBClassifier
        models["xgboost"] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=1
        )
    except ImportError:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
            verbose=-1
        )
    except ImportError:
        pass

    return models

# ------------------------------------------------------------
# 12. HYPERPARAMETER TUNING FUNCTIONS
# ------------------------------------------------------------
def tune_random_forest(X_train, y_train):
    param_grid = {
        "n_estimators": [200, 300],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        param_distributions=param_grid,
        n_iter=10,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=1,
        verbose=0
    )

    search.fit(X_train, y_train)
    print("  -> Best Random Forest params:", search.best_params_)
    return search.best_estimator_

def tune_xgboost(X_train, y_train):
    from xgboost import XGBClassifier
    param_grid = {
        "n_estimators": [200, 300],
        "max_depth": [4, 6],
        "learning_rate": [0.03, 0.05],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }

    search = RandomizedSearchCV(
        estimator=XGBClassifier(
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=1
        ),
        param_distributions=param_grid,
        n_iter=10,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=1,
        verbose=0
    )

    search.fit(X_train, y_train)
    print("  -> Best XGBoost params:", search.best_params_)
    return search.best_estimator_

# ------------------------------------------------------------
# 13. TRAIN, TUNE, COMPARE, AND EVALUATE ALL MODELS
# ------------------------------------------------------------
all_results = []
best_model_results = []

for task_name, target_column in TARGETS.items():
    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(subset=[target_column]).reset_index(drop=True)
    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(X, y_encoded, groups=household_of(task_data))  # [FIX] split by household
    print(f"Data split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    models = get_models()

    # Tune selected models
    if task_name in ["occupation", "income_class"]:
        print("Tuning Random Forest...")
        try:
            models["random_forest_tuned"] = tune_random_forest(X_train_scaled, y_train)
        except Exception as e:
            print("  Tuning skipped:", e)

    if task_name == "primary_mode":
        print("Tuning XGBoost...")
        try:
            models["xgboost_tuned"] = tune_xgboost(X_train_scaled, y_train)
        except Exception as e:
            print("  Tuning skipped:", e)

    validation_results = []

    for model_name, model in models.items():
        print(f"\nEvaluating: {model_name}...")
        try:
            model.fit(X_train_scaled, y_train)

            val_pred = np.asarray(model.predict(X_val_scaled)).ravel()
            val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)

            test_pred = np.asarray(model.predict(X_test_scaled)).ravel()
            test_accuracy = accuracy_score(y_test, test_pred)
            test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
            test_balanced_accuracy = balanced_accuracy_score(y_test, test_pred)

            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })

            all_results.append({
                "task": task_name,
                "model": model_name,
                "validation_macro_f1": val_macro_f1,
                "test_accuracy": test_accuracy,
                "test_macro_f1": test_macro_f1,
                "test_balanced_accuracy": test_balanced_accuracy
            })

            print(f"  -> Validation Macro F1: {val_macro_f1:.4f}")
            print(f"  -> Test Accuracy:       {test_accuracy:.4f}")
            print(f"  -> Test Macro F1:       {test_macro_f1:.4f}")
            print(f"  -> Test Balanced Acc:   {test_balanced_accuracy:.4f}")

            # Confusion matrix
            cm = confusion_matrix(y_test, test_pred, labels=range(len(label_encoder.classes_)))
            plt.figure(figsize=(7, 5))
            sns.heatmap(
                cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_
            )
            plt.title(f"{task_name} | {model_name}")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()

            confusion_file = os.path.join(
                OUTPUT_FOLDER,
                f"confusion_matrix_{task_name}_{model_name}.png"
            )
            plt.savefig(confusion_file, dpi=300)
            plt.close()

        except Exception as error:
            print(f"  Failed to train {model_name}: {error}")

    # Select best model using validation macro F1
    if validation_results:
        validation_df = pd.DataFrame(validation_results)
        best_model_name = validation_df.sort_values(
            by="validation_macro_f1", ascending=False
        ).iloc[0]["model"]

        best_result = next(
            r for r in all_results
            if r["task"] == task_name and r["model"] == best_model_name
        )
        best_model_results.append(best_result)
        print(f"\n>> Best model for {task_name}: {best_model_name}")

# ------------------------------------------------------------
# 14. SAVE ALL RESULTS
# ------------------------------------------------------------
all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv(os.path.join(OUTPUT_FOLDER, "all_model_comparison_results.csv"), index=False)

best_results_df = pd.DataFrame(best_model_results)
best_results_df.to_csv(os.path.join(OUTPUT_FOLDER, "best_model_results.csv"), index=False)

person_df.to_csv(os.path.join(OUTPUT_FOLDER, "person_level_improved_mobility_graph_features.csv"), index=False)

# ------------------------------------------------------------
# 15. CREATE COMPARISON CHARTS
# ------------------------------------------------------------
for task_name in all_results_df["task"].unique():
    task_results = all_results_df[all_results_df["task"] == task_name].sort_values(
        by="validation_macro_f1", ascending=False
    )

    # Validation chart
    plt.figure(figsize=(10, 5))
    sns.barplot(data=task_results, x="model", y="validation_macro_f1", palette="viridis")
    plt.title(f"Validation Macro F1 | Task: {task_name}")
    plt.xlabel("ML Model")
    plt.ylabel("Validation Macro F1")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, f"validation_macro_f1_{task_name}.png"), dpi=300)
    plt.close()

    # Test chart
    plt.figure(figsize=(10, 5))
    sns.barplot(data=task_results, x="model", y="test_macro_f1", palette="mako")
    plt.title(f"Test Macro F1 | Task: {task_name}")
    plt.xlabel("ML Model")
    plt.ylabel("Test Macro F1")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, f"test_macro_f1_{task_name}.png"), dpi=300)
    plt.close()

# ------------------------------------------------------------
# 16. PRINT SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL BEST-MODEL RESULTS")
print("=" * 70)
print(best_results_df.to_string(index=False))

print("\n" + "=" * 70)
print("FULL MODEL-COMPARISON RESULTS")
print("=" * 70)
print(all_results_df.to_string(index=False))

print(f"\nAll artifacts successfully saved to: {OUTPUT_FOLDER}")


Loaded dataset shape: (8645, 28)
Total trips: 8645
Total unique travelers identified: 3383

Simplified occupation distribution (%):
occupation_simplified
employed         52.562175
homemaker        20.647773
student_child    18.403702
non_employed      6.119144
other             2.267206
Name: proportion, dtype: float64

Simplified mode distribution (%):
mode_simplified
bus                     29.462117
active_transport        24.430307
two_wheeler             18.577212
paid_private_vehicle    16.633892
private_car              6.246385
rail_transit             4.650087
Name: proportion, dtype: float64

Improved person-level mobility-graph dataset shape: (3383, 37)

TASK: occupation
Data split: Train=2375, Val=334, Test=674
Tuning Random Forest...
  -> Best Random Forest params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 10}

Evaluating: logistic_regression...
  -> Validation Macro F1: 0.5583
  -> Test Accuracy:       0.685

In [13]:
# ============================================================
# FINAL COMPLETE IMPROVED MOBILITY-GRAPH-BASED
# SOCIODEMOGRAPHIC AND SOCIOECONOMIC INFERENCE PIPELINE
# ============================================================
# Input : Noisy synthetic Bengaluru person-trip HTS dataset
# Output: Predict simplified occupation, income class,
#         and simplified primary mode from mobility graphs
#
# Key improvements:
# 1. Stronger mobility-graph and activity-routine features
# 2. Rare-class simplification
# 3. SMOTE applied only to training data
# 4. Two-stage occupation model
# 5. LightGBM tuning for primary mode
# 6. Random Forest tuning for occupation and income
# 7. Ordinal-aware income evaluation
# 8. 5-fold stratified cross-validation
# 9. Person-level 70/10/20 split
# 10. Test set used only once
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
INPUT_FILE = "output/bengaluru_hts_noisy_medium.csv"
OUTPUT_FOLDER = "output/final_improved_mobility_graph_results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
# [FIX] KeyError 'person_id' / 'home_zone': the noisy file has HH_ID, Person_ID and E2_Origin_PIN,
#       not the person_id / home_zone columns this draft expects.
#       Person_ID restarts at 1 in every household, so the person key needs HH_ID too.
df["person_id"] = df["HH_ID"].astype(str) + "_" + df["Person_ID"].astype(str)
# [FIX] home zone = origin PIN of each person's first trip (every diary day starts at home)
df["home_zone"] = df["person_id"].map(
    df.loc[df["Trip_no_full"] == 1].set_index("person_id")["E2_Origin_PIN"]
)

print("Loaded dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. STANDARDIZE DESTINATION COLUMN NAME
# ------------------------------------------------------------
# [ERROR] possible_destination_columns = [
# [ERROR]     "destination_zone",
# [ERROR]     "destination",
# [ERROR]     "dest_zone",
# [ERROR]     "trip_destination"
# [ERROR] ]
# [FIX] ValueError 'No destination-zone column found': the file names it E3_Dest_PIN
possible_destination_columns = [
    "destination_zone",
    "E3_Dest_PIN",
    "destination",
    "dest_zone",
    "trip_destination"
]

destination_column = None

for col in possible_destination_columns:
    if col in df.columns:
        destination_column = col
        break

if destination_column is None:
    raise ValueError(
        "No destination-zone column found. Expected one of: "
        "destination_zone, destination, dest_zone, trip_destination."
    )

df = df.rename(columns={destination_column: "destination_zone"})

print("\nDestination column standardized to: destination_zone")


# ------------------------------------------------------------
# 3. CONVERT DEPARTURE TIME TO MINUTES
# ------------------------------------------------------------
def hhmm_to_minutes(value):
    try:
        hour, minute = str(value).split(":")
        return int(hour) * 60 + int(minute)
    except:
        return np.nan


df["departure_time_min"] = df["departure_time_hhmm"].apply(
    hhmm_to_minutes
)


# ------------------------------------------------------------
# 4. SIMPLIFY RARE OCCUPATION AND MODE CLASSES
# ------------------------------------------------------------
def simplify_occupation(occupation):
    occupation = str(occupation).strip().lower()

    # [BUG] if "employ" in occupation:
    # [BUG]     return "employed"
    # [BUG] elif "student" in occupation or "child" in occupation:
    # [BUG]     return "student_child"
    # [BUG] elif "home" in occupation:
    # [BUG]     return "homemaker"
    # [BUG] elif "retire" in occupation or "unemploy" in occupation:
    # [BUG]     return "non_employed"
    # [BUG] else:
    # [BUG]     return "other"
    # [FIX] "unemployed" contains "employ", so every Unemployed person was counted as employed.
    #       Check unemployed / retired first.
    if "unemploy" in occupation or "retire" in occupation:
        return "non_employed"
    elif "employ" in occupation:
        return "employed"
    elif "student" in occupation or "child" in occupation:
        return "student_child"
    elif "home" in occupation:
        return "homemaker"
    else:
        return "other"


def simplify_mode(mode):
    mode = str(mode).strip().lower()
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in mode for word in ['ride-hail', 'taxi', 'rickshaw']):
        return "paid_private_vehicle"

    if "metro" in mode or "rail" in mode:
        return "rail_based_public_transport"
    elif "walk" in mode or "bicycle" in mode:
        return "active_transport"
    elif "bmtc" in mode or "bus" in mode:
        return "bus"
    elif "two" in mode:
        return "two_wheeler"
    elif "car" in mode:
        return "private_car"
    elif "auto" in mode or "app cab" in mode:
        return "paid_private_vehicle"
    else:
        return "other"


df["occupation_simplified"] = df["occupation"].apply(
    simplify_occupation
)

df["mode_simplified"] = df["mode"].apply(
    simplify_mode
)

print("\nSimplified occupation distribution:")
print(df["occupation_simplified"].value_counts(normalize=True) * 100)

print("\nSimplified mode distribution:")
print(df["mode_simplified"].value_counts(normalize=True) * 100)


# ------------------------------------------------------------
# 5. BUILD IMPROVED PERSON-LEVEL MOBILITY-GRAPH FEATURES
# ------------------------------------------------------------
def build_improved_mobility_graph_features(person_trips):

    person_trips = person_trips.sort_values(
        by=["departure_time_min"]
    )

    features = {}
    total_trips = len(person_trips)

    # --------------------------------------------------------
    # Graph nodes
    # --------------------------------------------------------
    visited_zones = pd.concat([
        person_trips["home_zone"],
        person_trips["destination_zone"]
    ]).dropna().unique()

    features["graph_num_nodes"] = len(visited_zones)
    features["graph_num_trips"] = total_trips

    # --------------------------------------------------------
    # Graph edges
    # --------------------------------------------------------
    edge_counts = person_trips.groupby(
        ["home_zone", "destination_zone"]
    ).size()

    features["graph_num_edges"] = len(edge_counts)
    features["graph_total_edge_weight"] = edge_counts.sum()
    features["graph_max_edge_weight"] = edge_counts.max()
    features["graph_mean_edge_weight"] = edge_counts.mean()

    possible_edges = len(visited_zones) * (len(visited_zones) - 1)

    features["graph_density"] = (
        len(edge_counts) / possible_edges
        if possible_edges > 0
        else 0
    )

    # --------------------------------------------------------
    # Destination diversity
    # --------------------------------------------------------
    destination_counts = person_trips[
        "destination_zone"
    ].value_counts(normalize=True)

    features["num_unique_destinations"] = len(destination_counts)

    features["destination_entropy"] = -np.sum(
        destination_counts * np.log(destination_counts + 1e-9)
    )

    features["max_destination_share"] = destination_counts.max()

    # --------------------------------------------------------
    # Distance features
    # --------------------------------------------------------
    features["mean_trip_distance"] = person_trips[
        "trip_distance_km"
    ].mean()

    features["max_trip_distance"] = person_trips[
        "trip_distance_km"
    ].max()

    features["min_trip_distance"] = person_trips[
        "trip_distance_km"
    ].min()

    features["std_trip_distance"] = person_trips[
        "trip_distance_km"
    ].std()

    features["short_trip_share"] = np.mean(
        person_trips["trip_distance_km"] < 3
    )

    features["medium_trip_share"] = np.mean(
        (person_trips["trip_distance_km"] >= 3)
        & (person_trips["trip_distance_km"] < 10)
    )

    features["long_trip_share"] = np.mean(
        person_trips["trip_distance_km"] >= 10
    )

    # --------------------------------------------------------
    # Duration and cost features
    # --------------------------------------------------------
    features["mean_trip_duration"] = person_trips[
        "trip_duration_min"
    ].mean()

    features["max_trip_duration"] = person_trips[
        "trip_duration_min"
    ].max()

    features["mean_travel_cost"] = person_trips[
        "travel_cost_inr"
    ].mean()

    # --------------------------------------------------------
    # Temporal mobility features
    # --------------------------------------------------------
    features["mean_departure_time"] = person_trips[
        "departure_time_min"
    ].mean()

    features["std_departure_time"] = person_trips[
        "departure_time_min"
    ].std()

    features["morning_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 360)
        & (person_trips["departure_time_min"] < 720)
    )

    features["afternoon_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 720)
        & (person_trips["departure_time_min"] < 1020)
    )

    features["evening_trip_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1200)
    )

    features["peak_morning_share"] = np.mean(
        (person_trips["departure_time_min"] >= 480)
        & (person_trips["departure_time_min"] < 600)
    )

    features["peak_evening_share"] = np.mean(
        (person_trips["departure_time_min"] >= 1020)
        & (person_trips["departure_time_min"] < 1140)
    )

    # --------------------------------------------------------
    # Trip-purpose features
    # --------------------------------------------------------
    purpose_counts = person_trips["trip_purpose"].value_counts(
        normalize=True
    )

    features["purpose_entropy"] = -np.sum(
        purpose_counts * np.log(purpose_counts + 1e-9)
    )

    work_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("work", na=False)
    ]

    features["work_trip_share"] = len(work_trips) / total_trips

    features["mean_work_distance"] = (
        work_trips["trip_distance_km"].mean()
        if len(work_trips) > 0
        else 0
    )

    education_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("education|school|college", na=False)
    ]

    features["education_trip_share"] = (
        len(education_trips) / total_trips
    )

    features["mean_education_distance"] = (
        education_trips["trip_distance_km"].mean()
        if len(education_trips) > 0
        else 0
    )

    shopping_trips = person_trips[
        person_trips["trip_purpose"]
        .astype(str)
        .str.lower()
        .str.contains("shop", na=False)
    ]

    features["shopping_trip_share"] = (
        len(shopping_trips) / total_trips
    )

    # --------------------------------------------------------
    # Return-home feature
    # --------------------------------------------------------
    home_return_trips = person_trips[
        person_trips["destination_zone"] == person_trips["home_zone"]
    ]

    features["return_home_share"] = (
        len(home_return_trips) / total_trips
    )

    # --------------------------------------------------------
    # Additional activity-routine features
    # --------------------------------------------------------
    features["work_regular_pattern"] = (
        features["work_trip_share"]
        * features["peak_morning_share"]
    )

    features["education_regular_pattern"] = (
        features["education_trip_share"]
        * features["peak_morning_share"]
    )

    features["non_work_activity_share"] = (
        features["shopping_trip_share"]
        + features["return_home_share"]
    )

    features["trips_per_visited_zone"] = (
        total_trips / len(visited_zones)
        if len(visited_zones) > 0
        else 0
    )

    features["edges_per_trip"] = (
        len(edge_counts) / total_trips
        if total_trips > 0
        else 0
    )

    features["distance_range"] = (
        features["max_trip_distance"]
        - features["min_trip_distance"]
    )

    features["long_short_ratio"] = (
        features["long_trip_share"]
        / (features["short_trip_share"] + 1e-9)
    )

    features["peak_trip_share"] = (
        features["peak_morning_share"]
        + features["peak_evening_share"]
    )

    features["off_peak_trip_share"] = (
        1 - features["peak_trip_share"]
    )

    features["cost_per_km"] = (
        features["mean_travel_cost"]
        / (features["mean_trip_distance"] + 1e-9)
    )

    features["dominant_zone_share"] = (
        edge_counts.max() / edge_counts.sum()
        if edge_counts.sum() > 0
        else 0
    )

    cost_counts = person_trips["travel_cost_inr"].value_counts(
        normalize=True
    )

    features["travel_cost_entropy"] = -np.sum(
        cost_counts * np.log(cost_counts + 1e-9)
    )

    # --------------------------------------------------------
    # Person-level labels
    # --------------------------------------------------------
    features["label_occupation_simplified"] = person_trips[
        "occupation_simplified"
    ].iloc[0]

    features["label_income"] = person_trips[
        "household_monthly_income_inr"
    ].iloc[0]

    features["label_primary_mode_simplified"] = person_trips[
        "mode_simplified"
    ].value_counts().idxmax()

    return features


# ------------------------------------------------------------
# 6. CREATE ONE ROW PER PERSON
# ------------------------------------------------------------
person_feature_list = []

for person_id, person_trips in df.groupby("person_id"):

    graph_features = build_improved_mobility_graph_features(
        person_trips
    )

    graph_features["person_id"] = person_id

    person_feature_list.append(graph_features)

person_df = pd.DataFrame(person_feature_list)

print("\nImproved person-level mobility-graph dataset:", person_df.shape)


# ------------------------------------------------------------
# 7. CREATE INCOME-CLASS LABEL
# ------------------------------------------------------------
def income_to_class(income):
    if pd.isna(income):
        return np.nan
    elif income < 50000:
        return "low_or_lower_middle"
    elif income < 100000:
        return "middle"
    else:
        return "high"


person_df["label_income_class"] = person_df[
    "label_income"
].apply(income_to_class)


# ------------------------------------------------------------
# 8. CREATE TWO-STAGE OCCUPATION LABEL
# ------------------------------------------------------------
person_df["binary_employment"] = person_df[
    "label_occupation_simplified"
].apply(
    lambda x: "employed"
    if x == "employed"
    else "non_employed"
)


# ------------------------------------------------------------
# 9. DEFINE TRANSFERABLE GRAPH-FEATURE SET
# ------------------------------------------------------------
base_graph_features = [
    "graph_num_nodes",
    "graph_num_trips",
    "graph_num_edges",
    "graph_total_edge_weight",
    "graph_max_edge_weight",
    "graph_mean_edge_weight",
    "graph_density",
    "num_unique_destinations",
    "destination_entropy",
    "max_destination_share",
    "mean_trip_distance",
    "max_trip_distance",
    "min_trip_distance",
    "std_trip_distance",
    "short_trip_share",
    "medium_trip_share",
    "long_trip_share",
    "mean_trip_duration",
    "max_trip_duration",
    "mean_travel_cost",
    "mean_departure_time",
    "std_departure_time",
    "morning_trip_share",
    "afternoon_trip_share",
    "evening_trip_share",
    "peak_morning_share",
    "peak_evening_share",
    "purpose_entropy",
    "work_trip_share",
    "mean_work_distance",
    "education_trip_share",
    "mean_education_distance",
    "shopping_trip_share",
    "return_home_share",
    "work_regular_pattern",
    "education_regular_pattern",
    "non_work_activity_share",
    "trips_per_visited_zone",
    "edges_per_trip",
    "distance_range",
    "long_short_ratio",
    "peak_trip_share",
    "off_peak_trip_share",
    "cost_per_km",
    "dominant_zone_share",
    "travel_cost_entropy"
]

purpose_features = [
    col for col in person_df.columns
    if col.startswith("purpose_share_")
]

FEATURE_COLUMNS = base_graph_features + purpose_features

FEATURE_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col in person_df.columns
]

numeric_features = person_df[FEATURE_COLUMNS].select_dtypes(
    include=[np.number]
).columns

person_df[numeric_features] = person_df[numeric_features].fillna(
    person_df[numeric_features].median()
)

print("\nFinal number of mobility-graph features:", len(FEATURE_COLUMNS))


# ------------------------------------------------------------
# 10. PERSON-LEVEL 70 / 10 / 20 SPLIT
# ------------------------------------------------------------
# [BUG] def split_person_data(X, y):
# [BUG]
# [BUG]     X_train, X_temp, y_train, y_temp = train_test_split(
# [BUG]         X,
# [BUG]         y,
# [BUG]         test_size=0.30,
# [BUG]         stratify=y,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     X_val, X_test, y_val, y_test = train_test_split(
# [BUG]         X_temp,
# [BUG]         y_temp,
# [BUG]         test_size=0.6667,
# [BUG]         stratify=y_temp,
# [BUG]         random_state=42
# [BUG]     )
# [BUG]
# [BUG]     return X_train, X_val, X_test, y_train, y_val, y_test
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
def household_of(frame):
    """Household ID of each row: person_id is built as HH_ID + "_" + Person_ID."""
    return frame["person_id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()


def household_split(X, y, groups, test_size=0.20, val_size=0.10, seed=42):
    """Train / validation / test split (70 / 10 / 20 by default) that keeps every household on
    one side. Retries the shuffle until every class appears in training (XGBoost needs that)."""
    from sklearn.model_selection import GroupShuffleSplit

    groups, y_arr = np.asarray(groups), np.asarray(y)
    rows = np.arange(len(y_arr))
    for attempt in range(100):
        outer = GroupShuffleSplit(n_splits=1, test_size=test_size + val_size, random_state=seed + attempt)
        train_idx, rest_idx = next(outer.split(rows, y_arr, groups))
        if set(np.unique(y_arr[train_idx])) == set(np.unique(y_arr)):
            break
    if val_size > 0:
        inner = GroupShuffleSplit(n_splits=1, test_size=test_size / (test_size + val_size), random_state=seed)
        val_rel, test_rel = next(inner.split(rest_idx, y_arr[rest_idx], groups[rest_idx]))
        val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]
    else:
        val_idx, test_idx = rest_idx[:0], rest_idx

    def take(a, idx):
        return a.iloc[idx] if hasattr(a, "iloc") else np.asarray(a)[idx]

    return (take(X, train_idx), take(X, val_idx), take(X, test_idx),
            take(y, train_idx), take(y, val_idx), take(y, test_idx))


def split_person_data(X, y, groups):
    return household_split(X, y, groups)


# ------------------------------------------------------------
# 11. DEFINE BASE ML MODELS
# ------------------------------------------------------------
def get_models():

    models = {
        "logistic_regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    }

    try:
        from xgboost import XGBClassifier

        models["xgboost"] = XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        )

    except ImportError:
        print("XGBoost is not installed.")

    try:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    except ImportError:
        print("LightGBM is not installed.")

    return models


# ------------------------------------------------------------
# 12. HYPERPARAMETER TUNING FUNCTIONS
# ------------------------------------------------------------
def tune_random_forest(X_train, y_train):

    param_grid = {
        "n_estimators": [300, 500, 800],
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        param_distributions=param_grid,
        n_iter=25,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best Random Forest parameters:")
    print(search.best_params_)

    return search.best_estimator_


def tune_lightgbm(X_train, y_train):

    from lightgbm import LGBMClassifier

    param_grid = {
        "n_estimators": [300, 500, 800],
        "num_leaves": [15, 31, 63],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_samples": [10, 20, 40],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0]
    }

    search = RandomizedSearchCV(
        estimator=LGBMClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        param_distributions=param_grid,
        n_iter=25,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best LightGBM parameters:")
    print(search.best_params_)

    return search.best_estimator_


# ------------------------------------------------------------
# 13. ORDINAL INCOME EVALUATION
# ------------------------------------------------------------
def adjacent_class_accuracy(y_true, y_pred, class_order):

    correct = 0
    total = len(y_true)

    for true_label, predicted_label in zip(y_true, y_pred):

        true_index = class_order.index(true_label)
        predicted_index = class_order.index(predicted_label)

        if abs(true_index - predicted_index) <= 1:
            correct += 1

    return correct / total


# ------------------------------------------------------------
# 14. PLOT CONFUSION MATRIX
# ------------------------------------------------------------
def plot_confusion_matrix(
    y_test,
    y_pred,
    label_encoder,
    task_name,
    model_name
):

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=range(len(label_encoder.classes_))
    )

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )

    plt.title(f"{task_name}\nModel: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    file_path = os.path.join(
        OUTPUT_FOLDER,
        f"confusion_matrix_{task_name}_{model_name}.png"
    )

    plt.savefig(file_path, dpi=300)
    plt.close()


# ------------------------------------------------------------
# 15. TRAIN, TUNE, COMPARE, AND EVALUATE
# ------------------------------------------------------------
all_results = []
best_model_results = []

for task_name, target_column in TARGETS.items() if False else []:

    pass


# Define prediction tasks
TARGETS = {
    "occupation": "label_occupation_simplified",
    "income_class": "label_income_class",
    "primary_mode": "label_primary_mode_simplified"
}

for task_name, target_column in TARGETS.items():

    print("\n" + "=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    task_data = person_df.dropna(
        subset=[target_column]
    ).reset_index(drop=True)

    X = task_data[FEATURE_COLUMNS]
    y = task_data[target_column]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_val, X_test, y_train, y_val, y_test = split_person_data(
        X,
        y_encoded,
        groups=household_of(task_data)  # [FIX] split by household
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------
    # SMOTE: training set only
    # --------------------------------------------------------
    use_smote = task_name in ["occupation", "income_class"]

    if use_smote:

        try:
            from imblearn.over_sampling import SMOTE

            smote = SMOTE(
                sampling_strategy="not majority",
                random_state=42,
                k_neighbors=3
            )

            X_train_model, y_train_model = smote.fit_resample(
                X_train_scaled,
                y_train
            )

            print("SMOTE applied to training data only.")

        except ImportError:
            print("imbalanced-learn not installed. SMOTE skipped.")
            X_train_model = X_train_scaled
            y_train_model = y_train

    else:
        X_train_model = X_train_scaled
        y_train_model = y_train

    models = get_models()

    # --------------------------------------------------------
    # Model-specific tuning
    # --------------------------------------------------------
    if task_name in ["occupation", "income_class"]:

        print("\nTuning Random Forest...")
        models["random_forest_tuned"] = tune_random_forest(
            X_train_model,
            y_train_model
        )

    if task_name == "primary_mode":

        print("\nTuning LightGBM...")
        try:
            models["lightgbm_tuned"] = tune_lightgbm(
                X_train_model,
                y_train_model
            )
        except ImportError:
            print("LightGBM tuning skipped.")

    # --------------------------------------------------------
    # Train and evaluate all models
    # --------------------------------------------------------
    validation_results = []

    for model_name, model in models.items():

        print(f"\nEvaluating model: {model_name}")

        try:
            model.fit(X_train_model, y_train_model)

            val_pred = model.predict(X_val_scaled)

            val_macro_f1 = f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )

            test_pred = model.predict(X_test_scaled)

            test_accuracy = accuracy_score(y_test, test_pred)

            test_macro_f1 = f1_score(
                y_test,
                test_pred,
                average="macro",
                zero_division=0
            )

            test_balanced_accuracy = balanced_accuracy_score(
                y_test,
                test_pred
            )

            test_mcc = matthews_corrcoef(y_test, test_pred)

            validation_results.append({
                "model": model_name,
                "validation_macro_f1": val_macro_f1
            })

            result = {
                "task": task_name,
                "model": model_name,
                "validation_macro_f1": val_macro_f1,
                "test_accuracy": test_accuracy,
                "test_macro_f1": test_macro_f1,
                "test_balanced_accuracy": test_balanced_accuracy,
                "test_mcc": test_mcc
            }

            # Income-specific ordinal metric
            if task_name == "income_class":

                class_order = [
                    "low_or_lower_middle",
                    "middle",
                    "high"
                ]

                y_test_labels = label_encoder.inverse_transform(y_test)
                y_pred_labels = label_encoder.inverse_transform(test_pred)

                result["income_adjacent_class_accuracy"] = (
                    adjacent_class_accuracy(
                        y_test_labels,
                        y_pred_labels,
                        class_order
                    )
                )

            all_results.append(result)

            print(f"Validation macro F1: {val_macro_f1:.4f}")
            print(f"Test accuracy: {test_accuracy:.4f}")
            print(f"Test macro F1: {test_macro_f1:.4f}")
            print(f"Test balanced accuracy: {test_balanced_accuracy:.4f}")
            print(f"Test MCC: {test_mcc:.4f}")

            if task_name == "income_class":
                print(
                    f"Income adjacent-class accuracy: "
                    f"{result['income_adjacent_class_accuracy']:.4f}"
                )

            print("\nClassification report:")
            print(
                classification_report(
                    y_test,
                    test_pred,
                    # [FIX] household split: a rare class can be missing from the test set
                    labels=np.arange(len(label_encoder.classes_)),
                    target_names=label_encoder.classes_,
                    zero_division=0
                )
            )

            plot_confusion_matrix(
                y_test,
                test_pred,
                label_encoder,
                task_name,
                model_name
            )

        except Exception as error:
            print(f"Failed to train {model_name}: {error}")

    # --------------------------------------------------------
    # Select best model using validation macro F1
    # --------------------------------------------------------
    validation_df = pd.DataFrame(validation_results)

    best_model_name = validation_df.sort_values(
        by="validation_macro_f1",
        ascending=False
    ).iloc[0]["model"]

    best_result = next(
        result for result in all_results
        if result["task"] == task_name
        and result["model"] == best_model_name
    )

    best_model_results.append(best_result)

    print(
        f"\nBest model for {task_name}: {best_model_name}"
    )


# ------------------------------------------------------------
# 16. TWO-STAGE OCCUPATION MODEL
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TWO-STAGE OCCUPATION MODEL")
print("=" * 70)

occupation_data = person_df.dropna(
    subset=["label_occupation_simplified"]
).reset_index(drop=True)

X_occ = occupation_data[FEATURE_COLUMNS]

binary_encoder = LabelEncoder()
y_binary = binary_encoder.fit_transform(
    occupation_data["binary_employment"]
)

# [BUG] X_train_binary, X_temp_binary, y_train_binary, y_temp_binary = train_test_split(
# [BUG]     X_occ,
# [BUG]     y_binary,
# [BUG]     test_size=0.30,
# [BUG]     stratify=y_binary,
# [BUG]     random_state=42
# [BUG] )
# [BUG]
# [BUG] X_val_binary, X_test_binary, y_val_binary, y_test_binary = train_test_split(
# [BUG]     X_temp_binary,
# [BUG]     y_temp_binary,
# [BUG]     test_size=0.6667,
# [BUG]     stratify=y_temp_binary,
# [BUG]     random_state=42
# [BUG] )
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
X_train_binary, X_val_binary, X_test_binary, y_train_binary, y_val_binary, y_test_binary = (
    household_split(X_occ, y_binary, household_of(occupation_data))
)

binary_scaler = StandardScaler()

X_train_binary_scaled = binary_scaler.fit_transform(X_train_binary)
X_val_binary_scaled = binary_scaler.transform(X_val_binary)
X_test_binary_scaled = binary_scaler.transform(X_test_binary)

try:
    from imblearn.over_sampling import SMOTE

    smote_binary = SMOTE(
        random_state=42,
        k_neighbors=3
    )

    X_train_binary_model, y_train_binary_model = smote_binary.fit_resample(
        X_train_binary_scaled,
        y_train_binary
    )

except ImportError:
    X_train_binary_model = X_train_binary_scaled
    y_train_binary_model = y_train_binary

binary_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

binary_model.fit(
    X_train_binary_model,
    y_train_binary_model
)

binary_pred = binary_model.predict(X_test_binary_scaled)

binary_accuracy = accuracy_score(
    y_test_binary,
    binary_pred
)

binary_macro_f1 = f1_score(
    y_test_binary,
    binary_pred,
    average="macro",
    zero_division=0
)

print(f"Stage 1 binary employment accuracy: {binary_accuracy:.4f}")
print(f"Stage 1 binary employment macro F1: {binary_macro_f1:.4f}")

print("\nStage 1 classification report:")
print(
    classification_report(
        y_test_binary,
        binary_pred,
        # [FIX] household split: a rare class can be missing from the test set
        labels=np.arange(len(binary_encoder.classes_)),
        target_names=binary_encoder.classes_,
        zero_division=0
    )
)

# Stage 2: classify non-employed persons
non_employed_data = occupation_data[
    occupation_data["label_occupation_simplified"] != "employed"
].reset_index(drop=True)

if len(non_employed_data) > 50:

    X_stage2 = non_employed_data[FEATURE_COLUMNS]
    y_stage2 = non_employed_data["label_occupation_simplified"]

    stage2_encoder = LabelEncoder()
    y_stage2_encoded = stage2_encoder.fit_transform(y_stage2)

    # [BUG] X_train_stage2, X_temp_stage2, y_train_stage2, y_temp_stage2 = train_test_split(
    # [BUG]     X_stage2,
    # [BUG]     y_stage2_encoded,
    # [BUG]     test_size=0.30,
    # [BUG]     stratify=y_stage2_encoded,
    # [BUG]     random_state=42
    # [BUG] )
    # [BUG]
    # [BUG] X_val_stage2, X_test_stage2, y_val_stage2, y_test_stage2 = train_test_split(
    # [BUG]     X_temp_stage2,
    # [BUG]     y_temp_stage2,
    # [BUG]     test_size=0.6667,
    # [BUG]     stratify=y_temp_stage2,
    # [BUG]     random_state=42
    # [BUG] )
    # [FIX] Splitting PEOPLE at random put members of the same household in both train and test
    #       (they share income, vehicles and travel patterns), which inflates scores.
    #       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
    X_train_stage2, X_val_stage2, X_test_stage2, y_train_stage2, y_val_stage2, y_test_stage2 = (
        household_split(X_stage2, y_stage2_encoded, household_of(non_employed_data))
    )

    stage2_scaler = StandardScaler()

    X_train_stage2_scaled = stage2_scaler.fit_transform(X_train_stage2)
    X_test_stage2_scaled = stage2_scaler.transform(X_test_stage2)

    try:
        smote_stage2 = SMOTE(
            sampling_strategy="not majority",
            random_state=42,
            k_neighbors=3
        )

        X_train_stage2_model, y_train_stage2_model = smote_stage2.fit_resample(
            X_train_stage2_scaled,
            y_train_stage2
        )

    except ImportError:
        X_train_stage2_model = X_train_stage2_scaled
        y_train_stage2_model = y_train_stage2

    stage2_model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    stage2_model.fit(
        X_train_stage2_model,
        y_train_stage2_model
    )

    stage2_pred = stage2_model.predict(X_test_stage2_scaled)

    stage2_accuracy = accuracy_score(
        y_test_stage2,
        stage2_pred
    )

    stage2_macro_f1 = f1_score(
        y_test_stage2,
        stage2_pred,
        average="macro",
        zero_division=0
    )

    print(f"\nStage 2 non-employed occupation accuracy: {stage2_accuracy:.4f}")
    print(f"Stage 2 non-employed occupation macro F1: {stage2_macro_f1:.4f}")

    print("\nStage 2 classification report:")
    print(
        classification_report(
            y_test_stage2,
            stage2_pred,
            # [FIX] household split: a rare class can be missing from the test set
            labels=np.arange(len(stage2_encoder.classes_)),
            target_names=stage2_encoder.classes_,
            zero_division=0
        )
    )

    plot_confusion_matrix(
        y_test_stage2,
        stage2_pred,
        stage2_encoder,
        "two_stage_non_employed_occupation",
        "random_forest"
    )


# ------------------------------------------------------------
# 17. SAVE ALL RESULTS
# ------------------------------------------------------------
all_results_df = pd.DataFrame(all_results)

all_results_file = os.path.join(
    OUTPUT_FOLDER,
    "all_model_comparison_results.csv"
)

all_results_df.to_csv(all_results_file, index=False)

best_results_df = pd.DataFrame(best_model_results)

best_results_file = os.path.join(
    OUTPUT_FOLDER,
    "best_model_results.csv"
)

best_results_df.to_csv(best_results_file, index=False)

person_feature_file = os.path.join(
    OUTPUT_FOLDER,
    "person_level_improved_mobility_graph_features.csv"
)

person_df.to_csv(person_feature_file, index=False)


# ------------------------------------------------------------
# 18. CREATE COMPARISON CHARTS
# ------------------------------------------------------------
for task_name in all_results_df["task"].unique():

    task_results = all_results_df[
        all_results_df["task"] == task_name
    ].sort_values(
        by="validation_macro_f1",
        ascending=False
    )

    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=task_results,
        x="model",
        y="validation_macro_f1",
        palette="viridis"
    )

    plt.title(
        f"Validation Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Validation Macro F1")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

    chart_file = os.path.join(
        OUTPUT_FOLDER,
        f"validation_macro_f1_{task_name}.png"
    )

    plt.savefig(chart_file, dpi=300)
    plt.close()

    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=task_results,
        x="model",
        y="test_macro_f1",
        palette="mako"
    )

    plt.title(
        f"Test Macro F1 Comparison\nTask: {task_name}"
    )
    plt.xlabel("ML Model")
    plt.ylabel("Test Macro F1")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

    test_chart_file = os.path.join(
        OUTPUT_FOLDER,
        f"test_macro_f1_{task_name}.png"
    )

    plt.savefig(test_chart_file, dpi=300)
    plt.close()


# ------------------------------------------------------------
# 19. FEATURE IMPORTANCE FOR BEST OCCUPATION MODEL
# ------------------------------------------------------------
occupation_best_model_name = best_results_df[
    best_results_df["task"] == "occupation"
].iloc[0]["model"]

occupation_best_model = next(
    result for result in all_results
    if result["task"] == "occupation"
    and result["model"] == occupation_best_model_name
)

print("\nBest occupation model:", occupation_best_model_name)

# Re-train a Random Forest for feature importance
occupation_data = person_df.dropna(
    subset=["label_occupation_simplified"]
).reset_index(drop=True)

X_importance = occupation_data[FEATURE_COLUMNS]
y_importance = occupation_data["label_occupation_simplified"]

importance_encoder = LabelEncoder()
y_importance_encoded = importance_encoder.fit_transform(y_importance)

# [BUG] X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
# [BUG]     X_importance,
# [BUG]     y_importance_encoded,
# [BUG]     test_size=0.20,
# [BUG]     stratify=y_importance_encoded,
# [BUG]     random_state=42
# [BUG] )
# [FIX] Splitting PEOPLE at random put members of the same household in both train and test
#       (they share income, vehicles and travel patterns), which inflates scores.
#       Split by HOUSEHOLD instead, with the same 70 / 10 / 20 shares.
X_train_imp, _, X_test_imp, y_train_imp, _, y_test_imp = household_split(
    X_importance, y_importance_encoded, household_of(occupation_data), val_size=0
)

importance_scaler = StandardScaler()

X_train_imp_scaled = importance_scaler.fit_transform(X_train_imp)
X_test_imp_scaled = importance_scaler.transform(X_test_imp)

importance_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

importance_model.fit(X_train_imp_scaled, y_train_imp)

importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": importance_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

print("\nTop 20 important mobility-graph features:")
print(importance_df.head(20))

plt.figure(figsize=(10, 8))

sns.barplot(
    data=importance_df.head(20),
    x="importance",
    y="feature",
    palette="viridis"
)

plt.title("Top 20 Feature Importances for Occupation Prediction")
plt.tight_layout()

importance_file = os.path.join(
    OUTPUT_FOLDER,
    "feature_importance_occupation.png"
)

plt.savefig(importance_file, dpi=300)
plt.close()


# ------------------------------------------------------------
# 20. PRINT FINAL RESULTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL BEST-MODEL RESULTS")
print("=" * 70)
print(best_results_df)

print("\n" + "=" * 70)
print("FULL MODEL-COMPARISON RESULTS")
print("=" * 70)
print(all_results_df)

print(f"\nAll results saved in: {OUTPUT_FOLDER}")

Loaded dataset shape: (8645, 30)
Available columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr', 'person_id', 'home_zone']

Destination column standardized to: destination_zone

Simplified occupation distribution:
occupation_simplified
employed         52.562175
homemaker        20.647773
student_child    18.403702
non_employed      6.119144
other             2.267206
Name: proportion, dtype: float64

Simplified mode distribution:
mode_simplified
bus                            29.462117
active_transport               24.430307
two_wheel

KeyboardInterrupt: 

In [ ]:
# ============================================================
# COMPLETE CORRECTED BENGALURU HTS MOBILITY INFERENCE PIPELINE
#
# Required packages:
# pandas numpy scikit-learn matplotlib joblib
#
# Optional:
# xgboost lightgbm
#
# Outputs:
# - Cleaned column mapping and label audits
# - Person-level mobility features
# - Cross-validation model comparison
# - Selected-model test results
# - Classification reports and confusion matrices
# - Saved fitted pipelines and feature definitions
#
# Important:
# - Uses actual origin -> destination edges.
# - Does not assume each origin is home.
# - Does not use unknown home-return features.
# - Keeps household members together during splitting.
# - Imputation/scaling are fitted inside CV pipelines.
# - No SMOTE before cross-validation.
# - Income metrics do not change the training objective.
# ============================================================

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INPUT_FILE = Path("output/bengaluru_hts_noisy_medium.csv")
OUTPUT_DIR = Path("output/corrected_mobility_graph_results")

SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# Increase to 15-25 for a longer search after checking outputs.
SEARCH_ITERATIONS = 6
SEARCH_JOBS = -1

# "survey_enriched": includes observed purpose and travel cost.
# "mobility_only": excludes purpose and cost features.
#
# Survey purpose and cost should NOT be assumed available in
# telecom data. Choose mobility_only for a more restricted
# transfer-feature experiment.
FEATURE_PROFILE = "survey_enriched"

# False: missing-label records are excluded for that target.
# True: inconsistent person-level occupation/income stops execution.
STRICT_LABEL_CONSISTENCY = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. COLUMN MAPPING AND BASIC CLEANING
# ------------------------------------------------------------

ALIASES = {
    "household_id": ["household_id", "HH_ID", "Household_ID"],
    "person_id": ["person_id", "Person_ID", "PERSON_ID"],
    "origin_zone": [
        "origin_zone", "E2_Origin_PIN", "origin", "trip_origin"
    ],
    "destination_zone": [
        "destination_zone", "E3_Dest_PIN", "destination",
        "dest_zone", "trip_destination"
    ],
    "departure_time_hhmm": [
        "departure_time_hhmm", "E5_Departure"
    ],
    "trip_distance_km": [
        "trip_distance_km", "E7_Distance_km"
    ],
    "trip_duration_min": [
        "trip_duration_min", "E6_Travel_time_min"
    ],
    "travel_cost_inr": [
        "travel_cost_inr", "E10_Cost_INR"
    ],
    "trip_purpose": ["trip_purpose", "E4_Purpose"],
    "mode": ["mode", "E8_Primary_mode"],
    "occupation": ["occupation"],
    "household_monthly_income_inr": [
        "household_monthly_income_inr"
    ],
    "travel_date": ["travel_date", "Date_of_travel"],
    "trip_number": ["trip_number", "Trip_no_full"]
}


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    return re.sub(r"[\s_-]+", " ", text)


def simplify_occupation(value):
    text = normalize_text(value)

    if text in {"", "nan", "none", "unknown", "missing"}:
        return np.nan

    # Order matters: unemployed contains employed.
    if (
        "unemploy" in text
        or "non employ" in text
        or "not employ" in text
        or "retir" in text
    ):
        return "non_employed"

    if any(word in text for word in ["student", "child", "pupil"]):
        return "student_child"

    if any(
        word in text
        for word in ["homemaker", "home maker", "housewife", "househusband"]
    ):
        return "homemaker"

    if any(
        word in text
        for word in [
            "employ", "salar", "business", "worker",
            "professional", "self employ"
        ]
    ):
        return "employed"

    return "other"


def simplify_mode(value):
    text = normalize_text(value)
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in text for word in ['ride hail', 'rickshaw']):
        return "paid_private_vehicle"

    if text in {"", "nan", "none", "unknown", "missing"}:
        return np.nan

    if "metro" in text or "rail" in text or "train" in text:
        return "rail_based_public_transport"

    # Cab checks must precede private-car checks.
    if any(
        word in text
        for word in ["auto", "cab", "taxi", "uber", "ola"]
    ):
        return "paid_private_vehicle"

    if "bus" in text or "bmtc" in text:
        return "bus"

    if any(
        word in text
        for word in ["two wheeler", "motorcycle", "scooter", "motorbike"]
    ):
        return "two_wheeler"

    if any(
        word in text
        for word in ["walk", "bicycle", "cycling", "cycle"]
    ):
        return "active_transport"

    if "car" in text:
        return "private_car"

    return "other"


def parse_departure(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"(\d{1,2}):(\d{2})", str(value))

    if match is None:
        return np.nan

    hour, minute = map(int, match.groups())

    if not (0 <= hour < 24 and 0 <= minute < 60):
        return np.nan

    return float(hour * 60 + minute)


def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"CSV not found: {INPUT_FILE}\n"
            "Edit INPUT_FILE at the top of this script."
        )

    data = pd.read_csv(
        INPUT_FILE,
        dtype="string",
        low_memory=False
    )
    data.columns = data.columns.str.strip()

    print("Loaded dataset shape:", data.shape)
    print("Available columns:", data.columns.tolist())

    mapping = {}

    for canonical, choices in ALIASES.items():
        match = next((c for c in choices if c in data.columns), None)

        if match is not None:
            mapping[match] = canonical

    data = data.rename(columns=mapping)

    required = [
        "person_id", "origin_zone", "destination_zone",
        "departure_time_hhmm", "trip_distance_km",
        "trip_duration_min", "occupation", "mode",
        "household_monthly_income_inr"
    ]

    missing = [c for c in required if c not in data.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for optional in ["travel_cost_inr", "trip_purpose"]:
        if optional not in data.columns:
            data[optional] = pd.NA

    for column in [
        "person_id", "origin_zone", "destination_zone", "household_id"
    ]:
        if column in data.columns:
            data[column] = (
                data[column]
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
                .replace({
                    "": pd.NA,
                    "nan": pd.NA,
                    "None": pd.NA,
                    "<NA>": pd.NA
                })
            )

    missing_person = data["person_id"].isna().sum()
    data = data.dropna(subset=["person_id"]).copy()

    if "household_id" in data.columns:
        if data["household_id"].isna().any():
            raise ValueError(
                "Missing HH_ID values found. Resolve these before "
                "creating household-person identifiers."
            )

        # Length prefixes prevent ambiguous concatenation.
        data["traveler_id"] = (
            data["household_id"].str.len().astype(str)
            + ":" + data["household_id"]
            + "|" + data["person_id"]
        )
        data["split_group"] = data["household_id"]
    else:
        warnings.warn(
            "No household ID found. Splitting by person instead."
        )
        data["traveler_id"] = data["person_id"]
        data["split_group"] = data["traveler_id"]

    for column in [
        "trip_distance_km",
        "trip_duration_min",
        "travel_cost_inr",
        "household_monthly_income_inr"
    ]:
        values = (
            data[column].astype("string")
            .str.replace(",", "", regex=False)
        )
        data[column] = pd.to_numeric(values, errors="coerce")
        data.loc[data[column] < 0, column] = np.nan

    data["departure_min"] = data["departure_time_hhmm"].map(
        parse_departure
    )
    data["occupation_label"] = data["occupation"].map(
        simplify_occupation
    )
    data["mode_label"] = data["mode"].map(simplify_mode)

    for original, simplified, filename in [
        ("occupation", "occupation_label", "occupation_label_audit.csv"),
        ("mode", "mode_label", "mode_label_audit.csv")
    ]:
        (
            data.groupby(
                [original, simplified], dropna=False
            ).size().rename("trip_rows").reset_index()
            .to_csv(OUTPUT_DIR / filename, index=False)
        )

    pd.DataFrame(
        [{"original": k, "standardized": v} for k, v in mapping.items()]
    ).to_csv(OUTPUT_DIR / "column_mapping.csv", index=False)

    print("Removed rows with missing Person_ID:", missing_person)
    print("Total trips:", len(data))
    print("Unique travelers:", data["traveler_id"].nunique())
    print("Independent split groups:", data["split_group"].nunique())

    return data


# ------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ------------------------------------------------------------

def entropy(series):
    proportions = series.dropna().value_counts(normalize=True)

    if proportions.empty:
        return np.nan

    return float(-(proportions * np.log(proportions)).sum())


def share(mask, valid):
    count = int(valid.sum())

    if count == 0:
        return np.nan

    return float(mask[valid].mean())


def safe_ratio(numerator, denominator):
    if (
        pd.isna(numerator)
        or pd.isna(denominator)
        or denominator <= 0
    ):
        return np.nan

    return float(numerator / denominator)


def summarize_numeric(features, prefix, values):
    # [BUG] values = values.dropna()
    # [FIX] Now that trip underreporting really drops trips, some people have only one trip.
    #       These columns are pandas nullable numbers, whose std() of one value is <NA>, and
    #       float(<NA>) raises TypeError. Use plain floats (std of one value -> NaN).
    values = values.dropna().astype(float)

    for name, function in [
        ("mean", "mean"),
        ("median", "median"),
        ("min", "min"),
        ("max", "max"),
        ("std", "std")
    ]:
        features[f"{prefix}_{name}"] = (
            float(getattr(values, function)())
            if not values.empty else np.nan
        )

    features[f"{prefix}_missing_share"] = (
        1 - len(values) / features["trip_count"]
    )


def consistent_label(series):
    values = series.dropna().unique()

    if len(values) == 1:
        return values[0]

    # Do not silently choose an arbitrary label from conflicts.
    return np.nan


def dominant_label(series):
    counts = series.dropna().value_counts()

    if counts.empty:
        return np.nan, np.nan, False

    winners = counts[counts == counts.max()].index

    # A primary mode based on trip frequency is ambiguous in ties.
    if len(winners) != 1:
        return np.nan, float(counts.max() / counts.sum()), True

    return (
        winners[0],
        float(counts.max() / counts.sum()),
        False
    )


def income_class(value):
    if pd.isna(value):
        return np.nan

    if value < 50000:
        return "low_or_lower_middle"

    if value < 100000:
        return "middle"

    return "high"


def build_features(data):
    records = []

    for traveler, trips in data.groupby("traveler_id", sort=False):
        n = len(trips)

        f = {
            "traveler_id": traveler,
            "split_group": trips["split_group"].iloc[0],
            "trip_count": n
        }

        valid_edges = trips.dropna(
            subset=["origin_zone", "destination_zone"]
        )
        nodes = pd.concat([
            valid_edges["origin_zone"],
            valid_edges["destination_zone"]
        ]).unique()

        edges = valid_edges.groupby(
            ["origin_zone", "destination_zone"]
        ).size()

        nonself = valid_edges[
            valid_edges["origin_zone"] != valid_edges["destination_zone"]
        ]
        nonself_edges = nonself.groupby(
            ["origin_zone", "destination_zone"]
        ).size()

        node_count = len(nodes)

        f["graph_nodes"] = node_count
        f["graph_edges"] = len(edges)
        f["graph_nonself_edges"] = len(nonself_edges)
        f["graph_valid_trip_share"] = len(valid_edges) / n
        f["graph_density"] = (
            len(nonself_edges) / (node_count * (node_count - 1))
            if node_count > 1 else 0.0
        )
        f["graph_selfloop_share"] = (
            float(
                (
                    valid_edges["origin_zone"]
                    == valid_edges["destination_zone"]
                ).mean()
            ) if len(valid_edges) else np.nan
        )
        f["graph_max_edge_weight"] = (
            float(edges.max()) if len(edges) else np.nan
        )
        f["graph_mean_edge_weight"] = (
            float(edges.mean()) if len(edges) else np.nan
        )
        f["graph_dominant_edge_share"] = (
            float(edges.max() / edges.sum()) if len(edges) else np.nan
        )
        f["trips_per_node"] = safe_ratio(n, node_count)
        f["unique_origins"] = trips["origin_zone"].nunique()
        f["unique_destinations"] = trips["destination_zone"].nunique()
        f["destination_entropy"] = entropy(trips["destination_zone"])
        f["origin_entropy"] = entropy(trips["origin_zone"])

        edge_set = set(nonself_edges.index.tolist())
        f["graph_reciprocity"] = (
            sum((v, u) in edge_set for u, v in edge_set) / len(edge_set)
            if edge_set else 0.0
        )

        distance = trips["trip_distance_km"]
        duration = trips["trip_duration_min"]
        time = trips["departure_min"]
        cost = trips["travel_cost_inr"]

        summarize_numeric(f, "distance", distance)
        summarize_numeric(f, "duration", duration)

        valid_distance = distance.notna()

        f["short_trip_share"] = share(
            distance < 3, valid_distance
        )
        f["medium_trip_share"] = share(
            (distance >= 3) & (distance < 10), valid_distance
        )
        f["long_trip_share"] = share(
            distance >= 10, valid_distance
        )
        f["distance_range"] = (
            distance.max() - distance.min()
            if valid_distance.any() else np.nan
        )

        # Speed is computed only from valid positive durations.
        speed_valid = distance.notna() & duration.notna() & (duration > 0)
        speed = distance[speed_valid] / (duration[speed_valid] / 60)
        summarize_numeric(f, "speed", speed)

        valid_time = time.notna()
        f["time_missing_share"] = float(time.isna().mean())

        windows = {
            "morning_share": (360, 720),
            "afternoon_share": (720, 1020),
            "evening_share": (1020, 1200),
            "peak_morning_share": (480, 600),
            "peak_evening_share": (1020, 1140)
        }

        for name, (start, end) in windows.items():
            f[name] = share(
                (time >= start) & (time < end), valid_time
            )

        angles = 2 * np.pi * time.dropna() / 1440

        if len(angles):
            f["time_sin_mean"] = float(np.sin(angles).mean())
            f["time_cos_mean"] = float(np.cos(angles).mean())
            f["time_concentration"] = float(np.hypot(
                f["time_sin_mean"], f["time_cos_mean"]
            ))
        else:
            f["time_sin_mean"] = np.nan
            f["time_cos_mean"] = np.nan
            f["time_concentration"] = np.nan

        f["peak_trip_share"] = (
            f["peak_morning_share"] + f["peak_evening_share"]
        )

        purpose = trips["trip_purpose"].map(normalize_text)
        valid_purpose = purpose.ne("")

        masks = {
            "work": purpose.str.contains(
                r"\bwork\b|office|business", regex=True
            ),
            "education": purpose.str.contains(
                r"education|school|college|university", regex=True
            ),
            "shopping": purpose.str.contains("shop", regex=False)
        }

        f["purpose_entropy"] = entropy(
            purpose.where(valid_purpose)
        )

        for name, mask in masks.items():
            f[f"{name}_trip_share"] = share(mask, valid_purpose)
            f[f"{name}_distance_mean"] = distance[mask].mean()

        # These are interactions, not proof of multi-day regularity.
        f["work_peak_interaction"] = (
            f["work_trip_share"] * f["peak_morning_share"]
        )
        f["education_peak_interaction"] = (
            f["education_trip_share"] * f["peak_morning_share"]
        )

        summarize_numeric(f, "cost", cost)

        valid_cost_distance = (
            cost.notna() & distance.notna() & (distance > 0)
        )
        f["cost_per_km"] = (
            float(
                cost[valid_cost_distance].sum()
                / distance[valid_cost_distance].sum()
            )
            if valid_cost_distance.any() else np.nan
        )

        f["label_occupation"] = consistent_label(
            trips["occupation_label"]
        )
        # [BUG] income = consistent_label(
        # [BUG]     trips["household_monthly_income_inr"]
        # [BUG] )
        # [BUG] f["label_income_class"] = income_class(income)
        # [FIX] Consistency was checked on exact rupee amounts. Once noise was added they differed on
        #       almost every trip, so 3,119 of 3,383 travellers lost their income label (leaving the
        #       51-household test set). Check the income CLASS instead.
        f["label_income_class"] = consistent_label(
            trips["household_monthly_income_inr"].map(income_class)
        )

        mode, dominant_share, tied = dominant_label(trips["mode_label"])
        f["label_primary_mode"] = mode

        # Label diagnostics are deliberately not model inputs.
        f["audit_mode_dominant_share"] = dominant_share
        f["audit_mode_tied"] = tied
        f["audit_occupation_conflict"] = (
            trips["occupation_label"].dropna().nunique() > 1
        )
        f["audit_income_conflict"] = (
            trips["household_monthly_income_inr"].dropna().nunique() > 1
        )

        records.append(f)

    result = pd.DataFrame(records)
    result = result.replace([np.inf, -np.inf], np.nan)

    conflicts = result[
        result["audit_occupation_conflict"]
        | result["audit_income_conflict"]
    ]
    conflicts.to_csv(OUTPUT_DIR / "label_conflicts.csv", index=False)

    if STRICT_LABEL_CONSISTENCY and not conflicts.empty:
        raise ValueError(
            "Conflicting person-level labels found. "
            "See label_conflicts.csv."
        )

    print("\nPerson-level feature table:", result.shape)
    print("Conflicting-label travelers:", len(conflicts))
    print("Tied primary-mode travelers:", result["audit_mode_tied"].sum())

    return result


# ------------------------------------------------------------
# 4. HOUSEHOLD-AWARE HOLDOUT AND CROSS-VALIDATION
# ------------------------------------------------------------

def make_holdout(table):
    groups = table["split_group"]
    targets = [
        "label_occupation", "label_income_class", "label_primary_mode"
    ]

    splitter = GroupShuffleSplit(
        n_splits=100,
        test_size=TEST_SIZE,
        random_state=SEED
    )

    for train_index, test_index in splitter.split(table, groups=groups):
        train = table.iloc[train_index]
        test = table.iloc[test_index]

        valid = True

        for target in targets:
            all_classes = set(table[target].dropna())
            train_classes = set(train[target].dropna())
            test_classes = set(test[target].dropna())

            if (
                len(all_classes) < 2
                or train_classes != all_classes
                or test_classes != all_classes
            ):
                valid = False
                break

            per_class_groups = (
                train.dropna(subset=[target])
                .groupby(target)["split_group"].nunique()
            )

            if per_class_groups.min() < 2:
                valid = False
                break

        if valid:
            assert set(train["split_group"]).isdisjoint(
                set(test["split_group"])
            )
            return train_index, test_index

    raise ValueError(
        "Could not create a household-disjoint split with adequate "
        "class coverage. Check rare classes and label audits."
    )


def make_cv(X, y, groups):
    minimum_groups = (
        pd.DataFrame({"y": y, "g": groups})
        .groupby("y")["g"].nunique().min()
    )

    maximum_folds = min(CV_FOLDS, int(minimum_groups))
    expected = set(np.unique(y))

    for folds in range(maximum_folds, 1, -1):
        for attempt in range(10):
            cv = StratifiedGroupKFold(
                n_splits=folds,
                shuffle=True,
                random_state=SEED + attempt
            )
            splits = list(cv.split(X, y, groups))

            if all(
                set(y[a]) == expected and set(y[b]) == expected
                for a, b in splits
            ):
                return splits

    raise ValueError(
        "Insufficient independent groups for class-complete CV folds."
    )


# ------------------------------------------------------------
# 5. MODEL CANDIDATES
# ------------------------------------------------------------

def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def candidates():
    models = {
        "majority_baseline": (
            DummyClassifier(strategy="most_frequent"), {}
        ),
        "logistic_regression": (
            LogisticRegression(
                max_iter=4000,
                random_state=SEED
            ),
            {
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"]
            }
        ),
        "random_forest": (
            RandomForestClassifier(
                random_state=SEED,
                n_jobs=1
            ),
            {
                "model__n_estimators": [200, 400],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [
                    None, "balanced", "balanced_subsample"
                ]
            }
        ),
        "extra_trees": (
            ExtraTreesClassifier(
                random_state=SEED,
                n_jobs=1
            ),
            {
                "model__n_estimators": [200, 400],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [None, "balanced"]
            }
        )
    }

    try:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = (
            LGBMClassifier(
                random_state=SEED,
                n_jobs=1,
                verbosity=-1,
                subsample_freq=1
            ),
            {
                "model__n_estimators": [150, 300, 500],
                "model__learning_rate": [0.03, 0.07],
                "model__num_leaves": [7, 15, 31],
                "model__min_child_samples": [10, 25, 50],
                "model__class_weight": [None, "balanced"],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        print("Optional LightGBM unavailable; skipping.")

    try:
        from xgboost import XGBClassifier

        models["xgboost"] = (
            XGBClassifier(
                random_state=SEED,
                n_jobs=1,
                tree_method="hist",
                eval_metric="mlogloss"
            ),
            {
                "model__n_estimators": [150, 300, 500],
                "model__max_depth": [2, 4, 6],
                "model__learning_rate": [0.03, 0.07],
                "model__min_child_weight": [1, 5, 10],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        print("Optional XGBoost unavailable; skipping.")

    return models


# ------------------------------------------------------------
# 6. REPORTING
# ------------------------------------------------------------

def save_confusion(y_true, y_pred, classes, task):
    labels = np.arange(len(classes))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)

    pd.DataFrame(
        matrix, index=classes, columns=classes
    ).to_csv(OUTPUT_DIR / f"{task}_confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set_xticks(labels)
    ax.set_yticks(labels)
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{task}: selected model")

    for row in labels:
        for col in labels:
            ax.text(
                col, row, str(matrix[row, col]),
                ha="center", va="center"
            )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{task}_confusion_matrix.png",
        dpi=200
    )
    plt.close(fig)


# ------------------------------------------------------------
# 7. MAIN WORKFLOW
# ------------------------------------------------------------

def main():
    if FEATURE_PROFILE not in {"survey_enriched", "mobility_only"}:
        raise ValueError("Invalid FEATURE_PROFILE.")

    data = load_data()
    people = build_features(data)

    excluded = {
        "traveler_id", "split_group",
        "label_occupation", "label_income_class", "label_primary_mode"
    }

    features = [
        c for c in people.columns
        if c not in excluded and not c.startswith("audit_")
    ]

    if FEATURE_PROFILE == "mobility_only":
        survey_only = {
            "purpose_entropy",
            "work_trip_share", "education_trip_share",
            "shopping_trip_share",
            "work_distance_mean", "education_distance_mean",
            "shopping_distance_mean",
            "work_peak_interaction", "education_peak_interaction"
        }
        features = [
            c for c in features
            if c not in survey_only and not c.startswith("cost_")
        ]

    people.to_csv(
        OUTPUT_DIR / "person_level_mobility_features.csv",
        index=False
    )

    train_index, test_index = make_holdout(people)

    people["split"] = "development"
    people.loc[test_index, "split"] = "test"

    people[["traveler_id", "split_group", "split"]].to_csv(
        OUTPUT_DIR / "split_assignments.csv", index=False
    )

    print("\nFeature profile:", FEATURE_PROFILE)
    print("Number of candidate features:", len(features))
    print("Development travelers:", len(train_index))
    print("Held-out test travelers:", len(test_index))

    targets = {
        "occupation": "label_occupation",
        "income_class": "label_income_class",
        "primary_mode": "label_primary_mode"
    }

    all_cv_results = []
    final_results = []
    failures = []

    for task, target in targets.items():
        print("\n" + "=" * 70)
        print("TASK:", task)
        print("=" * 70)

        development = people.iloc[train_index].dropna(
            subset=[target]
        ).copy()
        test = people.iloc[test_index].dropna(
            subset=[target]
        ).copy()

        # Remove entirely missing columns based on development data only.
        task_features = [
            c for c in features if development[c].notna().any()
        ]

        # [BUG] X_dev = development[task_features].astype(float)
        # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
        #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
        #       the model pipelines' SimpleImputer already fills NaN.
        X_dev = development[task_features].astype(object).where(development[task_features].notna(), np.nan).astype(float)
        # [BUG] X_test = test[task_features].astype(float)
        # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
        #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
        #       the model pipelines' SimpleImputer already fills NaN.
        X_test = test[task_features].astype(object).where(test[task_features].notna(), np.nan).astype(float)

        encoder = LabelEncoder()
        y_dev = encoder.fit_transform(development[target])
        y_test = encoder.transform(test[target])

        groups = development["split_group"].to_numpy()
        folds = make_cv(X_dev, y_dev, groups)

        labels = np.arange(len(encoder.classes_))
        scorer = make_scorer(
            f1_score,
            labels=labels,
            average="macro",
            zero_division=0
        )

        print("Development label counts:")
        print(development[target].value_counts())
        print("CV folds:", len(folds))

        trained = {}
        task_rows = []

        for name, (model, parameters) in candidates().items():
            print(f"\nTraining/searching: {name}")

            try:
                search = RandomizedSearchCV(
                    estimator=make_pipeline(model),
                    param_distributions=parameters,
                    n_iter=(
                        SEARCH_ITERATIONS if parameters else 1
                    ),
                    scoring=scorer,
                    cv=folds,
                    refit=True,
                    n_jobs=SEARCH_JOBS,
                    random_state=SEED,
                    error_score="raise",
                    return_train_score=False
                )

                search.fit(X_dev, y_dev)
                best_position = search.best_index_

                row = {
                    "task": task,
                    "model": name,
                    "cv_macro_f1": float(search.best_score_),
                    "cv_std": float(
                        search.cv_results_["std_test_score"][
                            best_position
                        ]
                    ),
                    "best_parameters": json.dumps(
                        search.best_params_, default=str
                    )
                }

                trained[name] = search.best_estimator_
                task_rows.append(row)
                all_cv_results.append(row)

                pd.DataFrame(search.cv_results_).to_csv(
                    OUTPUT_DIR / f"{task}_{name}_search.csv",
                    index=False
                )

                print(
                    f"CV macro F1: {row['cv_macro_f1']:.4f} "
                    f"+/- {row['cv_std']:.4f}"
                )

            except Exception as error:
                print(f"Model failed: {error}")
                failures.append({
                    "task": task,
                    "model": name,
                    "error": str(error)
                })

        if not task_rows:
            raise RuntimeError(f"No model succeeded for {task}.")

        ranking = pd.DataFrame(task_rows).sort_values(
            "cv_macro_f1", ascending=False
        )
        winner = ranking.iloc[0]["model"]
        selected = trained[winner]

        print("\nSelected using CV only:", winner)

        # Test predictions are made only for the selected model.
        predictions = selected.predict(X_test).astype(int)

        result = {
            "task": task,
            "selected_model": winner,
            "feature_profile": FEATURE_PROFILE,
            "development_n": len(development),
            "test_n": len(test),
            "selected_cv_macro_f1": ranking.iloc[0]["cv_macro_f1"],
            "test_accuracy": accuracy_score(y_test, predictions),
            "test_macro_f1": f1_score(
                y_test, predictions,
                labels=labels,
                average="macro",
                zero_division=0
            ),
            "test_balanced_accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "test_mcc": matthews_corrcoef(y_test, predictions)
        }

        true_names = encoder.inverse_transform(y_test)
        predicted_names = encoder.inverse_transform(predictions)

        if task == "income_class":
            order = {
                "low_or_lower_middle": 0,
                "middle": 1,
                "high": 2
            }
            true_order = np.array([order[v] for v in true_names])
            pred_order = np.array([order[v] for v in predicted_names])

            result["income_ordinal_mae"] = float(
                np.abs(true_order - pred_order).mean()
            )
            result["income_quadratic_weighted_kappa"] = (
                cohen_kappa_score(
                    true_order, pred_order,
                    labels=[0, 1, 2],
                    weights="quadratic"
                )
            )

            # Supplementary only: always predicting "middle"
            # achieves 100% on this metric with three ordered classes.
            result["income_within_one_class_accuracy"] = float(
                (np.abs(true_order - pred_order) <= 1).mean()
            )

        final_results.append(result)

        report = classification_report(
            y_test, predictions,
            labels=labels,
            target_names=encoder.classes_,
            zero_division=0,
            output_dict=True
        )
        pd.DataFrame(report).transpose().to_csv(
            OUTPUT_DIR / f"{task}_classification_report.csv"
        )

        print(classification_report(
            y_test, predictions,
            labels=labels,
            target_names=encoder.classes_,
            zero_division=0
        ))

        save_confusion(
            y_test, predictions, encoder.classes_, task
        )

        prediction_table = test[
            ["traveler_id", "split_group"]
        ].copy()
        prediction_table["true_label"] = true_names
        prediction_table["predicted_label"] = predicted_names
        prediction_table.to_csv(
            OUTPUT_DIR / f"{task}_test_predictions.csv",
            index=False
        )

        # Importance uses development data, not test data.
        importance = permutation_importance(
            selected,
            X_dev,
            y_dev,
            scoring=scorer,
            n_repeats=3,
            random_state=SEED,
            n_jobs=SEARCH_JOBS
        )

        importance_table = pd.DataFrame({
            "feature": task_features,
            "development_permutation_importance": (
                importance.importances_mean
            ),
            "importance_std": importance.importances_std
        }).sort_values(
            "development_permutation_importance",
            ascending=False
        )
        importance_table.to_csv(
            OUTPUT_DIR / f"{task}_development_feature_importance.csv",
            index=False
        )

        joblib.dump(
            {
                "pipeline": selected,
                "label_encoder": encoder,
                "features": task_features,
                "feature_profile": FEATURE_PROFILE,
                "target": target
            },
            OUTPUT_DIR / f"{task}_selected_model.joblib"
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(
            ranking["model"],
            ranking["cv_macro_f1"],
            yerr=ranking["cv_std"],
            capsize=4
        )
        ax.set_ylabel("Cross-validation macro F1")
        ax.set_title(f"{task}: development-set model comparison")
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"{task}_cv_comparison.png",
            dpi=200
        )
        plt.close(fig)

        # Save incrementally so completed tasks survive later errors.
        pd.DataFrame(all_cv_results).to_csv(
            OUTPUT_DIR / "all_model_cv_comparison.csv",
            index=False
        )
        pd.DataFrame(final_results).to_csv(
            OUTPUT_DIR / "selected_model_test_results.csv",
            index=False
        )

    pd.DataFrame(
        failures, columns=["task", "model", "error"]
    ).to_csv(OUTPUT_DIR / "model_failures.csv", index=False)

    configuration = {
        "input_file": str(INPUT_FILE),
        "feature_profile": FEATURE_PROFILE,
        "random_seed": SEED,
        "test_fraction_requested": TEST_SIZE,
        "cv_folds_requested": CV_FOLDS,
        "search_iterations": SEARCH_ITERATIONS,
        "features": features,
        "notes": [
            "PIN codes are graph nodes, not exact locations.",
            "No verified home-return feature is used.",
            "Primary mode is the most frequent observed trip mode.",
            "Travelers with tied primary modes are excluded "
            "from primary-mode training/evaluation.",
            "Conflicting occupation/income labels are excluded "
            "for the relevant target.",
            "Survey purpose and cost are not assumed available "
            "from telecom observations.",
            "Model performance on synthetic data does not establish "
            "performance on real survey or telecom data.",
            "Previously inspected test data are not a fresh "
            "confirmatory holdout."
        ]
    }

    with open(
        OUTPUT_DIR / "run_configuration.json",
        "w",
        encoding="utf-8"
    ) as handle:
        json.dump(configuration, handle, indent=2, default=str)

    print("\n" + "=" * 70)
    print("CROSS-VALIDATION MODEL COMPARISON")
    print("=" * 70)
    print(pd.DataFrame(all_cv_results).to_string(index=False))

    print("\n" + "=" * 70)
    print("SELECTED-MODEL HELD-OUT TEST RESULTS")
    print("=" * 70)
    print(pd.DataFrame(final_results).to_string(index=False))

    print("\nCompleted.")
    print("Results directory:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()

Loaded dataset shape: (8645, 28)
Available columns: ['HH_ID', 'Person_ID', 'Date_of_travel', 'Day_of_week', 'Trip_no_full', 'E2_Origin_type', 'E2_Origin_PIN', 'E3_Destination_name', 'E3_Dest_PIN', 'E4_Purpose', 'E5_Departure', 'E6_Travel_time_min', 'E7_Distance_km', 'E8_Primary_mode', 'E9_Access_mode', 'E10_Cost_INR', 'E11_Members_together', 'E12_Return_trip', 'Usual_commute_leg', 'In_field_diary', 'mode', 'trip_purpose', 'trip_distance_km', 'trip_duration_min', 'departure_time_hhmm', 'travel_cost_inr', 'occupation', 'household_monthly_income_inr']
Removed rows with missing Person_ID: 0
Total trips: 8645
Unique travelers: 3383
Independent split groups: 1165

Person-level feature table: (3383, 73)
Conflicting-label travelers: 0
Tied primary-mode travelers: 250

Feature profile: survey_enriched
Number of candidate features: 64
Development travelers: 2689
Held-out test travelers: 694

TASK: occupation
Development label counts:
label_occupation
employed         1353
student_child     602
h

In [ ]:
# ============================================================
# CORRECTED BENGALURU HTS MOBILITY INFERENCE
#
# Required packages:
# pandas numpy scikit-learn matplotlib joblib
#
# Optional models:
# lightgbm xgboost
#
# Main corrections:
# - Actual origin -> destination graphs
# - Correct unemployed occupation mapping
# - Income category consistency checked at household level
# - Optional authoritative household-income file
# - No arbitrary repair of genuinely conflicting labels
# - Household-aware cross-validation
# - Pipeline-based preprocessing
# - CV model selection; selected model tested afterward
# - Previous split reused when available
# ============================================================

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INPUT_FILE = Path("output/bengaluru_hts_noisy_medium.csv")
OUTPUT_DIR = Path("output/label_corrected_mobility_results")

# Reuse the household split from your previous run if available.
PREVIOUS_SPLIT_FILE = Path(
    "output/corrected_mobility_graph_results/split_assignments.csv"
)

# Optional authoritative household income table.
#
# Set to a Path if you have a VERIFIED household-level file:
#
# VERIFIED_INCOME_FILE = Path("data/verified_household_income.csv")
#
# Required columns in that file:
# HH_ID, household_monthly_income_inr
#
# Leave as None when unavailable.
VERIFIED_INCOME_FILE = None

SEED = 42
TEST_SIZE = 0.20
MAX_CV_FOLDS = 5

# Increase after first checking label-audit outputs.
SEARCH_ITERATIONS = 10
SEARCH_JOBS = -1

# Options: "survey_enriched", "mobility_only"
FEATURE_PROFILE = "survey_enriched"

# Require at least this many development households per class.
MIN_DEVELOPMENT_GROUPS_PER_CLASS = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. TEXT / LABEL HELPERS
# ------------------------------------------------------------

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA
        })
    )


def text(value):
    if pd.isna(value):
        return ""
    return re.sub(
        r"[\s_-]+", " ", str(value).strip().lower()
    )


def occupation_label(value):
    value = text(value)

    if value in {"", "nan", "none", "unknown", "missing"}:
        return np.nan

    if any(s in value for s in [
        "unemploy", "non employ", "not employ", "retir"
    ]):
        return "non_employed"

    if any(s in value for s in ["student", "child", "pupil"]):
        return "student_child"

    if any(s in value for s in [
        "homemaker", "home maker", "housewife", "househusband"
    ]):
        return "homemaker"

    if any(s in value for s in [
        "employ", "salar", "business",
        "worker", "professional"
    ]):
        return "employed"

    return "other"


def mode_label(value):
    value = text(value)
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in value for word in ['ride hail', 'rickshaw']):
        return "paid_private_vehicle"

    if value in {"", "nan", "none", "unknown", "missing"}:
        return np.nan

    if any(s in value for s in ["metro", "rail", "train"]):
        return "rail_based_public_transport"

    if any(s in value for s in [
        "auto", "cab", "taxi", "uber", "ola"
    ]):
        return "paid_private_vehicle"

    if "bus" in value or "bmtc" in value:
        return "bus"

    if any(s in value for s in [
        "two wheeler", "motorcycle", "motorbike", "scooter"
    ]):
        return "two_wheeler"

    if any(s in value for s in [
        "walk", "bicycle", "cycling", "cycle"
    ]):
        return "active_transport"

    if "car" in value:
        return "private_car"

    return "other"


def income_class(value):
    if pd.isna(value):
        return np.nan
    if value < 50000:
        return "low_or_lower_middle"
    if value < 100000:
        return "middle"
    return "high"


def departure_minutes(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"(\d{1,2}):(\d{2})", str(value))
    if match is None:
        return np.nan

    hour, minute = map(int, match.groups())

    if 0 <= hour < 24 and 0 <= minute < 60:
        return float(hour * 60 + minute)

    return np.nan


def unique_label(series):
    values = series.dropna().unique()
    return values[0] if len(values) == 1 else np.nan


def entropy(series):
    probabilities = series.dropna().value_counts(normalize=True)

    if probabilities.empty:
        return np.nan

    return float(
        -(probabilities * np.log(probabilities)).sum()
    )


def conditional_share(mask, valid):
    return float(mask[valid].mean()) if valid.any() else np.nan


# ------------------------------------------------------------
# 3. LOAD ACTUAL SURVEY COLUMNS
# ------------------------------------------------------------

def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_FILE}"
        )

    data = pd.read_csv(
        INPUT_FILE, dtype="string", low_memory=False
    )
    data.columns = data.columns.str.strip()

    aliases = {
        "household_id": ["household_id", "HH_ID"],
        "person_id": ["person_id", "Person_ID"],
        "origin_zone": ["origin_zone", "E2_Origin_PIN"],
        "destination_zone": [
            "destination_zone", "E3_Dest_PIN"
        ],
        "distance": ["trip_distance_km", "E7_Distance_km"],
        "duration": [
            "trip_duration_min", "E6_Travel_time_min"
        ],
        "cost": ["travel_cost_inr", "E10_Cost_INR"],
        "departure": [
            "departure_time_hhmm", "E5_Departure"
        ],
        "purpose": ["trip_purpose", "E4_Purpose"],
        "mode_raw": ["mode", "E8_Primary_mode"],
        "occupation_raw": ["occupation"],
        "income": ["household_monthly_income_inr"]
    }

    mapping = {}

    for canonical, options in aliases.items():
        original = next(
            (c for c in options if c in data.columns),
            None
        )
        if original is not None:
            mapping[original] = canonical

    data = data.rename(columns=mapping)

    required = [
        "household_id", "person_id",
        "origin_zone", "destination_zone",
        "distance", "duration", "departure",
        "mode_raw", "occupation_raw", "income"
    ]

    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    for optional in ["cost", "purpose"]:
        if optional not in data.columns:
            data[optional] = pd.NA

    for column in [
        "household_id", "person_id",
        "origin_zone", "destination_zone"
    ]:
        data[column] = clean_id(data[column])

    if data["household_id"].isna().any():
        raise ValueError(
            "Missing HH_ID values. Resolve household IDs first."
        )

    data = data.dropna(subset=["person_id"]).copy()

    data["traveler_id"] = (
        data["household_id"].str.len().astype(str)
        + ":" + data["household_id"]
        + "|" + data["person_id"]
    )

    for column in ["distance", "duration", "cost", "income"]:
        values = (
            data[column].astype("string")
            .str.replace(",", "", regex=False)
        )
        data[column] = pd.to_numeric(values, errors="coerce")
        data.loc[data[column] < 0, column] = np.nan

    data["departure_min"] = data["departure"].map(
        departure_minutes
    )
    data["occupation_class"] = data["occupation_raw"].map(
        occupation_label
    )
    data["mode_class"] = data["mode_raw"].map(mode_label)
    data["income_class_row"] = data["income"].map(income_class)

    for original, label, filename in [
        (
            "occupation_raw",
            "occupation_class",
            "occupation_mapping_audit.csv"
        ),
        (
            "mode_raw",
            "mode_class",
            "mode_mapping_audit.csv"
        )
    ]:
        (
            data.groupby([original, label], dropna=False)
            .size().rename("trip_rows").reset_index()
            .to_csv(OUTPUT_DIR / filename, index=False)
        )

    pd.DataFrame([
        {"original": k, "standardized": v}
        for k, v in mapping.items()
    ]).to_csv(OUTPUT_DIR / "column_mapping.csv", index=False)

    print("Trips:", len(data))
    print("Travelers:", data["traveler_id"].nunique())
    print("Households:", data["household_id"].nunique())

    return data


# ------------------------------------------------------------
# 4. HOUSEHOLD INCOME AUDIT / LABEL RESOLUTION
# ------------------------------------------------------------

def resolve_household_income(data):
    audit = data.groupby("household_id").agg(
        trip_rows=("income", "size"),
        known_income_rows=("income", "count"),
        distinct_amounts=("income", "nunique"),
        distinct_classes=("income_class_row", "nunique")
    )

    observed_labels = (
        data.groupby("household_id")["income_class_row"]
        .agg(unique_label)
    )

    audit["observed_consistent_class"] = observed_labels

    audit["observed_status"] = np.select(
        [
            audit["distinct_classes"].eq(0),
            audit["distinct_classes"].eq(1),
            audit["distinct_classes"].gt(1)
        ],
        [
            "missing",
            "consistent_category",
            "conflicting_categories"
        ],
        default="unknown"
    )

    resolved = observed_labels.copy()
    source = "observed_household_category_consistency"

    if VERIFIED_INCOME_FILE is not None:
        verified_path = Path(VERIFIED_INCOME_FILE)

        verified = pd.read_csv(
            verified_path, dtype="string"
        )
        verified.columns = verified.columns.str.strip()

        if "HH_ID" in verified.columns:
            verified = verified.rename(
                columns={"HH_ID": "household_id"}
            )

        required = [
            "household_id", "household_monthly_income_inr"
        ]
        missing = [
            c for c in required if c not in verified.columns
        ]
        if missing:
            raise ValueError(
                f"Verified income file missing columns: {missing}"
            )

        verified["household_id"] = clean_id(
            verified["household_id"]
        )

        if verified["household_id"].isna().any():
            raise ValueError(
                "Verified income file contains missing household IDs."
            )

        amounts = (
            verified["household_monthly_income_inr"]
            .str.replace(",", "", regex=False)
        )
        verified["verified_amount"] = pd.to_numeric(
            amounts, errors="coerce"
        )

        if (
            verified["verified_amount"].isna().any()
            or verified["verified_amount"].lt(0).any()
        ):
            raise ValueError(
                "Verified income file contains invalid amounts."
            )

        if verified["household_id"].duplicated().any():
            raise ValueError(
                "Verified income file must have ONE row per household."
            )

        verified["verified_class"] = (
            verified["verified_amount"].map(income_class)
        )

        lookup = verified.set_index(
            "household_id"
        )["verified_class"]

        # Authoritative mode: use only verified household labels.
        # Do not silently fill unmatched households with noisy labels.
        resolved = pd.Series(
            audit.index.map(lookup),
            index=audit.index
        )
        source = "verified_household_income_file"

    audit["resolved_income_class"] = resolved
    audit["label_source"] = source
    audit.to_csv(
        OUTPUT_DIR / "household_income_audit.csv"
    )

    conflicting_ids = audit.index[
        audit["distinct_classes"] > 1
    ]
    data.loc[
        data["household_id"].isin(conflicting_ids),
        [
            "household_id", "person_id", "traveler_id",
            "income", "income_class_row"
        ]
    ].to_csv(
        OUTPUT_DIR / "income_conflicting_trip_rows.csv",
        index=False
    )

    print("\nHOUSEHOLD INCOME AUDIT")
    print(audit["observed_status"].value_counts())
    print("Income label source:", source)
    print("Resolved households:", resolved.notna().sum())

    return resolved


# ------------------------------------------------------------
# 5. PERSON-LEVEL FEATURES
# ------------------------------------------------------------

def add_numeric_summary(record, prefix, values, total_rows):
    # [BUG] valid = values.dropna()
    # [FIX] Now that trip underreporting really drops trips, some people have only one trip.
    #       These columns are pandas nullable numbers, whose std() of one value is <NA>, and
    #       float(<NA>) raises TypeError. Use plain floats (std of one value -> NaN).
    valid = values.dropna().astype(float)

    for name in ["mean", "median", "min", "max", "std"]:
        record[f"{prefix}_{name}"] = (
            float(getattr(valid, name)())
            if len(valid) else np.nan
        )

    record[f"{prefix}_missing_share"] = (
        1 - len(valid) / total_rows
    )


def build_people(data, household_income):
    records = []

    for traveler, trips in data.groupby(
        "traveler_id", sort=False
    ):
        n = len(trips)
        household = trips["household_id"].iloc[0]

        r = {
            "traveler_id": traveler,
            "household_id": household,
            "trip_count": n
        }

        valid_edges = trips.dropna(
            subset=["origin_zone", "destination_zone"]
        )

        nodes = pd.concat([
            valid_edges["origin_zone"],
            valid_edges["destination_zone"]
        ]).unique()

        edges = valid_edges.groupby(
            ["origin_zone", "destination_zone"]
        ).size()

        nonself = valid_edges[
            valid_edges["origin_zone"]
            != valid_edges["destination_zone"]
        ]
        nonself_edges = nonself.groupby(
            ["origin_zone", "destination_zone"]
        ).size()

        node_count = len(nodes)
        edge_set = set(nonself_edges.index.tolist())

        r["graph_nodes"] = node_count
        r["graph_edges"] = len(edges)
        r["graph_density"] = (
            len(nonself_edges) / (node_count * (node_count - 1))
            if node_count > 1 else 0.0
        )
        r["valid_edge_share"] = len(valid_edges) / n
        r["dominant_edge_share"] = (
            float(edges.max() / edges.sum())
            if len(edges) else np.nan
        )
        r["edge_repeat_intensity"] = (
            len(valid_edges) / len(edges)
            if len(edges) else np.nan
        )
        r["graph_reciprocity"] = (
            sum((b, a) in edge_set for a, b in edge_set)
            / len(edge_set)
            if edge_set else 0.0
        )
        r["selfloop_share"] = (
            float((
                valid_edges["origin_zone"]
                == valid_edges["destination_zone"]
            ).mean())
            if len(valid_edges) else np.nan
        )
        r["destination_entropy"] = entropy(
            trips["destination_zone"]
        )
        r["origin_entropy"] = entropy(trips["origin_zone"])
        r["unique_destinations"] = (
            trips["destination_zone"].nunique()
        )

        distance = trips["distance"]
        duration = trips["duration"]
        cost = trips["cost"]
        time = trips["departure_min"]

        add_numeric_summary(r, "distance", distance, n)
        add_numeric_summary(r, "duration", duration, n)
        add_numeric_summary(r, "cost", cost, n)

        valid_speed = (
            distance.notna()
            & duration.notna()
            & duration.gt(0)
        )
        speed = (
            distance[valid_speed]
            / (duration[valid_speed] / 60)
        )
        add_numeric_summary(r, "speed", speed, n)

        for name, mask in {
            "short_trip_share": distance.lt(3),
            "medium_trip_share": (
                distance.ge(3) & distance.lt(10)
            ),
            "long_trip_share": distance.ge(10)
        }.items():
            r[name] = conditional_share(mask, distance.notna())

        for name, start, end in [
            ("morning_share", 360, 720),
            ("afternoon_share", 720, 1020),
            ("evening_share", 1020, 1200),
            ("morning_peak_share", 480, 600),
            ("evening_peak_share", 1020, 1140)
        ]:
            r[name] = conditional_share(
                time.ge(start) & time.lt(end),
                time.notna()
            )

        angles = 2 * np.pi * time.dropna() / 1440
        r["time_sin_mean"] = (
            float(np.sin(angles).mean())
            if len(angles) else np.nan
        )
        r["time_cos_mean"] = (
            float(np.cos(angles).mean())
            if len(angles) else np.nan
        )
        r["time_missing_share"] = float(time.isna().mean())

        purpose = trips["purpose"].map(text)
        valid_purpose = purpose.ne("")

        r["purpose_entropy"] = entropy(
            purpose.where(valid_purpose)
        )

        for name, pattern in {
            "work": r"\bwork\b|office|business",
            "education": r"education|school|college|university",
            "shopping": r"shop"
        }.items():
            mask = purpose.str.contains(pattern, regex=True)
            r[f"purpose_{name}_share"] = conditional_share(
                mask, valid_purpose
            )
            r[f"purpose_{name}_distance"] = (
                distance[mask].mean()
            )

        valid_cost_distance = (
            cost.notna()
            & distance.notna()
            & distance.gt(0)
        )
        r["cost_per_km"] = (
            float(
                cost[valid_cost_distance].sum()
                / distance[valid_cost_distance].sum()
            )
            if valid_cost_distance.any() else np.nan
        )

        r["label_occupation"] = unique_label(
            trips["occupation_class"]
        )
        r["label_income_class"] = household_income.get(
            household, np.nan
        )

        counts = trips["mode_class"].dropna().value_counts()
        winners = (
            counts[counts == counts.max()].index
            if len(counts) else []
        )
        r["label_primary_mode"] = (
            winners[0] if len(winners) == 1 else np.nan
        )

        r["audit_mode_tied"] = len(winners) > 1
        r["audit_occupation_conflict"] = (
            trips["occupation_class"].dropna().nunique() > 1
        )

        records.append(r)

    people = pd.DataFrame(records).replace(
        [np.inf, -np.inf], np.nan
    )

    people.to_csv(
        OUTPUT_DIR / "person_level_features.csv",
        index=False
    )

    print("\nPerson-level table:", people.shape)
    print(
        "Occupation conflicts:",
        people["audit_occupation_conflict"].sum()
    )
    print("Mode ties:", people["audit_mode_tied"].sum())

    return people


# ------------------------------------------------------------
# 6. REUSABLE HOUSEHOLD SPLIT
# ------------------------------------------------------------

def assign_split(people):
    if PREVIOUS_SPLIT_FILE.exists():
        previous = pd.read_csv(
            PREVIOUS_SPLIT_FILE, dtype="string"
        )

        if not {"split_group", "split"}.issubset(previous.columns):
            raise ValueError(
                "Previous split file lacks split_group/split."
            )

        previous["split_group"] = clean_id(
            previous["split_group"]
        )

        consistency = previous.groupby(
            "split_group"
        )["split"].nunique()

        if consistency.gt(1).any():
            raise ValueError(
                "Previous split assigns a household to both partitions."
            )

        lookup = (
            previous.drop_duplicates("split_group")
            .set_index("split_group")["split"]
        )

        split = people["household_id"].map(lookup)

        if split.isna().any():
            raise ValueError(
                "Some households are absent from previous split. "
                "Use a deliberately defined new split rather than "
                "silently assigning them."
            )

        if not set(split.unique()).issubset({"development", "test"}):
            raise ValueError("Unexpected previous split labels.")

        print("\nReusing previous household split.")
        print(
            "Reminder: this test sample has already been inspected; "
            "results are exploratory, not fresh confirmation."
        )

    else:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=SEED
        )
        development_idx, test_idx = next(
            splitter.split(
                people, groups=people["household_id"]
            )
        )

        split = pd.Series(
            "development", index=people.index
        )
        split.iloc[test_idx] = "test"
        print("\nCreated a new household-disjoint split.")

    people["split"] = split

    development_households = set(
        people.loc[
            people["split"].eq("development"), "household_id"
        ]
    )
    test_households = set(
        people.loc[
            people["split"].eq("test"), "household_id"
        ]
    )
    assert development_households.isdisjoint(test_households)

    people[
        ["traveler_id", "household_id", "split"]
    ].to_csv(
        OUTPUT_DIR / "split_assignments.csv", index=False
    )

    return people


def cv_splits(X, y, groups):
    independent_counts = (
        pd.DataFrame({"label": y, "group": groups})
        .groupby("label")["group"].nunique()
    )

    maximum = min(
        MAX_CV_FOLDS, int(independent_counts.min())
    )
    classes = set(np.unique(y))

    for folds in range(maximum, 1, -1):
        for attempt in range(10):
            splitter = StratifiedGroupKFold(
                n_splits=folds,
                shuffle=True,
                random_state=SEED + attempt
            )
            splits = list(splitter.split(X, y, groups))

            if all(
                set(y[a]) == classes and set(y[b]) == classes
                for a, b in splits
            ):
                return splits

    raise ValueError(
        "Not enough independent households for class-complete CV."
    )


# ------------------------------------------------------------
# 7. CANDIDATE MODELS
# ------------------------------------------------------------

def pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="median", add_indicator=True
        )),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def candidate_models():
    candidates = {
        "majority_baseline": (
            DummyClassifier(strategy="most_frequent"),
            {}
        ),
        "logistic_regression": (
            LogisticRegression(
                max_iter=5000, random_state=SEED
            ),
            {
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"]
            }
        ),
        "random_forest": (
            RandomForestClassifier(
                n_jobs=1, random_state=SEED
            ),
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [
                    None, "balanced", "balanced_subsample"
                ]
            }
        ),
        "extra_trees": (
            ExtraTreesClassifier(
                n_jobs=1, random_state=SEED
            ),
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [None, "balanced"]
            }
        )
    }

    try:
        from lightgbm import LGBMClassifier

        candidates["lightgbm"] = (
            LGBMClassifier(
                random_state=SEED,
                n_jobs=1,
                verbosity=-1,
                subsample_freq=1
            ),
            {
                "model__n_estimators": [150, 300, 500],
                "model__learning_rate": [0.03, 0.07],
                "model__num_leaves": [7, 15, 31],
                "model__min_child_samples": [10, 25, 50],
                "model__class_weight": [None, "balanced"],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        print("LightGBM not installed; skipping.")

    try:
        from xgboost import XGBClassifier

        candidates["xgboost"] = (
            XGBClassifier(
                random_state=SEED,
                n_jobs=1,
                tree_method="hist",
                eval_metric="mlogloss"
            ),
            {
                "model__n_estimators": [150, 300, 500],
                "model__learning_rate": [0.03, 0.07],
                "model__max_depth": [2, 4, 6],
                "model__min_child_weight": [1, 5, 10],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        print("XGBoost not installed; skipping.")

    return candidates


# ------------------------------------------------------------
# 8. EVALUATION OUTPUTS
# ------------------------------------------------------------

def save_confusion(y_true, y_pred, class_names, task):
    indices = np.arange(len(class_names))
    matrix = confusion_matrix(
        y_true, y_pred, labels=indices
    )

    pd.DataFrame(
        matrix, index=class_names, columns=class_names
    ).to_csv(OUTPUT_DIR / f"{task}_confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set_xticks(indices)
    ax.set_yticks(indices)
    ax.set_xticklabels(
        class_names, rotation=45, ha="right"
    )
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(task)

    for i in indices:
        for j in indices:
            ax.text(
                j, i, str(matrix[i, j]),
                ha="center", va="center"
            )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{task}_confusion_matrix.png",
        dpi=200
    )
    plt.close(fig)


# ------------------------------------------------------------
# 9. MAIN
# ------------------------------------------------------------

def main():
    if FEATURE_PROFILE not in {
        "survey_enriched", "mobility_only"
    }:
        raise ValueError("Invalid FEATURE_PROFILE.")

    data = load_data()
    income_labels = resolve_household_income(data)
    people = build_people(data, income_labels)
    people = assign_split(people)

    targets = {
        "occupation": "label_occupation",
        "income_class": "label_income_class",
        "primary_mode": "label_primary_mode"
    }

    excluded = {
        "traveler_id", "household_id", "split",
        *targets.values()
    }

    features = [
        c for c in people.columns
        if c not in excluded and not c.startswith("audit_")
    ]

    if FEATURE_PROFILE == "mobility_only":
        features = [
            c for c in features
            if not c.startswith("purpose_")
            and not c.startswith("cost_")
        ]

    all_cv = []
    results = []
    failures = []
    coverage = []

    for task, target in targets.items():
        print("\n" + "=" * 70)
        print("TASK:", task)
        print("=" * 70)

        development = people.loc[
            people["split"].eq("development")
        ].dropna(subset=[target]).copy()

        test = people.loc[
            people["split"].eq("test")
        ].dropna(subset=[target]).copy()

        counts = (
            development.groupby(target)["household_id"]
            .nunique()
        )

        coverage.append({
            "task": task,
            "development_travelers": len(development),
            "test_travelers": len(test),
            "development_households": (
                development["household_id"].nunique()
            ),
            "test_households": test["household_id"].nunique()
        })

        print("Development label counts:")
        print(development[target].value_counts())
        print("Independent development households per class:")
        print(counts)
        print("Test travelers:", len(test))

        if (
            len(counts) < 2
            or counts.min() < MIN_DEVELOPMENT_GROUPS_PER_CLASS
            or test.empty
        ):
            message = (
                "Task skipped: insufficient independent labeled "
                "households. Review label audits or provide verified labels."
            )
            print(message)
            failures.append({
                "task": task,
                "model": "ALL",
                "error": message
            })
            continue

        if not set(test[target]).issubset(set(development[target])):
            failures.append({
                "task": task,
                "model": "ALL",
                "error": "Test contains a class absent from development."
            })
            print("Task skipped: unseen test class.")
            continue

        task_features = [
            c for c in features
            if development[c].notna().any()
        ]

        # [BUG] X_dev = development[task_features].astype(float)
        # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
        #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
        #       the model pipelines' SimpleImputer already fills NaN.
        X_dev = development[task_features].astype(object).where(development[task_features].notna(), np.nan).astype(float)
        # [BUG] X_test = test[task_features].astype(float)
        # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
        #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
        #       the model pipelines' SimpleImputer already fills NaN.
        X_test = test[task_features].astype(object).where(test[task_features].notna(), np.nan).astype(float)

        encoder = LabelEncoder()
        y_dev = encoder.fit_transform(development[target])
        y_test = encoder.transform(test[target])

        classes = np.arange(len(encoder.classes_))
        groups = development["household_id"].to_numpy()

        try:
            folds = cv_splits(X_dev, y_dev, groups)
        except ValueError as error:
            print(error)
            failures.append({
                "task": task, "model": "ALL", "error": str(error)
            })
            continue

        scorer = make_scorer(
            f1_score,
            labels=classes,
            average="macro",
            zero_division=0
        )

        trained = {}
        task_cv = []

        for name, (model, parameters) in candidate_models().items():
            print("\nTraining:", name)

            try:
                search = RandomizedSearchCV(
                    pipeline(model),
                    param_distributions=parameters,
                    n_iter=(
                        SEARCH_ITERATIONS if parameters else 1
                    ),
                    scoring=scorer,
                    cv=folds,
                    refit=True,
                    n_jobs=SEARCH_JOBS,
                    random_state=SEED,
                    error_score="raise"
                )

                search.fit(X_dev, y_dev)
                position = search.best_index_

                row = {
                    "task": task,
                    "model": name,
                    "cv_macro_f1": float(search.best_score_),
                    "cv_std": float(
                        search.cv_results_["std_test_score"][
                            position
                        ]
                    ),
                    "parameters": json.dumps(
                        search.best_params_, default=str
                    )
                }

                trained[name] = search.best_estimator_
                task_cv.append(row)
                all_cv.append(row)

                print(
                    f"CV macro F1: {row['cv_macro_f1']:.4f} "
                    f"+/- {row['cv_std']:.4f}"
                )

                pd.DataFrame(search.cv_results_).to_csv(
                    OUTPUT_DIR / f"{task}_{name}_search.csv",
                    index=False
                )

            except Exception as error:
                print("Failed:", error)
                failures.append({
                    "task": task,
                    "model": name,
                    "error": str(error)
                })

        if not task_cv:
            continue

        ranking = pd.DataFrame(task_cv).sort_values(
            "cv_macro_f1", ascending=False
        )
        winner = ranking.iloc[0]["model"]
        selected = trained[winner]

        predictions = selected.predict(X_test).astype(int)

        # Baseline label is learned from development labels,
        # not chosen from the test distribution.
        majority_class = int(
            np.bincount(y_dev).argmax()
        )
        baseline_predictions = np.full(
            len(y_test), majority_class
        )

        result = {
            "task": task,
            "selected_model": winner,
            "feature_profile": FEATURE_PROFILE,
            "development_n": len(development),
            "test_n": len(test),
            "cv_macro_f1": ranking.iloc[0]["cv_macro_f1"],
            "test_accuracy": accuracy_score(
                y_test, predictions
            ),
            "baseline_test_accuracy": accuracy_score(
                y_test, baseline_predictions
            ),
            "test_macro_f1": f1_score(
                y_test, predictions,
                labels=classes,
                average="macro",
                zero_division=0
            ),
            "baseline_test_macro_f1": f1_score(
                y_test, baseline_predictions,
                labels=classes,
                average="macro",
                zero_division=0
            ),
            "test_balanced_accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "test_mcc": matthews_corrcoef(
                y_test, predictions
            )
        }

        true_names = encoder.inverse_transform(y_test)
        predicted_names = encoder.inverse_transform(predictions)

        if task == "income_class":
            order = {
                "low_or_lower_middle": 0,
                "middle": 1,
                "high": 2
            }
            actual_order = np.array([
                order[x] for x in true_names
            ])
            predicted_order = np.array([
                order[x] for x in predicted_names
            ])

            result["income_ordinal_mae"] = float(
                np.abs(actual_order - predicted_order).mean()
            )
            result["income_weighted_kappa"] = (
                cohen_kappa_score(
                    actual_order,
                    predicted_order,
                    labels=[0, 1, 2],
                    weights="quadratic"
                )
            )

        results.append(result)

        print("\nSelected:", winner)
        print(classification_report(
            y_test, predictions,
            labels=classes,
            target_names=encoder.classes_,
            zero_division=0
        ))

        report = classification_report(
            y_test, predictions,
            labels=classes,
            target_names=encoder.classes_,
            zero_division=0,
            output_dict=True
        )
        pd.DataFrame(report).transpose().to_csv(
            OUTPUT_DIR / f"{task}_classification_report.csv"
        )

        save_confusion(
            y_test, predictions, encoder.classes_, task
        )

        prediction_table = test[
            ["traveler_id", "household_id"]
        ].copy()
        prediction_table["actual"] = true_names
        prediction_table["predicted"] = predicted_names
        prediction_table.to_csv(
            OUTPUT_DIR / f"{task}_test_predictions.csv",
            index=False
        )

        joblib.dump(
            {
                "pipeline": selected,
                "label_encoder": encoder,
                "features": task_features,
                "feature_profile": FEATURE_PROFILE,
                "target": target
            },
            OUTPUT_DIR / f"{task}_model.joblib"
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(
            ranking["model"],
            ranking["cv_macro_f1"],
            yerr=ranking["cv_std"],
            capsize=4
        )
        ax.set_ylabel("Cross-validation macro F1")
        ax.set_title(task)
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"{task}_cv_comparison.png",
            dpi=200
        )
        plt.close(fig)

        pd.DataFrame(all_cv).to_csv(
            OUTPUT_DIR / "all_model_cv_comparison.csv",
            index=False
        )
        pd.DataFrame(results).to_csv(
            OUTPUT_DIR / "selected_model_test_results.csv",
            index=False
        )

    pd.DataFrame(coverage).to_csv(
        OUTPUT_DIR / "label_coverage.csv", index=False
    )
    pd.DataFrame(
        failures, columns=["task", "model", "error"]
    ).to_csv(
        OUTPUT_DIR / "failures_and_skipped_tasks.csv",
        index=False
    )

    configuration = {
        "feature_profile": FEATURE_PROFILE,
        "seed": SEED,
        "search_iterations": SEARCH_ITERATIONS,
        "income_policy": (
            "Verified household file when provided; otherwise "
            "unanimous observed income category within household."
        ),
        "verified_income_file": (
            str(VERIFIED_INCOME_FILE)
            if VERIFIED_INCOME_FILE is not None else None
        ),
        "previous_split_file": str(PREVIOUS_SPLIT_FILE),
        "notes": [
            "Different exact amounts within the same category "
            "do not invalidate a household category.",
            "Conflicting categories are not resolved arbitrarily.",
            "Household income labels are not demographic input features.",
            "Primary mode is the unique most frequent observed trip mode.",
            "Survey purpose and cost are excluded in mobility_only mode.",
            "Repeated use of an inspected test set is exploratory.",
            "Synthetic-data performance does not establish real-data "
            "or telecom-transfer performance."
        ]
    }

    with open(
        OUTPUT_DIR / "configuration.json",
        "w",
        encoding="utf-8"
    ) as handle:
        json.dump(configuration, handle, indent=2)

    print("\n" + "=" * 70)
    print("FINAL SELECTED-MODEL RESULTS")
    print("=" * 70)

    if results:
        print(pd.DataFrame(results).to_string(index=False))
    else:
        print("No task had sufficient valid labels.")

    print("\nResults directory:", OUTPUT_DIR.resolve())
    print(
        "\nInspect household_income_audit.csv before "
        "interpreting income performance."
    )


if __name__ == "__main__":
    main()

Trips: 8645
Travelers: 3383
Households: 1165

HOUSEHOLD INCOME AUDIT
observed_status
consistent_category    1123
missing                  42
Name: count, dtype: int64
Income label source: observed_household_category_consistency
Resolved households: 1123

Person-level table: (3383, 62)
Occupation conflicts: 0
Mode ties: 250

Reusing previous household split.
Reminder: this test sample has already been inspected; results are exploratory, not fresh confirmation.

TASK: occupation
Development label counts:
label_occupation
employed         1353
student_child     602
homemaker         517
non_employed      157
other              60
Name: count, dtype: int64
Independent development households per class:
label_occupation
employed         868
homemaker        449
non_employed     147
other             56
student_child    450
Name: household_id, dtype: int64
Test travelers: 694

Training: majority_baseline
CV macro F1: 0.1339 +/- 0.0033

Training: logistic_regression
CV macro F1: 0.5329 +/- 0.0

In [ ]:
# ============================================================
# BENGALURU HTS: PERSON AND HOUSEHOLD MOBILITY INFERENCE
#
# Tasks:
#   Occupation   -> one row per traveler
#   Primary mode -> one row per traveler
#   Income class -> one row per household
#
# Required:
# pandas numpy scikit-learn matplotlib joblib
#
# Optional:
# lightgbm
#
# No installation is attempted automatically.
# ============================================================

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    f1_score,
    make_scorer,
    matthews_corrcoef
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INPUT_FILE = Path("output/bengaluru_hts_noisy_medium.csv")
OUTPUT_DIR = Path("output/person_household_corrected_results")

PREVIOUS_SPLIT_FILE = Path(
    "output/corrected_mobility_graph_results/split_assignments.csv"
)

# Optional authoritative label files.
#
# Income file:
# HH_ID, household_monthly_income_inr
#
# Occupation file:
# HH_ID, Person_ID, occupation
#
# When provided, ONLY matched verified labels are used
# for that target. No silent fallback to noisy labels.
VERIFIED_INCOME_FILE = None
VERIFIED_OCCUPATION_FILE = None

# Use "survey_enriched" for comparison with previous runs.
# Use "mobility_only" to exclude observed purpose and cost.
FEATURE_PROFILE = "survey_enriched"

SEED = 42
CV_FOLDS = 5
SEARCH_ITERATIONS = 10
SEARCH_JOBS = -1

# Keep False during further model development.
# The previous test set has already been inspected.
EVALUATE_EXISTING_TEST = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({
            "": pd.NA, "nan": pd.NA,
            "None": pd.NA, "<NA>": pd.NA
        })
    )


def traveler_key(households, persons):
    return (
        households.str.len().astype(str)
        + ":" + households + "|" + persons
    )


def normalize(value):
    if pd.isna(value):
        return ""
    return re.sub(
        r"[\s_-]+", " ", str(value).strip().lower()
    )


def occupation_class(value):
    value = normalize(value)

    if value in {"", "unknown", "missing", "none", "nan"}:
        return np.nan

    if any(s in value for s in [
        "unemploy", "non employ", "not employ", "retir"
    ]):
        return "non_employed"

    if any(s in value for s in [
        "student", "child", "pupil"
    ]):
        return "student_child"

    if any(s in value for s in [
        "homemaker", "home maker",
        "housewife", "househusband"
    ]):
        return "homemaker"

    if any(s in value for s in [
        "employ", "salar", "business",
        "worker", "professional"
    ]):
        return "employed"

    return "other"


def mode_class(value):
    value = normalize(value)
    # [FIX] With the Codebook mode names, "Ride-hail car" fell into private_car, "Ride-hail
    #       two-wheeler" into two_wheeler and "Cycle-rickshaw" into active transport (and, in the
    #       11th-13th cells, "Taxi" into other). They are all paid vehicles.
    if any(word in value for word in ['ride hail', 'rickshaw']):
        return "paid_private_vehicle"

    if value in {"", "unknown", "missing", "none", "nan"}:
        return np.nan

    if any(s in value for s in ["metro", "rail", "train"]):
        return "rail_based_public_transport"

    if any(s in value for s in [
        "auto", "cab", "taxi", "uber", "ola"
    ]):
        return "paid_private_vehicle"

    if "bus" in value or "bmtc" in value:
        return "bus"

    if any(s in value for s in [
        "two wheeler", "motorcycle", "motorbike", "scooter"
    ]):
        return "two_wheeler"

    if any(s in value for s in [
        "walk", "bicycle", "cycling", "cycle"
    ]):
        return "active_transport"

    if "car" in value:
        return "private_car"

    return "other"


def income_class(value):
    if pd.isna(value):
        return np.nan
    if value < 50000:
        return "low_or_lower_middle"
    if value < 100000:
        return "middle"
    return "high"


def numeric(series):
    return pd.to_numeric(
        series.astype("string").str.replace(
            ",", "", regex=False
        ),
        errors="coerce"
    )


def parse_time(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"(\d{1,2}):(\d{2})", str(value))
    if not match:
        return np.nan

    hour, minute = map(int, match.groups())

    if 0 <= hour < 24 and 0 <= minute < 60:
        return float(hour * 60 + minute)

    return np.nan


def single_label(series):
    values = series.dropna().unique()
    return values[0] if len(values) == 1 else np.nan


def entropy(series):
    p = series.dropna().value_counts(normalize=True)
    return float(-(p * np.log(p)).sum()) if len(p) else np.nan


def share(mask, valid):
    return float(mask[valid].mean()) if valid.any() else np.nan


# ------------------------------------------------------------
# 3. LOAD AND STANDARDIZE
# ------------------------------------------------------------

def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input CSV not found: {INPUT_FILE}"
        )

    data = pd.read_csv(INPUT_FILE, dtype="string")
    data.columns = data.columns.str.strip()

    aliases = {
        "household_id": ["HH_ID", "household_id"],
        "person_id": ["Person_ID", "person_id"],
        "origin": ["E2_Origin_PIN", "origin_zone"],
        "destination": ["E3_Dest_PIN", "destination_zone"],
        "distance": ["trip_distance_km", "E7_Distance_km"],
        "duration": ["trip_duration_min", "E6_Travel_time_min"],
        "cost": ["travel_cost_inr", "E10_Cost_INR"],
        "departure": ["departure_time_hhmm", "E5_Departure"],
        "purpose": ["trip_purpose", "E4_Purpose"],
        "raw_mode": ["mode", "E8_Primary_mode"],
        "raw_occupation": ["occupation"],
        "income": ["household_monthly_income_inr"]
    }

    mapping = {}

    for canonical, options in aliases.items():
        found = next(
            (name for name in options if name in data.columns),
            None
        )
        if found is not None:
            mapping[found] = canonical

    data = data.rename(columns=mapping)

    required = [
        "household_id", "person_id",
        "origin", "destination",
        "distance", "duration", "departure",
        "raw_mode", "raw_occupation", "income"
    ]

    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for optional in ["purpose", "cost"]:
        if optional not in data.columns:
            data[optional] = pd.NA

    for column in [
        "household_id", "person_id", "origin", "destination"
    ]:
        data[column] = clean_id(data[column])

    if data["household_id"].isna().any():
        raise ValueError("Missing household IDs must be resolved.")

    data = data.dropna(subset=["person_id"]).copy()

    data["traveler_id"] = traveler_key(
        data["household_id"], data["person_id"]
    )

    for column in ["distance", "duration", "cost", "income"]:
        data[column] = numeric(data[column])
        data.loc[data[column] < 0, column] = np.nan

    data["departure_min"] = data["departure"].map(parse_time)
    data["occupation_label"] = (
        data["raw_occupation"].map(occupation_class)
    )
    data["mode_label"] = data["raw_mode"].map(mode_class)
    data["income_category"] = data["income"].map(income_class)

    for original, label, filename in [
        ("raw_occupation", "occupation_label", "occupation_mapping.csv"),
        ("raw_mode", "mode_label", "mode_mapping.csv")
    ]:
        (
            data.groupby([original, label], dropna=False)
            .size().rename("trip_rows").reset_index()
            .to_csv(OUTPUT_DIR / filename, index=False)
        )

    print("Trips:", len(data))
    print("Travelers:", data["traveler_id"].nunique())
    print("Households:", data["household_id"].nunique())

    return data


# ------------------------------------------------------------
# 4. LABEL AUDITS AND AUTHORITATIVE OVERRIDES
# ------------------------------------------------------------

def resolve_labels(data):
    occupation_audit = data.groupby("traveler_id").agg(
        household_id=("household_id", "first"),
        person_id=("person_id", "first"),
        known_rows=("occupation_label", "count"),
        distinct_labels=("occupation_label", "nunique")
    )

    occupation = data.groupby(
        "traveler_id"
    )["occupation_label"].agg(single_label)

    occupation_audit["observed_label"] = occupation
    occupation_audit["status"] = np.select(
        [
            occupation_audit["distinct_labels"].eq(0),
            occupation_audit["distinct_labels"].eq(1)
        ],
        ["missing", "consistent"],
        default="conflicting"
    )

    income_audit = data.groupby("household_id").agg(
        known_income_rows=("income", "count"),
        distinct_amounts=("income", "nunique"),
        distinct_categories=("income_category", "nunique")
    )

    income = data.groupby(
        "household_id"
    )["income_category"].agg(single_label)

    income_audit["observed_label"] = income
    income_audit["status"] = np.select(
        [
            income_audit["distinct_categories"].eq(0),
            income_audit["distinct_categories"].eq(1)
        ],
        ["missing", "consistent_category"],
        default="conflicting_categories"
    )

    if VERIFIED_OCCUPATION_FILE is not None:
        verified = pd.read_csv(
            VERIFIED_OCCUPATION_FILE, dtype="string"
        )
        required = {"HH_ID", "Person_ID", "occupation"}

        if not required.issubset(verified.columns):
            raise ValueError(
                "Verified occupation file needs "
                "HH_ID, Person_ID, occupation."
            )

        hh = clean_id(verified["HH_ID"])
        person = clean_id(verified["Person_ID"])

        if hh.isna().any() or person.isna().any():
            raise ValueError("Missing IDs in verified occupation file.")

        verified["traveler_id"] = traveler_key(hh, person)
        verified["label"] = verified["occupation"].map(
            occupation_class
        )

        if verified["traveler_id"].duplicated().any():
            raise ValueError(
                "Verified occupation file must have one row per person."
            )

        lookup = verified.set_index("traveler_id")["label"]
        occupation = pd.Series(
            occupation_audit.index.map(lookup),
            index=occupation_audit.index
        )

    if VERIFIED_INCOME_FILE is not None:
        verified = pd.read_csv(
            VERIFIED_INCOME_FILE, dtype="string"
        )
        required = {"HH_ID", "household_monthly_income_inr"}

        if not required.issubset(verified.columns):
            raise ValueError(
                "Verified income file needs HH_ID and "
                "household_monthly_income_inr."
            )

        verified["household_id"] = clean_id(verified["HH_ID"])
        amounts = numeric(
            verified["household_monthly_income_inr"]
        )

        if (
            verified["household_id"].isna().any()
            or amounts.isna().any()
            or amounts.lt(0).any()
            or verified["household_id"].duplicated().any()
        ):
            raise ValueError(
                "Verified income file has invalid amounts, "
                "missing IDs, or duplicated households."
            )

        verified["label"] = amounts.map(income_class)
        lookup = verified.set_index("household_id")["label"]

        income = pd.Series(
            income_audit.index.map(lookup),
            index=income_audit.index
        )

    occupation_audit["resolved_label"] = occupation
    income_audit["resolved_label"] = income

    occupation_audit.to_csv(
        OUTPUT_DIR / "occupation_label_audit.csv"
    )
    income_audit.to_csv(
        OUTPUT_DIR / "household_income_audit.csv"
    )

    conflict_people = occupation_audit.index[
        occupation_audit["status"].eq("conflicting")
    ]
    conflict_households = income_audit.index[
        income_audit["status"].eq("conflicting_categories")
    ]

    data.loc[
        data["traveler_id"].isin(conflict_people),
        [
            "household_id", "person_id",
            "raw_occupation", "occupation_label"
        ]
    ].to_csv(
        OUTPUT_DIR / "occupation_conflict_rows.csv", index=False
    )

    data.loc[
        data["household_id"].isin(conflict_households),
        [
            "household_id", "person_id",
            "income", "income_category"
        ]
    ].to_csv(
        OUTPUT_DIR / "income_conflict_rows.csv", index=False
    )

    print("\nOccupation label audit:")
    print(occupation_audit["status"].value_counts())
    print("\nHousehold income audit:")
    print(income_audit["status"].value_counts())

    return occupation, income


# ------------------------------------------------------------
# 5. MOBILITY FEATURES
# ------------------------------------------------------------

def numeric_summary(record, name, values, n):
    # [BUG] valid = values.dropna()
    # [FIX] Now that trip underreporting really drops trips, some people have only one trip.
    #       These columns are pandas nullable numbers, whose std() of one value is <NA>, and
    #       float(<NA>) raises TypeError. Use plain floats (std of one value -> NaN).
    valid = values.dropna().astype(float)

    for statistic in ["mean", "median", "min", "max", "std"]:
        record[f"{name}_{statistic}"] = (
            float(getattr(valid, statistic)())
            if len(valid) else np.nan
        )

    record[f"{name}_missing_share"] = 1 - len(valid) / n


def mobility_features(trips):
    n = len(trips)
    result = {"trip_count": n}

    valid = trips.dropna(subset=["origin", "destination"])
    nodes = pd.concat([
        valid["origin"], valid["destination"]
    ]).unique()

    edges = valid.groupby(["origin", "destination"]).size()
    nonself = valid[valid["origin"] != valid["destination"]]
    directed = set(
        nonself[["origin", "destination"]].itertuples(
            index=False, name=None
        )
    )

    result["node_count"] = len(nodes)
    result["edge_count"] = len(edges)
    result["valid_edge_share"] = len(valid) / n

    result["density"] = (
        len(directed) / (len(nodes) * (len(nodes) - 1))
        if len(nodes) > 1 else 0.0
    )
    result["reciprocity"] = (
        sum((b, a) in directed for a, b in directed)
        / len(directed)
        if directed else 0.0
    )
    result["dominant_edge_share"] = (
        float(edges.max() / edges.sum())
        if len(edges) else np.nan
    )
    result["destination_entropy"] = entropy(
        trips["destination"]
    )
    result["origin_entropy"] = entropy(trips["origin"])

    distance = trips["distance"]
    duration = trips["duration"]
    time = trips["departure_min"]

    numeric_summary(result, "distance", distance, n)
    numeric_summary(result, "duration", duration, n)

    valid_speed = (
        distance.notna() & duration.notna() & duration.gt(0)
    )
    speed = (
        distance[valid_speed]
        / (duration[valid_speed] / 60)
    )
    numeric_summary(result, "speed", speed, n)

    for name, mask in {
        "short_share": distance.lt(3),
        "medium_share": distance.ge(3) & distance.lt(10),
        "long_share": distance.ge(10)
    }.items():
        result[name] = share(mask, distance.notna())

    for name, start, end in [
        ("morning_share", 360, 720),
        ("afternoon_share", 720, 1020),
        ("evening_share", 1020, 1200),
        ("morning_peak_share", 480, 600),
        ("evening_peak_share", 1020, 1140)
    ]:
        result[name] = share(
            time.ge(start) & time.lt(end), time.notna()
        )

    angles = 2 * np.pi * time.dropna() / 1440
    result["time_sin_mean"] = (
        float(np.sin(angles).mean()) if len(angles) else np.nan
    )
    result["time_cos_mean"] = (
        float(np.cos(angles).mean()) if len(angles) else np.nan
    )

    if FEATURE_PROFILE == "survey_enriched":
        cost = trips["cost"]
        numeric_summary(result, "cost", cost, n)

        valid_ratio = (
            cost.notna() & distance.notna() & distance.gt(0)
        )
        result["cost_per_km"] = (
            float(
                cost[valid_ratio].sum()
                / distance[valid_ratio].sum()
            )
            if valid_ratio.any() else np.nan
        )

        purpose = trips["purpose"].map(normalize)
        known = purpose.ne("")
        result["purpose_entropy"] = entropy(
            purpose.where(known)
        )

        for name, pattern in {
            "work": r"\bwork\b|office|business",
            "education": r"education|school|college|university",
            "shopping": r"shop"
        }.items():
            result[f"purpose_{name}_share"] = share(
                purpose.str.contains(pattern, regex=True),
                known
            )

    return result


def create_tables(data, occupation, income):
    records = []

    for traveler, trips in data.groupby(
        "traveler_id", sort=False
    ):
        record = mobility_features(trips)
        record["record_id"] = traveler
        record["household_id"] = trips["household_id"].iloc[0]
        record["label_occupation"] = occupation.get(
            traveler, np.nan
        )

        counts = trips["mode_label"].dropna().value_counts()
        winners = (
            counts[counts == counts.max()].index
            if len(counts) else []
        )
        record["label_primary_mode"] = (
            winners[0] if len(winners) == 1 else np.nan
        )
        records.append(record)

    people = pd.DataFrame(records).replace(
        [np.inf, -np.inf], np.nan
    )

    excluded = {
        "record_id", "household_id",
        "label_occupation", "label_primary_mode"
    }
    person_features = [
        c for c in people.columns if c not in excluded
    ]

    # Aggregate person features, not raw zone identities.
    # Household counts refer to observed survey travelers only.
    households = people.groupby("household_id")[
        person_features
    ].agg(["mean", "max", "std"])

    households.columns = [
        f"person_{feature}_{aggregation}"
        for feature, aggregation in households.columns
    ]

    households["observed_traveler_count"] = (
        people.groupby("household_id").size()
    )
    households["observed_trip_count"] = (
        data.groupby("household_id").size()
    )

    households = households.reset_index()
    households["record_id"] = households["household_id"]
    households["label_income_class"] = (
        households["household_id"].map(income)
    )

    households = households.replace(
        [np.inf, -np.inf], np.nan
    )

    people.to_csv(
        OUTPUT_DIR / "person_level_table.csv", index=False
    )
    households.to_csv(
        OUTPUT_DIR / "household_level_table.csv", index=False
    )

    return people, households


# ------------------------------------------------------------
# 6. PRESERVE HOUSEHOLD ASSIGNMENTS
# ------------------------------------------------------------

def household_split(household_ids):
    ids = pd.Index(household_ids.unique())

    if PREVIOUS_SPLIT_FILE.exists():
        previous = pd.read_csv(
            PREVIOUS_SPLIT_FILE, dtype="string"
        )

        if not {"split_group", "split"}.issubset(previous.columns):
            raise ValueError(
                "Previous split file needs split_group and split."
            )

        previous["split_group"] = clean_id(
            previous["split_group"]
        )

        if (
            previous.groupby("split_group")["split"]
            .nunique().gt(1).any()
        ):
            raise ValueError(
                "A household appears in multiple previous partitions."
            )

        lookup = (
            previous.drop_duplicates("split_group")
            .set_index("split_group")["split"]
        )
        assignment = pd.Series(ids.map(lookup), index=ids)

        if assignment.isna().any():
            raise ValueError(
                "Some households are absent from the previous split."
            )

        print("\nReusing previous household split.")

    else:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=SEED
        )
        _, test_idx = next(
            splitter.split(ids.to_numpy(), groups=ids.to_numpy())
        )
        assignment = pd.Series("development", index=ids)
        assignment.iloc[test_idx] = "test"
        print("\nCreated household-disjoint development/test split.")

    if not set(assignment.unique()).issubset(
        {"development", "test"}
    ):
        raise ValueError("Unexpected split labels.")

    assignment.rename("split").to_csv(
        OUTPUT_DIR / "household_split.csv",
        index_label="household_id"
    )
    return assignment


def make_folds(X, y, groups):
    class_group_counts = (
        pd.DataFrame({"label": y, "group": groups})
        .groupby("label")["group"].nunique()
    )

    maximum = min(CV_FOLDS, int(class_group_counts.min()))
    expected = set(np.unique(y))

    for fold_count in range(maximum, 1, -1):
        for attempt in range(10):
            cv = StratifiedGroupKFold(
                n_splits=fold_count,
                shuffle=True,
                random_state=SEED + attempt
            )
            folds = list(cv.split(X, y, groups))

            if all(
                set(y[train]) == expected
                and set(y[valid]) == expected
                for train, valid in folds
            ):
                return folds

    raise ValueError(
        "Insufficient independent labeled households for CV."
    )


# ------------------------------------------------------------
# 7. MODELS AND SEARCH SPACES
# ------------------------------------------------------------

def model_candidates():
    candidates = {
        "majority_baseline": (
            DummyClassifier(strategy="most_frequent"),
            {}
        ),
        "logistic_regression": (
            LogisticRegression(
                max_iter=5000, random_state=SEED
            ),
            {
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"]
            }
        ),
        "random_forest": (
            RandomForestClassifier(
                n_jobs=1, random_state=SEED
            ),
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [
                    None, "balanced", "balanced_subsample"
                ]
            }
        ),
        "extra_trees": (
            ExtraTreesClassifier(
                n_jobs=1, random_state=SEED
            ),
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [8, 16, None],
                "model__min_samples_leaf": [1, 2, 5, 10],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [None, "balanced"]
            }
        )
    }

    try:
        from lightgbm import LGBMClassifier

        candidates["lightgbm"] = (
            LGBMClassifier(
                n_jobs=1,
                random_state=SEED,
                verbosity=-1,
                subsample_freq=1
            ),
            {
                "model__n_estimators": [150, 300, 500],
                "model__learning_rate": [0.03, 0.07],
                "model__num_leaves": [7, 15, 31],
                "model__min_child_samples": [10, 25, 50],
                "model__class_weight": [None, "balanced"],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        print("Optional LightGBM not installed; skipping.")

    return candidates


def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="median", add_indicator=True
        )),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


# ------------------------------------------------------------
# 8. OPTIONAL TEST REPORT
# ------------------------------------------------------------

def test_report(model, X, y, encoder, task, y_development):
    prediction = model.predict(X).astype(int)
    labels = np.arange(len(encoder.classes_))

    majority = int(np.bincount(y_development).argmax())
    baseline = np.full(len(y), majority)

    result = {
        "task": task,
        "test_n": len(y),
        "test_accuracy": accuracy_score(y, prediction),
        "baseline_accuracy": accuracy_score(y, baseline),
        "test_macro_f1": f1_score(
            y, prediction,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "test_balanced_accuracy": balanced_accuracy_score(
            y, prediction
        ),
        "test_mcc": matthews_corrcoef(y, prediction)
    }

    report = classification_report(
        y, prediction,
        labels=labels,
        target_names=encoder.classes_,
        zero_division=0,
        output_dict=True
    )
    pd.DataFrame(report).transpose().to_csv(
        OUTPUT_DIR / f"{task}_test_report.csv"
    )

    matrix = confusion_matrix(y, prediction, labels=labels)
    pd.DataFrame(
        matrix,
        index=encoder.classes_,
        columns=encoder.classes_
    ).to_csv(
        OUTPUT_DIR / f"{task}_test_confusion_matrix.csv"
    )

    if task == "income_household":
        order = {
            "low_or_lower_middle": 0,
            "middle": 1,
            "high": 2
        }
        actual = np.array([
            order[v] for v in encoder.inverse_transform(y)
        ])
        predicted = np.array([
            order[v]
            for v in encoder.inverse_transform(prediction)
        ])
        result["ordinal_mae"] = float(
            np.abs(actual - predicted).mean()
        )
        result["weighted_kappa"] = cohen_kappa_score(
            actual, predicted,
            labels=[0, 1, 2],
            weights="quadratic"
        )

    return result


# ------------------------------------------------------------
# 9. TRAINING WORKFLOW
# ------------------------------------------------------------

def main():
    if FEATURE_PROFILE not in {
        "survey_enriched", "mobility_only"
    }:
        raise ValueError("Invalid FEATURE_PROFILE.")

    data = load_data()
    occupation, income = resolve_labels(data)
    people, households = create_tables(
        data, occupation, income
    )
    assignment = household_split(data["household_id"])

    tasks = {
        "occupation_person": (
            people, "label_occupation"
        ),
        "primary_mode_person": (
            people, "label_primary_mode"
        ),
        "income_household": (
            households, "label_income_class"
        )
    }

    comparisons = []
    selected_summary = []
    test_results = []
    failures = []

    for task, (table, target) in tasks.items():
        print("\n" + "=" * 70)
        print("TASK:", task)
        print("=" * 70)

        table = table.copy()
        table["split"] = table["household_id"].map(assignment)

        development = table.loc[
            table["split"].eq("development")
        ].dropna(subset=[target]).copy()

        test = table.loc[
            table["split"].eq("test")
        ].dropna(subset=[target]).copy()

        print("Development label counts:")
        print(development[target].value_counts())

        group_counts = (
            development.groupby(target)["household_id"]
            .nunique()
        )
        print("Independent households per class:")
        print(group_counts)

        if len(group_counts) < 2 or group_counts.min() < 3:
            failures.append({
                "task": task,
                "model": "ALL",
                "error": "Insufficient independent labeled households."
            })
            print("Skipped: insufficient independent labels.")
            continue

        features = [
            c for c in table.columns
            if c not in {
                "record_id", "household_id", "split",
                "label_occupation",
                "label_primary_mode",
                "label_income_class"
            }
            and development[c].notna().any()
        ]

        # [BUG] X = development[features].astype(float)
        # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
        #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
        #       the model pipelines' SimpleImputer already fills NaN.
        X = development[features].astype(object).where(development[features].notna(), np.nan).astype(float)
        encoder = LabelEncoder()
        y = encoder.fit_transform(development[target])
        groups = development["household_id"].to_numpy()

        try:
            folds = make_folds(X, y, groups)
        except ValueError as error:
            failures.append({
                "task": task, "model": "ALL", "error": str(error)
            })
            print(error)
            continue

        labels = np.arange(len(encoder.classes_))

        scoring = {
            "macro_f1": make_scorer(
                f1_score,
                labels=labels,
                average="macro",
                zero_division=0
            ),
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        }

        fitted = {}
        task_rows = []

        for name, (model, parameters) in model_candidates().items():
            print("\nTraining:", name)

            try:
                search = RandomizedSearchCV(
                    make_pipeline(model),
                    param_distributions=parameters,
                    n_iter=SEARCH_ITERATIONS if parameters else 1,
                    scoring=scoring,
                    refit="macro_f1",
                    cv=folds,
                    n_jobs=SEARCH_JOBS,
                    random_state=SEED,
                    error_score="raise"
                )

                search.fit(X, y)
                position = search.best_index_
                scores = search.cv_results_

                row = {
                    "task": task,
                    "model": name,
                    "development_n": len(development),
                    "development_households": (
                        development["household_id"].nunique()
                    ),
                    "cv_macro_f1": float(
                        scores["mean_test_macro_f1"][position]
                    ),
                    "cv_macro_f1_std": float(
                        scores["std_test_macro_f1"][position]
                    ),
                    "cv_accuracy": float(
                        scores["mean_test_accuracy"][position]
                    ),
                    "cv_balanced_accuracy": float(
                        scores["mean_test_balanced_accuracy"][position]
                    ),
                    "parameters": json.dumps(
                        search.best_params_, default=str
                    )
                }

                comparisons.append(row)
                task_rows.append(row)
                fitted[name] = search.best_estimator_

                print(
                    f"CV macro F1: {row['cv_macro_f1']:.4f} "
                    f"+/- {row['cv_macro_f1_std']:.4f}"
                )

                pd.DataFrame(scores).to_csv(
                    OUTPUT_DIR / f"{task}_{name}_search.csv",
                    index=False
                )

            except Exception as error:
                print("Model failed:", error)
                failures.append({
                    "task": task,
                    "model": name,
                    "error": str(error)
                })

        if not task_rows:
            continue

        ranking = pd.DataFrame(task_rows).sort_values(
            "cv_macro_f1", ascending=False
        )
        winner = ranking.iloc[0]["model"]
        selected = fitted[winner]

        summary = ranking.iloc[0].to_dict()
        summary["feature_profile"] = FEATURE_PROFILE
        summary["feature_count"] = len(features)
        selected_summary.append(summary)

        print("\nSelected using development CV:", winner)

        joblib.dump(
            {
                "pipeline": selected,
                "label_encoder": encoder,
                "features": features,
                "target": target,
                "task": task,
                "feature_profile": FEATURE_PROFILE,
                "training_partition": "development"
            },
            OUTPUT_DIR / f"{task}_selected_model.joblib"
        )

        with open(
            OUTPUT_DIR / f"{task}_features.json",
            "w", encoding="utf-8"
        ) as handle:
            json.dump(features, handle, indent=2)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(
            ranking["model"],
            ranking["cv_macro_f1"],
            yerr=ranking["cv_macro_f1_std"],
            capsize=4
        )
        ax.set_title(task)
        ax.set_ylabel("Development CV macro F1")
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"{task}_cv_comparison.png",
            dpi=200
        )
        plt.close(fig)

        if EVALUATE_EXISTING_TEST:
            if test.empty:
                print("No labeled test records; test skipped.")
            elif not set(test[target]).issubset(
                set(encoder.classes_)
            ):
                print("Unseen test class; test skipped.")
            else:
                warnings.warn(
                    "Evaluating an already inspected test set. "
                    "Treat these results as exploratory."
                )
                # [BUG] X_test = test[features].astype(float)
                # [FIX] Now that trip underreporting drops trips, some features are missing (<NA>) for people
                #       left with one trip, and astype(float) cannot convert <NA>. Turn it into NaN first;
                #       the model pipelines' SimpleImputer already fills NaN.
                X_test = test[features].astype(object).where(test[features].notna(), np.nan).astype(float)
                y_test = encoder.transform(test[target])
                report = test_report(
                    selected, X_test, y_test,
                    encoder, task, y
                )
                test_results.append(report)
        else:
            print(
                "Existing test set not evaluated: "
                "development-only model selection."
            )

        pd.DataFrame(comparisons).to_csv(
            OUTPUT_DIR / "all_model_cv_comparison.csv",
            index=False
        )
        pd.DataFrame(selected_summary).to_csv(
            OUTPUT_DIR / "selected_models_cv_summary.csv",
            index=False
        )

    pd.DataFrame(
        failures, columns=["task", "model", "error"]
    ).to_csv(
        OUTPUT_DIR / "failures_and_skipped_tasks.csv",
        index=False
    )

    if test_results:
        pd.DataFrame(test_results).to_csv(
            OUTPUT_DIR / "exploratory_test_results.csv",
            index=False
        )

    config = {
        "input_file": str(INPUT_FILE),
        "feature_profile": FEATURE_PROFILE,
        "income_unit": "household",
        "occupation_unit": "person",
        "primary_mode_unit": "person",
        "evaluate_existing_test": EVALUATE_EXISTING_TEST,
        "search_iterations": SEARCH_ITERATIONS,
        "seed": SEED,
        "notes": [
            "Conflicting labels are not repaired arbitrarily.",
            "Verified label files override observed labels.",
            "Income aggregation uses observed survey travelers, "
            "not necessarily every household member.",
            "Mode ties are excluded from primary-mode modelling.",
            "CV scores are used for model selection and are not "
            "an independent final generalization estimate.",
            "Household income results are not directly comparable "
            "with previous traveler-level income results.",
            "Telecom transfer requires matching feature definitions "
            "and matching units of observation.",
            "Synthetic-data results require real-data validation."
        ]
    }

    with open(
        OUTPUT_DIR / "configuration.json",
        "w", encoding="utf-8"
    ) as handle:
        json.dump(config, handle, indent=2)

    print("\n" + "=" * 70)
    print("SELECTED MODELS: DEVELOPMENT CROSS-VALIDATION")
    print("=" * 70)

    if selected_summary:
        columns = [
            "task", "model", "development_n",
            "development_households",
            "cv_macro_f1", "cv_macro_f1_std",
            "cv_accuracy", "cv_balanced_accuracy"
        ]
        print(
            pd.DataFrame(selected_summary)[columns]
            .to_string(index=False)
        )

    print("\nCompleted.")
    print("Results directory:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()

Trips: 8645
Travelers: 3383
Households: 1165

Occupation label audit:
status
consistent    3383
Name: count, dtype: int64

Household income audit:
status
consistent_category    1123
missing                  42
Name: count, dtype: int64

Reusing previous household split.

TASK: occupation_person
Development label counts:
label_occupation
employed         1353
student_child     602
homemaker         517
non_employed      157
other              60
Name: count, dtype: int64
Independent households per class:
label_occupation
employed         868
homemaker        449
non_employed     147
other             56
student_child    450
Name: household_id, dtype: int64

Training: majority_baseline
CV macro F1: 0.1339 +/- 0.0033

Training: logistic_regression
CV macro F1: 0.5294 +/- 0.0129

Training: random_forest
CV macro F1: 0.5278 +/- 0.0195

Training: extra_trees
CV macro F1: 0.5495 +/- 0.0115

Training: lightgbm
CV macro F1: 0.5398 +/- 0.0169

Selected using development CV: extra_trees
Existing 

In [ ]:
# ============================================================
# NEXT-STAGE MOBILITY INFERENCE IMPROVEMENT EXPERIMENT
#
# Reads the corrected tables from the PREVIOUS successful run.
#
# Improvements:
# - Balanced Random Forest candidate, if installed
# - Regularized Random Forest / Extra Trees
# - Nonlinear RBF-SVM
# - Optional LightGBM
# - Nested household-aware cross-validation
# - Outer-fold metrics and diagnostic out-of-fold predictions
# - Previously inspected test households remain unused
#
# Required:
# pandas numpy scikit-learn matplotlib joblib
#
# Optional:
# imbalanced-learn lightgbm
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedGroupKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SOURCE_DIR = Path(
    "output/person_household_corrected_results"
)

OUTPUT_DIR = Path(
    "output/nested_balanced_mobility_experiment"
)

SEED = 42

# Nested CV is more expensive than the earlier experiment.
OUTER_FOLDS = 3
INNER_FOLDS = 3

# Each model family gets its own search budget.
# Increase only after checking that the experiment runs correctly.
SEARCH_ITERATIONS_PER_MODEL = 5

SEARCH_JOBS = -1

# Run all tasks, or remove names to focus on one task.
TASKS_TO_RUN = [
    "occupation_person",
    "primary_mode_person",
    "income_household"
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. HOUSEHOLD-AWARE CV SPLITS
# ------------------------------------------------------------

def make_group_folds(X, y, groups, requested, seed):
    group_counts = (
        pd.DataFrame({"label": y, "group": groups})
        .groupby("label")["group"]
        .nunique()
    )

    maximum = min(requested, int(group_counts.min()))
    expected_classes = set(np.unique(y))

    for fold_count in range(maximum, 1, -1):
        for attempt in range(15):
            splitter = StratifiedGroupKFold(
                n_splits=fold_count,
                shuffle=True,
                random_state=seed + attempt
            )

            splits = list(splitter.split(X, y, groups))

            valid = all(
                set(y[train]) == expected_classes
                and set(y[valid]) == expected_classes
                for train, valid in splits
            )

            if valid:
                for train, valid in splits:
                    assert set(groups[train]).isdisjoint(
                        set(groups[valid])
                    )
                return splits

    raise ValueError(
        "Cannot create class-complete household-aware folds. "
        "Check minority-class household counts."
    )


# ------------------------------------------------------------
# 3. CANDIDATES
# ------------------------------------------------------------

def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def candidate_models():
    candidates = {
        "random_forest": (
            RandomForestClassifier(
                n_jobs=1,
                random_state=SEED
            ),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [6, 12, None],
                "model__min_samples_leaf": [1, 3, 8, 15],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [
                    None,
                    "balanced",
                    "balanced_subsample"
                ]
            }
        ),

        "extra_trees": (
            ExtraTreesClassifier(
                n_jobs=1,
                random_state=SEED
            ),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [6, 12, None],
                "model__min_samples_leaf": [1, 3, 8, 15],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [
                    None, "balanced"
                ]
            }
        ),

        "rbf_svm": (
            SVC(
                kernel="rbf",
                probability=False,
                cache_size=512
            ),
            {
                "model__C": [0.1, 1.0, 10.0, 50.0],
                "model__gamma": [
                    "scale", 0.001, 0.01, 0.1
                ],
                "model__class_weight": [
                    None, "balanced"
                ]
            }
        )
    }

    try:
        from imblearn.ensemble import (
            BalancedRandomForestClassifier
        )

        candidates["balanced_random_forest"] = (
            BalancedRandomForestClassifier(
                n_jobs=1,
                random_state=SEED,
                sampling_strategy="all",
                replacement=True,
                bootstrap=False
            ),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [4, 8, 16, None],
                "model__min_samples_leaf": [1, 3, 8],
                "model__max_features": ["sqrt", 0.7]
            }
        )
    except ImportError:
        warnings.warn(
            "imbalanced-learn is not installed. "
            "Balanced Random Forest will be skipped."
        )

    try:
        from lightgbm import LGBMClassifier

        candidates["lightgbm"] = (
            LGBMClassifier(
                n_jobs=1,
                random_state=SEED,
                verbosity=-1,
                subsample_freq=1
            ),
            {
                "model__n_estimators": [150, 300],
                "model__learning_rate": [0.03, 0.07],
                "model__num_leaves": [7, 15, 31],
                "model__min_child_samples": [15, 30, 60],
                "model__reg_lambda": [0.0, 1.0, 5.0],
                "model__class_weight": [None, "balanced"],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]
            }
        )
    except ImportError:
        warnings.warn(
            "LightGBM is not installed; skipping."
        )

    return candidates


# ------------------------------------------------------------
# 4. INNER MODEL SELECTION
# ------------------------------------------------------------

def select_model(X, y, folds, labels, task, stage):
    scorer = make_scorer(
        f1_score,
        labels=labels,
        average="macro",
        zero_division=0
    )

    candidates = candidate_models()

    # Always retain a simple fallback / benchmark.
    candidates["majority_baseline"] = (
        DummyClassifier(strategy="most_frequent"),
        {}
    )

    comparison = []
    successful = {}

    for name, (estimator, parameters) in candidates.items():
        print(f"  Searching: {name}")

        try:
            search = RandomizedSearchCV(
                estimator=make_pipeline(estimator),
                param_distributions=parameters,
                n_iter=(
                    SEARCH_ITERATIONS_PER_MODEL
                    if parameters else 1
                ),
                scoring=scorer,
                cv=folds,
                refit=True,
                n_jobs=SEARCH_JOBS,
                random_state=SEED,
                error_score="raise"
            )

            search.fit(X, y)

            row = {
                "task": task,
                "stage": stage,
                "model": name,
                "inner_cv_macro_f1": float(
                    search.best_score_
                ),
                "parameters": json.dumps(
                    search.best_params_, default=str
                ),
                "error": ""
            }

            successful[name] = search.best_estimator_
            comparison.append(row)

            print(
                "    Inner CV macro F1:",
                round(search.best_score_, 4)
            )

        except Exception as error:
            print(f"    Failed: {error}")

            comparison.append({
                "task": task,
                "stage": stage,
                "model": name,
                "inner_cv_macro_f1": np.nan,
                "parameters": "",
                "error": str(error)
            })

    valid = pd.DataFrame(comparison).dropna(
        subset=["inner_cv_macro_f1"]
    )

    if valid.empty:
        raise RuntimeError(
            f"No model succeeded for {task}, {stage}."
        )

    winner = valid.sort_values(
        "inner_cv_macro_f1",
        ascending=False
    ).iloc[0]["model"]

    return winner, successful[winner], comparison


# ------------------------------------------------------------
# 5. METRICS
# ------------------------------------------------------------

def metrics(y_true, prediction, labels):
    return {
        "accuracy": accuracy_score(
            y_true, prediction
        ),
        "macro_f1": f1_score(
            y_true,
            prediction,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, prediction
        ),
        "mcc": matthews_corrcoef(
            y_true, prediction
        )
    }


def save_diagnostics(
    task, development, y, predictions, encoder
):
    labels = np.arange(len(encoder.classes_))

    report = classification_report(
        y,
        predictions,
        labels=labels,
        target_names=encoder.classes_,
        zero_division=0,
        output_dict=True
    )

    # This pooled report is diagnostic.
    # Mean outer-fold scores are the primary evaluation output.
    pd.DataFrame(report).transpose().to_csv(
        OUTPUT_DIR / f"{task}_pooled_oof_report.csv"
    )

    matrix = confusion_matrix(
        y, predictions, labels=labels
    )

    pd.DataFrame(
        matrix,
        index=encoder.classes_,
        columns=encoder.classes_
    ).to_csv(
        OUTPUT_DIR / f"{task}_oof_confusion_matrix.csv"
    )

    diagnostic = development[
        ["record_id", "household_id"]
    ].copy()

    diagnostic["actual"] = encoder.inverse_transform(y)
    diagnostic["predicted"] = (
        encoder.inverse_transform(predictions)
    )
    diagnostic["correct"] = (
        diagnostic["actual"]
        == diagnostic["predicted"]
    )

    diagnostic.to_csv(
        OUTPUT_DIR / f"{task}_oof_predictions.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set_xticks(labels)
    ax.set_yticks(labels)
    ax.set_xticklabels(
        encoder.classes_,
        rotation=45,
        ha="right"
    )
    ax.set_yticklabels(encoder.classes_)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{task}: nested-CV predictions")

    for i in labels:
        for j in labels:
            ax.text(
                j, i, str(matrix[i, j]),
                ha="center", va="center"
            )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{task}_oof_confusion_matrix.png",
        dpi=200
    )
    plt.close(fig)


# ------------------------------------------------------------
# 6. LOAD PREVIOUS CORRECTED OUTPUTS
# ------------------------------------------------------------

def load_inputs():
    files = {
        "people": SOURCE_DIR / "person_level_table.csv",
        "households": SOURCE_DIR / "household_level_table.csv",
        "split": SOURCE_DIR / "household_split.csv"
    }

    missing = [
        str(path)
        for path in files.values()
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Run the previous corrected pipeline first. "
            f"Missing files: {missing}"
        )

    people = pd.read_csv(
        files["people"],
        dtype={
            "record_id": "string",
            "household_id": "string"
        }
    )

    households = pd.read_csv(
        files["households"],
        dtype={
            "record_id": "string",
            "household_id": "string"
        }
    )

    split = pd.read_csv(
        files["split"],
        dtype={
            "household_id": "string",
            "split": "string"
        }
    )

    if split["household_id"].duplicated().any():
        raise ValueError(
            "household_split.csv must contain one row "
            "per household."
        )

    assignment = split.set_index(
        "household_id"
    )["split"]

    return people, households, assignment


# ------------------------------------------------------------
# 7. MAIN NESTED-CV EXPERIMENT
# ------------------------------------------------------------

def main():
    people, households, assignment = load_inputs()

    tasks = {
        "occupation_person": (
            people, "label_occupation"
        ),
        "primary_mode_person": (
            people, "label_primary_mode"
        ),
        "income_household": (
            households, "label_income_class"
        )
    }

    all_outer_rows = []
    all_inner_rows = []
    summaries = []
    task_errors = []

    for task in TASKS_TO_RUN:
        print("\n" + "=" * 75)
        print("TASK:", task)
        print("=" * 75)

        try:
            table, target = tasks[task]
            table = table.copy()

            table["split"] = table["household_id"].map(
                assignment
            )

            if table["split"].isna().any():
                raise ValueError(
                    "Some household IDs have no split assignment."
                )

            development = table.loc[
                table["split"].eq("development")
            ].dropna(subset=[target]).reset_index(drop=True)

            # No previously inspected test records enter this experiment.
            print(
                "Development observations:",
                len(development)
            )
            print(
                "Development households:",
                development["household_id"].nunique()
            )
            print(development[target].value_counts())

            excluded = {
                "record_id",
                "household_id",
                "split",
                "label_occupation",
                "label_primary_mode",
                "label_income_class"
            }

            features = [
                c for c in development.columns
                if c not in excluded
                and not c.startswith("audit_")
            ]

            # Preserve the existing feature definitions.
            # Imputation is fitted separately in each pipeline fold.
            X = development[features].apply(
                pd.to_numeric, errors="raise"
            ).replace([np.inf, -np.inf], np.nan)

            encoder = LabelEncoder()
            y = encoder.fit_transform(
                development[target]
            )
            groups = (
                development["household_id"].to_numpy()
            )
            labels = np.arange(len(encoder.classes_))

            outer_folds = make_group_folds(
                X, y, groups,
                OUTER_FOLDS, SEED
            )

            oof_predictions = np.full(
                len(y), -1, dtype=int
            )
            task_outer_rows = []

            for fold_number, (
                train_index, valid_index
            ) in enumerate(outer_folds, start=1):

                print(
                    f"\nOUTER FOLD "
                    f"{fold_number}/{len(outer_folds)}"
                )

                X_train = X.iloc[train_index]
                X_valid = X.iloc[valid_index]

                y_train = y[train_index]
                y_valid = y[valid_index]

                train_groups = groups[train_index]

                inner_folds = make_group_folds(
                    X_train,
                    y_train,
                    train_groups,
                    INNER_FOLDS,
                    SEED + 100 * fold_number
                )

                winner, selected, comparison = (
                    select_model(
                        X_train,
                        y_train,
                        inner_folds,
                        labels,
                        task,
                        f"outer_fold_{fold_number}"
                    )
                )

                all_inner_rows.extend(comparison)

                prediction = selected.predict(
                    X_valid
                ).astype(int)

                oof_predictions[valid_index] = prediction

                baseline = DummyClassifier(
                    strategy="most_frequent"
                )
                baseline.fit(
                    np.zeros((len(y_train), 1)),
                    y_train
                )
                baseline_prediction = baseline.predict(
                    np.zeros((len(y_valid), 1))
                )

                model_metrics = metrics(
                    y_valid, prediction, labels
                )
                baseline_metrics = metrics(
                    y_valid, baseline_prediction, labels
                )

                row = {
                    "task": task,
                    "outer_fold": fold_number,
                    "selected_model": winner,
                    "validation_n": len(valid_index),
                    **model_metrics,
                    **{
                        f"baseline_{name}": value
                        for name, value
                        in baseline_metrics.items()
                    }
                }

                all_outer_rows.append(row)
                task_outer_rows.append(row)

                print("Selected:", winner)
                print(
                    "Outer macro F1:",
                    round(row["macro_f1"], 4)
                )
                print(
                    "Outer baseline macro F1:",
                    round(
                        row["baseline_macro_f1"], 4
                    )
                )

                pd.DataFrame(all_outer_rows).to_csv(
                    OUTPUT_DIR / "outer_fold_results.csv",
                    index=False
                )
                pd.DataFrame(all_inner_rows).to_csv(
                    OUTPUT_DIR / "inner_search_comparison.csv",
                    index=False
                )

            if (oof_predictions < 0).any():
                raise RuntimeError(
                    "Some observations lack outer-fold predictions."
                )

            save_diagnostics(
                task,
                development,
                y,
                oof_predictions,
                encoder
            )

            fold_table = pd.DataFrame(task_outer_rows)

            summary = {
                "task": task,
                "development_n": len(development),
                "development_households": (
                    development["household_id"].nunique()
                ),
                "outer_folds": len(outer_folds)
            }

            for metric_name in [
                "accuracy",
                "macro_f1",
                "balanced_accuracy",
                "mcc",
                "baseline_accuracy",
                "baseline_macro_f1"
            ]:
                summary[f"{metric_name}_mean"] = float(
                    fold_table[metric_name].mean()
                )
                summary[f"{metric_name}_std"] = float(
                    fold_table[metric_name].std(ddof=1)
                )

            summaries.append(summary)

            # Refit the selection procedure on all development data.
            # Its inner score is NOT the nested performance estimate.
            print(
                "\nSelecting final model using all "
                "development data..."
            )

            final_folds = make_group_folds(
                X, y, groups,
                INNER_FOLDS, SEED + 999
            )

            final_name, final_model, final_comparison = (
                select_model(
                    X, y, final_folds,
                    labels, task,
                    "final_development_refit"
                )
            )

            all_inner_rows.extend(final_comparison)

            joblib.dump(
                {
                    "pipeline": final_model,
                    "label_encoder": encoder,
                    "features": features,
                    "task": task,
                    "target": target,
                    "selected_model": final_name,
                    "training_partition": "development",
                    "source_directory": str(SOURCE_DIR)
                },
                OUTPUT_DIR / f"{task}_final_model.joblib"
            )

            print("Final fitted model:", final_name)

        except Exception as error:
            print("TASK FAILED:", error)
            task_errors.append({
                "task": task,
                "error": str(error)
            })

        pd.DataFrame(summaries).to_csv(
            OUTPUT_DIR / "nested_cv_summary.csv",
            index=False
        )
        pd.DataFrame(all_inner_rows).to_csv(
            OUTPUT_DIR / "inner_search_comparison.csv",
            index=False
        )

    pd.DataFrame(
        task_errors, columns=["task", "error"]
    ).to_csv(
        OUTPUT_DIR / "task_errors.csv",
        index=False
    )

    configuration = {
        "source_directory": str(SOURCE_DIR),
        "outer_folds_requested": OUTER_FOLDS,
        "inner_folds_requested": INNER_FOLDS,
        "search_iterations_per_model": (
            SEARCH_ITERATIONS_PER_MODEL
        ),
        "seed": SEED,
        "test_set_used": False,
        "notes": [
            "Mean outer-fold metrics are the primary estimates.",
            "Pooled OOF classification reports are diagnostic.",
            "Balanced Random Forest is optional.",
            "Original corrected labels are not changed.",
            "Household income remains a household-level task.",
            "Final models are fitted on development observations.",
            "Repeated adaptive development still requires "
            "independent final validation."
        ]
    }

    with open(
        OUTPUT_DIR / "configuration.json",
        "w",
        encoding="utf-8"
    ) as handle:
        json.dump(configuration, handle, indent=2)

    print("\n" + "=" * 75)
    print("NESTED CROSS-VALIDATION SUMMARY")
    print("=" * 75)

    if summaries:
        columns = [
            "task",
            "development_n",
            "macro_f1_mean",
            "macro_f1_std",
            "accuracy_mean",
            "balanced_accuracy_mean",
            "baseline_macro_f1_mean"
        ]

        print(
            pd.DataFrame(summaries)[columns]
            .to_string(index=False)
        )
    else:
        print("No task completed. Inspect task_errors.csv.")

    print("\nCompleted.")
    print("Results directory:", OUTPUT_DIR.resolve())
    print("Previously inspected test households were NOT used.")


if __name__ == "__main__":
    main()


TASK: occupation_person
Development observations: 2689
Development households: 932
label_occupation
employed         1353
student_child     602
homemaker         517
non_employed      157
other              60
Name: count, dtype: int64

OUTER FOLD 1/3
  Searching: random_forest
    Inner CV macro F1: 0.5273
  Searching: extra_trees
    Inner CV macro F1: 0.5349
  Searching: rbf_svm
    Inner CV macro F1: 0.5356
  Searching: balanced_random_forest
    Inner CV macro F1: 0.5339
  Searching: lightgbm
    Inner CV macro F1: 0.5357
  Searching: majority_baseline
    Inner CV macro F1: 0.134
Selected: lightgbm
Outer macro F1: 0.5104
Outer baseline macro F1: 0.1337

OUTER FOLD 2/3
  Searching: random_forest
    Inner CV macro F1: 0.5517
  Searching: extra_trees
    Inner CV macro F1: 0.5395
  Searching: rbf_svm
    Inner CV macro F1: 0.5228
  Searching: balanced_random_forest
    Inner CV macro F1: 0.545
  Searching: lightgbm
    Inner CV macro F1: 0.5584
  Searching: majority_baseline
    I

In [ ]:
# ============================================================
# NESTED-CV MOBILITY INFERENCE EXPERIMENT  (CORRECTED VERSION)
#
# Paste into ONE Jupyter cell (or %run the .py file) and execute.
#
# Required : pandas numpy scikit-learn matplotlib joblib
# Optional : imbalanced-learn (Balanced RF), lightgbm
#
# INPUTS (from the previous pipeline, in SOURCE_DIR):
#   person_level_table.csv, household_level_table.csv, household_split.csv
#
# Design:
#  - Household-grouped, class-stratified outer + inner CV
#  - All preprocessing fitted inside training folds (Pipeline)
#  - Held-out "test" households are never touched
#  - Majority baseline is reported, but cannot be chosen as the
#    "winning" model (it is the reference, not a candidate)
#  - Leakage guard: columns whose names contain the target keyword
#    are excluded and logged
#  - Rare classes handled explicitly and logged (not a hard crash)
# ============================================================

from pathlib import Path
import json
import traceback
import warnings

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", message=".*The total space of parameters.*")


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SOURCE_DIR = Path("output/person_household_corrected_results")
OUTPUT_DIR = Path("output/nested_balanced_mobility_experiment_corrected")

SEED = 42
OUTER_FOLDS = 3
INNER_FOLDS = 3
SEARCH_ITERATIONS_PER_MODEL = 5
SEARCH_JOBS = -1          # use 1 if you get memory / process errors

TASKS_TO_RUN = [
    "occupation_person",
    "primary_mode_person",
    "income_household",
]

# Rare-class handling (counted in INDEPENDENT HOUSEHOLDS, not rows).
# A class needs at least this many households to be cross-validated.
MIN_HOUSEHOLDS_PER_CLASS = 3
RARE_CLASS_POLICY = "drop"     # "drop" (log + remove) or "error"

# Leakage guard: columns whose name contains these words are excluded
# from the predictors for that task (and logged).
LEAKAGE_KEYWORDS = {
    "occupation_person": ["occupation"],
    "primary_mode_person": ["primary_mode"],
    "income_household": ["income"],
}
DROP_LEAKAGE_SUSPECTS = True

LABEL_COLUMNS = {
    "label_occupation",
    "label_primary_mode",
    "label_income_class",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. OPTIONAL PACKAGE CHECKS
# ------------------------------------------------------------

BalancedRandomForestClassifier = None
LGBMClassifier = None
PACKAGE_STATUS = {}

try:
    import imblearn
    from imblearn.ensemble import BalancedRandomForestClassifier
    PACKAGE_STATUS["imbalanced-learn"] = {
        "available": True, "version": imblearn.__version__}
except ImportError as error:
    PACKAGE_STATUS["imbalanced-learn"] = {
        "available": False, "error": str(error)}

try:
    import lightgbm
    from lightgbm import LGBMClassifier
    PACKAGE_STATUS["lightgbm"] = {
        "available": True, "version": lightgbm.__version__}
except ImportError as error:
    PACKAGE_STATUS["lightgbm"] = {
        "available": False, "error": str(error)}


# ------------------------------------------------------------
# 3. INPUT LOADING AND VALIDATION
# ------------------------------------------------------------

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA})
    )


def load_inputs():
    files = {
        "people": SOURCE_DIR / "person_level_table.csv",
        "households": SOURCE_DIR / "household_level_table.csv",
        "split": SOURCE_DIR / "household_split.csv",
    }

    missing = [str(p) for p in files.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Required files were not found. Check SOURCE_DIR:\n"
            + "\n".join(missing)
        )

    id_types = {"record_id": "string", "household_id": "string"}
    people = pd.read_csv(files["people"], dtype=id_types)
    households = pd.read_csv(files["households"], dtype=id_types)
    split = pd.read_csv(files["split"], dtype="string")

    for name, table in [("people", people),
                        ("households", households),
                        ("split", split)]:
        if "household_id" not in table.columns:
            raise ValueError(f"{name} table has no household_id column.")
        table["household_id"] = clean_id(table["household_id"])
        if table["household_id"].isna().any():
            raise ValueError(f"Missing household IDs in {name} table.")

    for name, table in [("people", people), ("households", households)]:
        if "record_id" not in table.columns:
            raise ValueError(f"{name} table has no record_id column.")
        if table["record_id"].duplicated().any():
            raise ValueError(f"Duplicate record_id values in {name} table.")

    if households["household_id"].duplicated().any():
        raise ValueError(
            "household_level_table.csv must have one row per household.")

    if "split" not in split.columns:
        raise ValueError("household_split.csv has no split column.")

    split["split"] = split["split"].str.strip()

    if split["household_id"].duplicated().any():
        raise ValueError("household_split.csv must have one row per household.")
    if split["split"].isna().any():
        raise ValueError("Missing split assignments.")
    if not set(split["split"]).issubset({"development", "test"}):
        raise ValueError("Unexpected split names (need development/test).")

    assignment = split.set_index("household_id")["split"]
    return people, households, assignment


# ------------------------------------------------------------
# 4. HOUSEHOLD-AWARE, CLASS-STRATIFIED FOLDS
# ------------------------------------------------------------

def make_folds(y, groups, requested_folds, seed):
    """Folds where no household appears on both sides and every class
    appears in every train AND validation part."""
    y = np.asarray(y)
    groups = np.asarray(groups)

    independent_counts = (
        pd.DataFrame({"label": y, "household": groups})
        .groupby("label")["household"].nunique()
    )
    if len(independent_counts) < 2:
        raise ValueError("At least two classes are required.")

    maximum = min(requested_folds, int(independent_counts.min()))
    expected = set(np.unique(y))
    dummy_X = np.zeros((len(y), 1))

    for n_splits in range(maximum, 1, -1):
        for attempt in range(25):
            splitter = StratifiedGroupKFold(
                n_splits=n_splits, shuffle=True,
                random_state=seed + attempt)
            folds = list(splitter.split(dummy_X, y, groups))

            ok = all(
                set(y[tr]) == expected and set(y[va]) == expected
                and set(groups[tr]).isdisjoint(set(groups[va]))
                for tr, va in folds
            )
            if ok:
                return folds

    raise ValueError(
        "Not enough independent households for class-complete "
        "cross-validation. Raise MIN_HOUSEHOLDS_PER_CLASS or merge classes."
    )


# ------------------------------------------------------------
# 5. MODELS
# ------------------------------------------------------------

def build_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


def get_candidates():
    candidates = {
        "majority_baseline": (
            DummyClassifier(strategy="most_frequent"), {}),

        "random_forest": (
            RandomForestClassifier(random_state=SEED, n_jobs=1),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [6, 12, None],
                "model__min_samples_leaf": [1, 3, 8, 15],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [None, "balanced", "balanced_subsample"],
            }),

        "extra_trees": (
            ExtraTreesClassifier(random_state=SEED, n_jobs=1),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [6, 12, None],
                "model__min_samples_leaf": [1, 3, 8, 15],
                "model__max_features": ["sqrt", 0.7],
                "model__class_weight": [None, "balanced"],
            }),

        "rbf_svm": (
            SVC(kernel="rbf", cache_size=512),
            {
                "model__C": [0.1, 1.0, 10.0, 50.0],
                "model__gamma": ["scale", 0.001, 0.01, 0.1],
                "model__class_weight": [None, "balanced"],
            }),
    }

    if BalancedRandomForestClassifier is not None:
        candidates["balanced_random_forest"] = (
            BalancedRandomForestClassifier(
                random_state=SEED, n_jobs=1,
                sampling_strategy="all", replacement=True, bootstrap=False),
            {
                "model__n_estimators": [250, 450],
                "model__max_depth": [4, 8, 16, None],
                "model__min_samples_leaf": [1, 3, 8],
                "model__max_features": ["sqrt", 0.7],
            })

    if LGBMClassifier is not None:
        candidates["lightgbm"] = (
            LGBMClassifier(
                random_state=SEED, n_jobs=1, verbosity=-1, subsample_freq=1),
            {
                "model__n_estimators": [150, 300],
                "model__learning_rate": [0.03, 0.07],
                "model__num_leaves": [7, 15, 31],
                "model__min_child_samples": [15, 30, 60],
                "model__reg_lambda": [0.0, 1.0, 5.0],
                "model__class_weight": [None, "balanced"],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0],
            })

    return candidates


# ------------------------------------------------------------
# 6. INNER MODEL SEARCH
# ------------------------------------------------------------

def macro_f1(y_true, y_pred, labels):
    return f1_score(y_true, y_pred, labels=labels,
                    average="macro", zero_division=0)


def select_model(X, y, folds, labels, task, stage):
    scorer = make_scorer(f1_score, labels=labels,
                         average="macro", zero_division=0)
    rows, fitted = [], {}

    for name, (model, parameters) in get_candidates().items():
        print(f"  Searching: {name}", flush=True)
        try:
            if parameters:
                search = RandomizedSearchCV(
                    estimator=build_pipeline(model),
                    param_distributions=parameters,
                    n_iter=SEARCH_ITERATIONS_PER_MODEL,
                    scoring=scorer, cv=folds, refit=True,
                    n_jobs=SEARCH_JOBS, random_state=SEED,
                    error_score="raise")
                search.fit(X, y)
                pos = search.best_index_
                score = float(search.best_score_)
                std = float(search.cv_results_["std_test_score"][pos])
                params = search.best_params_
                estimator = search.best_estimator_
                pd.DataFrame(search.cv_results_).to_csv(
                    OUTPUT_DIR / f"{task}_{stage}_{name}_search.csv",
                    index=False)
            else:
                # Baseline: no hyper-parameters, evaluate on the same folds
                scores = []
                for tr, va in folds:
                    est = clone(build_pipeline(model))
                    est.fit(X.iloc[tr], y[tr])
                    scores.append(macro_f1(y[va], est.predict(X.iloc[va]), labels))
                score, std, params = float(np.mean(scores)), float(np.std(scores, ddof=1)), {}
                estimator = clone(build_pipeline(model)).fit(X, y)

            rows.append({
                "task": task, "stage": stage, "model": name,
                "inner_cv_macro_f1": score, "inner_cv_std": std,
                "parameters": json.dumps(params, default=str), "error": "",
            })
            fitted[name] = estimator
            print(f"    Inner CV macro F1: {score:.4f}", flush=True)

        except Exception as error:
            print(f"    Failed: {error}", flush=True)
            rows.append({
                "task": task, "stage": stage, "model": name,
                "inner_cv_macro_f1": np.nan, "inner_cv_std": np.nan,
                "parameters": "", "error": str(error),
            })

    valid = pd.DataFrame(rows).dropna(subset=["inner_cv_macro_f1"])
    if valid.empty:
        raise RuntimeError(f"No model succeeded for {task}, {stage}.")

    # The baseline is a reference, not a selectable model.
    contenders = valid[valid["model"] != "majority_baseline"]
    if contenders.empty:
        contenders = valid

    best = contenders.sort_values(
        "inner_cv_macro_f1", ascending=False, kind="stable").iloc[0]
    winner = best["model"]
    return winner, fitted[winner], rows, best


# ------------------------------------------------------------
# 7. METRICS AND DIAGNOSTICS
# ------------------------------------------------------------

def calculate_metrics(y_true, predictions, labels):
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": macro_f1(y_true, predictions, labels),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "mcc": matthews_corrcoef(y_true, predictions),
    }


def save_diagnostics(task, development, y, predictions,
                     fold_assignments, encoder):
    labels = np.arange(len(encoder.classes_))

    report = classification_report(
        y, predictions, labels=labels, target_names=encoder.classes_,
        zero_division=0, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(
        OUTPUT_DIR / f"{task}_pooled_oof_report.csv")

    matrix = confusion_matrix(y, predictions, labels=labels)
    pd.DataFrame(matrix, index=encoder.classes_,
                 columns=encoder.classes_).to_csv(
        OUTPUT_DIR / f"{task}_oof_confusion_matrix.csv")

    diagnostic = development[["record_id", "household_id"]].copy()
    diagnostic["outer_fold"] = fold_assignments
    diagnostic["actual"] = encoder.inverse_transform(y)
    diagnostic["predicted"] = encoder.inverse_transform(predictions)
    diagnostic["correct"] = diagnostic["actual"] == diagnostic["predicted"]
    diagnostic.to_csv(OUTPUT_DIR / f"{task}_oof_predictions.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set_xticks(labels)
    ax.set_yticks(labels)
    ax.set_xticklabels(encoder.classes_, rotation=45, ha="right")
    ax.set_yticklabels(encoder.classes_)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{task}: nested-CV out-of-fold predictions")

    threshold = matrix.max() / 2 if matrix.max() > 0 else 0
    for r in labels:
        for c in labels:
            ax.text(c, r, str(matrix[r, c]), ha="center", va="center",
                    color="white" if matrix[r, c] > threshold else "black")

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{task}_oof_confusion_matrix.png", dpi=200)
    plt.close(fig)


def save_progress(outer_rows, inner_rows, summaries, errors):
    if outer_rows:
        pd.DataFrame(outer_rows).to_csv(
            OUTPUT_DIR / "outer_fold_results.csv", index=False)
    if inner_rows:
        pd.DataFrame(inner_rows).to_csv(
            OUTPUT_DIR / "inner_search_comparison.csv", index=False)
    if summaries:
        pd.DataFrame(summaries).to_csv(
            OUTPUT_DIR / "nested_cv_summary.csv", index=False)
    pd.DataFrame(errors, columns=["task", "error", "traceback"]).to_csv(
        OUTPUT_DIR / "task_errors.csv", index=False)


# ------------------------------------------------------------
# 8. DATA PREPARATION HELPERS
# ------------------------------------------------------------

def handle_rare_classes(development, target, task):
    counts = development.groupby(target)["household_id"].nunique()
    rare = counts[counts < MIN_HOUSEHOLDS_PER_CLASS]

    if rare.empty:
        return development

    message = (
        f"Classes with < {MIN_HOUSEHOLDS_PER_CLASS} independent households: "
        + ", ".join(f"{k} ({v})" for k, v in rare.items())
    )
    if RARE_CLASS_POLICY == "error":
        raise ValueError(message)

    print("\nWARNING:", message)
    print("These classes are removed from this task (RARE_CLASS_POLICY='drop').")

    dropped = development[development[target].isin(rare.index)]
    dropped[["record_id", "household_id", target]].to_csv(
        OUTPUT_DIR / f"{task}_dropped_rare_class_rows.csv", index=False)

    return development.loc[
        ~development[target].isin(rare.index)].reset_index(drop=True)


def select_features(development, target, task):
    structural = {"record_id", "household_id", "split"} | LABEL_COLUMNS

    features, flagged = [], []
    keywords = [k.lower() for k in LEAKAGE_KEYWORDS.get(task, [])]

    for column in development.columns:
        if column in structural or column.startswith("audit_"):
            continue
        if any(k in column.lower() for k in keywords):
            flagged.append(column)
            if DROP_LEAKAGE_SUSPECTS:
                continue
        features.append(column)

    if flagged:
        print("\nLeakage-guard: columns matching target keywords:", flagged)
        print("  ->", "EXCLUDED" if DROP_LEAKAGE_SUSPECTS else "KEPT (check manually)")
        pd.Series(flagged, name="column").to_csv(
            OUTPUT_DIR / f"{task}_leakage_suspects.csv", index=False)

    if not features:
        raise ValueError("No predictor columns found.")

    numeric = development[features].apply(pd.to_numeric, errors="coerce")
    bad = [
        c for c in features
        if numeric[c].isna().sum() > development[c].isna().sum()
    ]
    if bad:
        raise ValueError(
            "Non-numeric predictor columns found (encode or exclude them): "
            + ", ".join(bad))

    X = numeric.replace([np.inf, -np.inf], np.nan).astype(float)
    return X, features


# ------------------------------------------------------------
# 9. MAIN WORKFLOW
# ------------------------------------------------------------

def main():
    print("=" * 75)
    print("CORRECTED NESTED-CV MOBILITY EXPERIMENT")
    print("=" * 75)

    print("\nOptional packages:")
    for package, status in PACKAGE_STATUS.items():
        if status["available"]:
            print(f"  {package}: available (version {status['version']})")
        else:
            print(f"  {package}: unavailable; related model skipped.")

    people, households, assignment = load_inputs()

    tasks = {
        "occupation_person": (people, "label_occupation"),
        "primary_mode_person": (people, "label_primary_mode"),
        "income_household": (households, "label_income_class"),
    }

    outer_rows, inner_rows, summaries, errors = [], [], [], []

    for task in TASKS_TO_RUN:
        print("\n" + "=" * 75)
        print("TASK:", task)
        print("=" * 75)

        try:
            if task not in tasks:
                raise ValueError(f"Unknown task: {task}")

            table, target = tasks[task]
            table = table.copy()

            required = {"record_id", "household_id", target}
            if not required.issubset(table.columns):
                raise ValueError(
                    f"Missing task columns: {required - set(table.columns)}")

            table["split"] = table["household_id"].map(assignment)
            if table["split"].isna().any():
                raise ValueError("Some households lack split assignments.")

            development = (
                table.loc[table["split"].eq("development")]
                .dropna(subset=[target])
                .reset_index(drop=True)
            )

            development = handle_rare_classes(development, target, task)

            print("Development observations:", len(development))
            print("Development households:",
                  development["household_id"].nunique())
            print("\nLabel counts:")
            print(development[target].value_counts())
            print("\nIndependent households per class:")
            print(development.groupby(target)["household_id"].nunique())

            X, features = select_features(development, target, task)

            encoder = LabelEncoder()
            y = encoder.fit_transform(development[target])
            groups = development["household_id"].to_numpy()
            labels = np.arange(len(encoder.classes_))

            outer_folds = make_folds(y, groups, OUTER_FOLDS, SEED)

            oof_predictions = np.full(len(y), -1, dtype=int)
            fold_assignments = np.full(len(y), -1, dtype=int)
            task_outer_rows = []

            for fold_number, (train_index, valid_index) in enumerate(
                    outer_folds, start=1):
                print(f"\nOUTER FOLD {fold_number}/{len(outer_folds)}",
                      flush=True)

                X_train = X.iloc[train_index]
                X_valid = X.iloc[valid_index]
                y_train = y[train_index]
                y_valid = y[valid_index]

                inner_folds = make_folds(
                    y_train, groups[train_index],
                    INNER_FOLDS, SEED + 100 * fold_number)

                winner, selected, comparison, best = select_model(
                    X_train, y_train, inner_folds, labels,
                    task, f"outer_fold_{fold_number}")
                inner_rows.extend(comparison)

                predictions = selected.predict(X_valid).astype(int)
                oof_predictions[valid_index] = predictions
                fold_assignments[valid_index] = fold_number

                # Baseline: majority class of the TRAINING fold
                majority = np.bincount(y_train, minlength=len(labels)).argmax()
                baseline_predictions = np.full(len(y_valid), majority)

                model_metrics = calculate_metrics(y_valid, predictions, labels)
                baseline_metrics = calculate_metrics(
                    y_valid, baseline_predictions, labels)

                row = {
                    "task": task,
                    "outer_fold": fold_number,
                    "selected_model": winner,
                    "selected_inner_cv_macro_f1": float(best["inner_cv_macro_f1"]),
                    "selected_parameters": best["parameters"],
                    "train_n": len(train_index),
                    "validation_n": len(valid_index),
                    "validation_households": len(set(groups[valid_index])),
                    **model_metrics,
                    **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
                }
                outer_rows.append(row)
                task_outer_rows.append(row)

                print("\nSelected:", winner)
                print(f"Outer macro F1:          {row['macro_f1']:.4f}")
                print(f"Outer accuracy:          {row['accuracy']:.4f}")
                print(f"Outer baseline macro F1: {row['baseline_macro_f1']:.4f}")

                save_progress(outer_rows, inner_rows, summaries, errors)

            if (oof_predictions < 0).any() or (fold_assignments < 0).any():
                raise RuntimeError("Missing outer-fold predictions.")

            save_diagnostics(task, development, y, oof_predictions,
                             fold_assignments, encoder)

            fold_table = pd.DataFrame(task_outer_rows)
            summary = {
                "task": task,
                "development_n": len(development),
                "development_households": development["household_id"].nunique(),
                "n_classes": len(labels),
                "n_features": len(features),
                "outer_folds": len(outer_folds),
            }
            for metric in ["accuracy", "macro_f1", "balanced_accuracy", "mcc",
                           "baseline_accuracy", "baseline_macro_f1",
                           "baseline_balanced_accuracy"]:
                summary[f"{metric}_mean"] = float(fold_table[metric].mean())
                summary[f"{metric}_std"] = float(fold_table[metric].std(ddof=1))
            summary["macro_f1_gain_over_baseline"] = (
                summary["macro_f1_mean"] - summary["baseline_macro_f1_mean"])
            summaries.append(summary)

            save_progress(outer_rows, inner_rows, summaries, errors)

            print("\nSelecting final model on all development observations...",
                  flush=True)

            final_folds = make_folds(y, groups, INNER_FOLDS, SEED + 999)
            final_name, final_model, final_comparison, _ = select_model(
                X, y, final_folds, labels, task, "final_development_refit")
            inner_rows.extend(final_comparison)

            joblib.dump(
                {
                    "pipeline": final_model,
                    "label_encoder": encoder,
                    "features": features,
                    "task": task,
                    "target": target,
                    "selected_model": final_name,
                    "training_partition": "development",
                    "source_directory": str(SOURCE_DIR),
                },
                OUTPUT_DIR / f"{task}_final_model.joblib")

            with open(OUTPUT_DIR / f"{task}_features.json", "w",
                      encoding="utf-8") as handle:
                json.dump(features, handle, indent=2)

            print("Final fitted model:", final_name)

        except Exception as error:
            print("\nTASK FAILED:", error, flush=True)
            errors.append({
                "task": task,
                "error": str(error),
                "traceback": traceback.format_exc(),
            })

        save_progress(outer_rows, inner_rows, summaries, errors)

    configuration = {
        "source_directory": str(SOURCE_DIR),
        "output_directory": str(OUTPUT_DIR),
        "outer_folds_requested": OUTER_FOLDS,
        "inner_folds_requested": INNER_FOLDS,
        "search_iterations_per_model": SEARCH_ITERATIONS_PER_MODEL,
        "search_jobs": SEARCH_JOBS,
        "seed": SEED,
        "tasks": TASKS_TO_RUN,
        "min_households_per_class": MIN_HOUSEHOLDS_PER_CLASS,
        "rare_class_policy": RARE_CLASS_POLICY,
        "drop_leakage_suspects": DROP_LEAKAGE_SUSPECTS,
        "optional_packages": PACKAGE_STATUS,
        "test_set_used": False,
        "notes": [
            "Mean outer-fold metrics are the main evaluation.",
            "Pooled OOF reports are diagnostic only (different models per fold).",
            "Standard deviations across 3 folds are not confidence intervals.",
            "Majority baseline is reported but not selectable as winner.",
            "Income is a household-level task.",
            "Final models use development households only.",
            "Independent final validation on test households is still required.",
        ],
    }
    with open(OUTPUT_DIR / "configuration.json", "w", encoding="utf-8") as h:
        json.dump(configuration, h, indent=2, default=str)

    print("\n" + "=" * 75)
    print("FINAL NESTED-CV SUMMARY")
    print("=" * 75)

    if summaries:
        summary_table = pd.DataFrame(summaries)
        columns = [
            "task", "development_n", "development_households", "n_classes",
            "accuracy_mean", "macro_f1_mean", "macro_f1_std",
            "balanced_accuracy_mean", "mcc_mean",
            "baseline_macro_f1_mean", "macro_f1_gain_over_baseline",
        ]
        with pd.option_context("display.max_columns", None,
                               "display.width", 200):
            print(summary_table[columns].round(4).to_string(index=False))
    else:
        print("No task completed successfully.")

    if errors:
        print("\nTasks with errors (see task_errors.csv):")
        for e in errors:
            print(" -", e["task"], ":", e["error"])

    print("\nAll outputs saved in:", OUTPUT_DIR.resolve())
    return pd.DataFrame(summaries), pd.DataFrame(outer_rows)


# In Jupyter: just run this cell.
summary_df, outer_df = main()
summary_df

CORRECTED NESTED-CV MOBILITY EXPERIMENT

Optional packages:
  imbalanced-learn: available (version 0.14.2)
  lightgbm: available (version 4.7.0)

TASK: occupation_person
Development observations: 2689
Development households: 932

Label counts:
label_occupation
employed         1353
student_child     602
homemaker         517
non_employed      157
other              60
Name: count, dtype: int64

Independent households per class:
label_occupation
employed         868
homemaker        449
non_employed     147
other             56
student_child    450
Name: household_id, dtype: int64

OUTER FOLD 1/3
  Searching: majority_baseline
    Inner CV macro F1: 0.1340
  Searching: random_forest
    Inner CV macro F1: 0.5273
  Searching: extra_trees
    Inner CV macro F1: 0.5349
  Searching: rbf_svm
    Inner CV macro F1: 0.5356
  Searching: balanced_random_forest
    Inner CV macro F1: 0.5339
  Searching: lightgbm
    Inner CV macro F1: 0.5357

Selected: lightgbm
Outer macro F1:          0.5104
Out

,task,development_n,development_households,n_classes,n_features,outer_folds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,...,balanced_accuracy_std,mcc_mean,mcc_std,baseline_accuracy_mean,baseline_accuracy_std,baseline_macro_f1_mean,baseline_macro_f1_std,baseline_balanced_accuracy_mean,baseline_balanced_accuracy_std,macro_f1_gain_over_baseline
0,occupation_person,2689,932,5,48,3,0.743338,0.018484,0.521262,0.010869,...,0.014718,0.635260,0.022897,0.503086,0.008744,0.133875,0.001547,0.200000,3.399350e-17,0.387388
1,primary_mode_person,2491,910,6,48,3,0.849962,0.004565,0.778088,0.002522,...,0.003920,0.806098,0.005976,0.332799,0.002919,0.083232,0.000547,0.166667,0.000000e+00,0.694855
2,income_household,896,896,3,146,3,0.512252,0.024497,0.348205,0.029911,...,0.039676,0.060536,0.051548,0.722098,0.006711,0.279538,0.001508,0.333333,0.000000e+00,0.068667
